In [ ]:
"""
VERİ HAZIRLAMA — TAR → LOCAL → NPY (OPTİMİZE)
=================================================
167GB RAM + 80GB GPU + 235GB Disk
Multiprocessing ile paralel okuma → max hız
"""

from google.colab import drive
drive.mount('/content/drive')

import os, time, gc
import cv2
import numpy as np
from pathlib import Path
from multiprocessing import Pool, cpu_count

ARCHIVE = Path('/content/drive/MyDrive/archive')
PURE_BG_DIR = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors/training/pure_background'
TESTING_DIR = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final'
NPY_DIR = ARCHIVE / 'npy_cache'
NPY_DIR.mkdir(exist_ok=True)

TAR_BG = ARCHIVE / 'pure_bg.tar'
TAR_TEST = ARCHIVE / 'test_final.tar'
LOCAL_BG = Path('/content/data/pure_background')
LOCAL_TEST = Path('/content/data/testing_final')

FRAMES_PER_CLIP = 16
WORKERS = min(cpu_count(), 8)
print(f"CPU cores: {cpu_count()} | Workers: {WORKERS}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# ADIM 1 — TAR
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("ADIM 1 — TAR")
print("=" * 60)

for tar_path, src_dir, name in [
    (TAR_BG, PURE_BG_DIR, 'pure_background'),
    (TAR_TEST, TESTING_DIR, 'testing_final'),
]:
    if not tar_path.exists():
        print(f"  {name}.tar oluşturuluyor...", flush=True)
        t0 = time.time()
        os.system(f'tar cf "{tar_path}" -C "{src_dir.parent}" {name}')
        print(f"  {time.time()-t0:.0f}s | {tar_path.stat().st_size/1024/1024:.0f}MB")
    else:
        print(f"  {tar_path.name} var ({tar_path.stat().st_size/1024/1024:.0f}MB)")
print()

# ═══════════════════════════════════════════════════════════════════════════════
# ADIM 2 — LOCAL'E AÇ
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("ADIM 2 — LOCAL'E AÇ")
print("=" * 60)

os.makedirs('/content/data', exist_ok=True)

for tar_path, local_dir, name in [
    (TAR_BG, LOCAL_BG, 'pure_bg'),
    (TAR_TEST, LOCAL_TEST, 'test'),
]:
    if not local_dir.exists():
        print(f"  {name} açılıyor...", flush=True)
        t0 = time.time()
        os.system(f'tar xf "{tar_path}" -C /content/data/')
        print(f"  {time.time()-t0:.0f}s")
    else:
        print(f"  {name} zaten local'de")
print()

# ═══════════════════════════════════════════════════════════════════════════════
# ADIM 3 — NPY (paralel okuma)
# ═══════════════════════════════════════════════════════════════════════════════

def load_clip_worker(args):
    """Tek klip yükle — worker fonksiyonu"""
    clip_dir, fs = args
    clip_dir = Path(clip_dir)
    jpgs = sorted(clip_dir.glob('frame_*.jpg'))
    frames = []
    for jpg in jpgs[:FRAMES_PER_CLIP]:
        img = cv2.imread(str(jpg), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            frames.append(cv2.resize(img, (fs, fs)).astype(np.float32) / 255.0)
    while len(frames) < FRAMES_PER_CLIP:
        frames.append(np.zeros((fs, fs), dtype=np.float32))
    return np.stack(frames)


for frame_size in [64, 128]:
    print("=" * 60)
    print(f"NPY {frame_size}x{frame_size}")
    print("=" * 60)

    # ── Pure Background ──
    bg_npy = NPY_DIR / f'pure_bg_{frame_size}.npy'
    if not bg_npy.exists():
        print(f"  pure_bg yükleniyor ({WORKERS} worker)...", flush=True)
        t0 = time.time()

        bg_dirs = []
        for dd in sorted(LOCAL_BG.iterdir()):
            if not dd.is_dir(): continue
            for cd in sorted(dd.iterdir()):
                if cd.is_dir():
                    bg_dirs.append((str(cd), frame_size))

        print(f"  {len(bg_dirs)} klip", flush=True)
        with Pool(WORKERS) as pool:
            bg_arr = pool.map(load_clip_worker, bg_dirs)

        bg_data = np.stack(bg_arr).astype(np.float16)
        print(f"  Shape: {bg_data.shape} | {bg_data.nbytes/1024/1024:.0f}MB")
        np.save(str(bg_npy), bg_data)
        del bg_data, bg_arr; gc.collect()
        print(f"  Kaydedildi: {time.time()-t0:.0f}s")
    else:
        print(f"  pure_bg_{frame_size}.npy var")

    # ── Testing Final ──
    test_npy = NPY_DIR / f'test_{frame_size}.npy'
    test_lbl_npy = NPY_DIR / f'test_labels_{frame_size}.npy'
    if not test_npy.exists():
        print(f"  testing_final yükleniyor ({WORKERS} worker)...", flush=True)
        t0 = time.time()

        test_dirs = []; test_labels = []; test_names = []
        for dd in sorted(LOCAL_TEST.iterdir()):
            if not dd.is_dir(): continue
            for cc in dd.iterdir():
                if not cc.is_dir(): continue
                lbl = 1 if cc.name == 'anomaly' else 0
                for cd in cc.iterdir():
                    if cd.is_dir():
                        test_dirs.append((str(cd), frame_size))
                        test_labels.append(lbl)
                        test_names.append(f"{dd.name}/{cc.name}/{cd.name}")

        print(f"  {len(test_dirs)} klip", flush=True)
        with Pool(WORKERS) as pool:
            test_arr = pool.map(load_clip_worker, test_dirs)

        test_data = np.stack(test_arr).astype(np.float16)
        t_lbl = np.array(test_labels)
        print(f"  Shape: {test_data.shape} | N:{(t_lbl==0).sum()} A:{(t_lbl==1).sum()}")
        np.save(str(test_npy), test_data)
        np.save(str(test_lbl_npy), t_lbl)
        np.save(str(NPY_DIR / f'test_names_{frame_size}.npy'), np.array(test_names))
        del test_data, test_arr; gc.collect()
        print(f"  Kaydedildi: {time.time()-t0:.0f}s")
    else:
        print(f"  test_{frame_size}.npy var")

    print()

# ═══════════════════════════════════════════════════════════════════════════════
# RAPOR
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("TAMAMLANDI")
print("=" * 60)
for f in sorted(NPY_DIR.glob('*.npy')):
    print(f"  {f.name}: {f.stat().st_size/1024/1024:.0f}MB")
print(f"\nBundan sonra eğitimde:")
print(f"  bg = np.load('{NPY_DIR}/pure_bg_64.npy').astype(np.float32)")
print(f"  → 30 saniyede yüklenir!")

In [ ]:
"""
CR-AE PROTOTİP v5 — NPY'DEN YÜKLE + EĞİT + TEST
=====================================================
NPY dosyaları hazır → 30sn yükleme → eğitim → test → AUC
"""

import subprocess
subprocess.run(['pip', 'install', 'torch torchvision scikit-learn', '-q'])

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, random, gc
import numpy as np
from pathlib import Path
from IPython.display import display, Image as IPImage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

SEED = 42; random.seed(SEED); torch.manual_seed(SEED)
NPY_DIR = Path('/content/drive/MyDrive/archive/npy_cache')

# ═══════════════════════════════════════════════════════════════════════════════
# VERİ YÜKLE (30sn)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("VERİ YÜKLEME (NPY)")
print("=" * 60)
t0 = time.time()

bg_all = np.load(str(NPY_DIR / 'pure_bg_64.npy')).astype(np.float32)
test_all = np.load(str(NPY_DIR / 'test_64.npy')).astype(np.float32)
test_labels = np.load(str(NPY_DIR / 'test_labels_64.npy'))

print(f"Pure BG: {bg_all.shape} | Test: {test_all.shape}")
print(f"Test N:{(test_labels==0).sum()} A:{(test_labels==1).sum()}")
print(f"Yükleme: {time.time()-t0:.1f}s\n")

# Prototip: %10 eğitim verisi
SAMPLE_RATIO = 0.10
n_sample = int(len(bg_all) * SAMPLE_RATIO)
idx = random.sample(range(len(bg_all)), n_sample)
bg_sample = bg_all[idx]
del bg_all; gc.collect()

# Train/val split
VAL_SPLIT = 0.15
n_val = int(len(bg_sample) * VAL_SPLIT)
perm = list(range(len(bg_sample))); random.shuffle(perm)
val_data = bg_sample[perm[:n_val]]
train_data = bg_sample[perm[n_val:]]
del bg_sample; gc.collect()

print(f"Prototip: Train:{len(train_data)} Val:{len(val_data)} Test:{len(test_all)}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════

BATCH_SIZE = 8

class DS(Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1)  # [16,1,64,64]

class TDS(Dataset):
    def __init__(self, d, l): self.d = d; self.l = l
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

train_dl = DataLoader(DS(train_data), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_dl = DataLoader(DS(val_data), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_dl = DataLoader(TDS(test_all, test_labels), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL — CR-AE with projected LSTM
# ═══════════════════════════════════════════════════════════════════════════════

class CRAE(nn.Module):
    """
    Encoder: Conv2D 1→16→32→64→128 (64→4)
    Project: 2048→256
    LSTM: 2 layer, hidden=256
    Unproject: 256→2048
    Decoder: ConvT2d 128→64→32→16→1 (4→64)
    """
    def __init__(self, lstm_h=256):
        super().__init__()
        self.feat = 128 * 4 * 4  # 2048

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.LeakyReLU(0.2),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),
        )
        self.proj = nn.Linear(self.feat, lstm_h)
        self.lstm = nn.LSTM(lstm_h, lstm_h, num_layers=2, batch_first=True, dropout=0.3)
        self.unproj = nn.Linear(lstm_h, self.feat)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(16), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )

    def forward(self, clip):
        B, T, C, H, W = clip.shape
        x = self.encoder(clip.view(B*T, C, H, W))         # [B*T, 128, 4, 4]
        x = self.proj(x.view(B*T, -1)).view(B, T, -1)     # [B, T, 256]
        x, _ = self.lstm(x)                                 # [B, T, 256]
        x = self.unproj(x.reshape(B*T, -1)).view(B*T, 128, 4, 4)
        x = self.decoder(x)                                 # [B*T, 1, 64, 64]
        return x.view(B, T, 1, H, W)

model = CRAE().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params (~{n_params*4/1024/1024:.1f}MB)")

# ═══════════════════════════════════════════════════════════════════════════════
# EĞİTİM
# ═══════════════════════════════════════════════════════════════════════════════

EPOCHS = 15
LR = 1e-3
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

print("\n" + "=" * 60)
print(f"EĞİTİM | Ep:{EPOCHS} B:{BATCH_SIZE} LR:{LR}")
print("=" * 60)

tl_h = []; vl_h = []; best_val = float('inf')
t0 = time.time()

for ep in range(EPOCHS):
    model.train(); el = nb = 0
    for b in train_dl:
        c = b.to(device); optimizer.zero_grad()
        loss = criterion(model(c), c); loss.backward(); optimizer.step()
        el += loss.item(); nb += 1
    tl = el/max(nb,1); tl_h.append(tl)

    model.eval(); vl = nv = 0
    with torch.no_grad():
        for b in val_dl:
            c = b.to(device); vl += criterion(model(c), c).item(); nv += 1
    vl /= max(nv,1); vl_h.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val = vl; torch.save(model.state_dict(), '/content/best.pth')

    print(f"  Ep{ep+1:>2}/{EPOCHS} T:{tl:.6f} V:{vl:.6f} LR:{optimizer.param_groups[0]['lr']:.6f} {time.time()-t0:.0f}s", flush=True)

tt = time.time()-t0
print(f"\nEğitim: {tt:.0f}s ({tt/60:.1f}dk)")

# ═══════════════════════════════════════════════════════════════════════════════
# GRAFİKLER
# ═══════════════════════════════════════════════════════════════════════════════

# Eğitim eğrisi
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(tl_h, 'b-', label='Train'); ax.plot(vl_h, 'r-', label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.set_title('CR-AE Prototip — Eğitim')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('/content/curve.png', dpi=100); plt.close()
display(IPImage(filename='/content/curve.png'))

# Reconstruction örnekleri
model.load_state_dict(torch.load('/content/best.pth')); model.eval()
s = torch.tensor(val_data[:3]).unsqueeze(2).to(device)
with torch.no_grad(): r = model(s)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    for j, (t, d) in enumerate([
        (f'K{i+1} F01 Orig', s[i,0,0].cpu()), (f'K{i+1} F01 Recon', r[i,0,0].cpu()),
        (f'K{i+1} F09 Orig', s[i,8,0].cpu()), (f'K{i+1} F09 Recon', r[i,8,0].cpu()),
    ]):
        axes[i,j].imshow(d.numpy(), cmap='gray', vmin=0, vmax=1)
        axes[i,j].set_title(t); axes[i,j].axis('off')
plt.tight_layout(); plt.savefig('/content/recon.png', dpi=100); plt.close()
display(IPImage(filename='/content/recon.png'))

# ═══════════════════════════════════════════════════════════════════════════════
# TEST
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("TEST")
print("=" * 60)

scores = []; labels = []
with torch.no_grad():
    for clips, lbls in test_dl:
        clips = clips.to(device); out = model(clips)
        for i in range(clips.shape[0]):
            scores.append(torch.mean((clips[i]-out[i])**2).item())
            labels.append(lbls[i].item())

scores = np.array(scores); labels = np.array(labels)
ns = scores[labels==0]; als = scores[labels==1]

print(f"Normal MSE:  {ns.mean():.6f} (±{ns.std():.6f})")
print(f"Anomali MSE: {als.mean():.6f} (±{als.std():.6f})")
print(f"Oran: {als.mean()/ns.mean():.2f}x")

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score
auc = roc_auc_score(labels, scores)
print(f"\n*** AUC-ROC: {auc:.4f} ***")

# Optimal threshold ile F1
precisions, recalls, thresholds_pr = precision_recall_curve(labels, scores)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]
print(f"Best F1: {best_f1:.4f} (threshold: {best_threshold:.6f})")

# Precision/Recall at best threshold
preds = (scores >= best_threshold).astype(int)
tp = ((preds == 1) & (labels == 1)).sum()
fp = ((preds == 1) & (labels == 0)).sum()
fn = ((preds == 0) & (labels == 1)).sum()
tn = ((preds == 0) & (labels == 0)).sum()
precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")
print(f"TP:{tp} FP:{fp} FN:{fn} TN:{tn}")

# Grafikler
fpr, tpr, _ = roc_curve(labels, scores)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc:.4f}')
axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(ns, bins=30, alpha=0.6, label='Normal', color='green')
axes[1].hist(als, bins=30, alpha=0.6, label='Anomali', color='red')
axes[1].axvline(best_threshold, color='black', ls='--', label=f'Threshold={best_threshold:.5f}')
axes[1].set_xlabel('MSE'); axes[1].set_ylabel('Count'); axes[1].set_title('Score Dağılımı')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(thresholds_pr, precisions[:-1], 'b-', label='Precision')
axes[2].plot(thresholds_pr, recalls[:-1], 'r-', label='Recall')
axes[2].plot(thresholds_pr, f1_scores[:-1], 'g-', label='F1')
axes[2].axvline(best_threshold, color='black', ls='--')
axes[2].set_xlabel('Threshold'); axes[2].set_title('Precision/Recall/F1')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('/content/test.png', dpi=100); plt.close()
display(IPImage(filename='/content/test.png'))

# ═══════════════════════════════════════════════════════════════════════════════
# ÖZET
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("PROTOTİP ÖZET")
print("=" * 60)
print(f"  Params      : {n_params:,}")
print(f"  Train/Val   : {len(train_data)}/{len(val_data)}")
print(f"  Test        : {len(test_all)} (N:{(test_labels==0).sum()} A:{(test_labels==1).sum()})")
print(f"  Best val    : {best_val:.6f}")
print(f"  Normal MSE  : {ns.mean():.6f}")
print(f"  Anomali MSE : {als.mean():.6f}")
print(f"  AUC-ROC     : {auc:.4f}")
print(f"  Best F1     : {best_f1:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")

if auc >= 0.70:
    print("\n  ✅ Başarılı — tam eğitime geçilebilir")
elif auc >= 0.55:
    print("\n  ⚠️ Orta — ayar gerekebilir")
else:
    print("\n  ❌ Düşük — mimari değişikliği gerek")
print("=" * 60)

In [ ]:
"""
CR-AE ASIL MODEL EĞİTİMİ (v3)
================================
Google Colab — A100 80GB GPU, 167GB RAM

Değişiklikler (v1 → v3):
  1. Val split: Rastgele → Ay/gün bazında stratified (data leakage yok)
  2. Batch size: 16 → 32 (A100 tensor core optimal)
  3. plt.savefig sıralama düzeltmesi
  4. weight_decay CONFIG'e eklendi (reproducibility)
  5. EER (Equal Error Rate) metriği eklendi
  6. Seasonal Robustness Score — ay bazlı AUC (concept drift analizi)
  7. Inference Latency ölçümü (tez hedefi: <30ms/frame)

Mimari: CNN Encoder → Projected LSTM → CNN Decoder
Veri: 1902 klip pure_background (128x128)
Test: 478 klip (323N + 155A)
NPY'den yükleme → 30sn
"""

import subprocess
subprocess.run(['pip', 'install', 'torch', 'torchvision', 'scikit-learn', '-q'])

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, random, gc, json
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Image as IPImage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB\n")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

NPY_DIR = Path('/content/drive/MyDrive/archive/npy_cache')
SAVE_DIR = Path('/content/drive/MyDrive/archive/models')
SAVE_DIR.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# KONFİGÜRASYON
# ═══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    'frame_size': 128,
    'frames_per_clip': 16,
    'batch_size': 32,          # v2: 16→32 (A100 tensor core optimal)
    'val_split': 0.15,
    'epochs': 100,
    'lr': 1e-3,
    'min_lr': 1e-6,
    'lstm_hidden': 256,
    'lstm_layers': 2,
    'dropout': 0.3,
    'patience': 15,            # early stopping
    'weight_decay': 1e-5,      # v3: config'de açıkça belirtildi
    'encoder_filters': [32, 64, 128, 256],
}

print("Konfigürasyon:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print()

# ═══════════════════════════════════════════════════════════════════════════════
# VERİ YÜKLE
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("VERİ YÜKLEME")
print("=" * 60)
t0 = time.time()

bg_all = np.load(str(NPY_DIR / 'pure_bg_128.npy')).astype(np.float32)
test_all = np.load(str(NPY_DIR / 'test_128.npy')).astype(np.float32)
test_labels = np.load(str(NPY_DIR / 'test_labels_128.npy'))
# Klip isimlerini yükle (gün bazlı split için)
bg_names = np.load(str(NPY_DIR / 'test_names_128.npy'), allow_pickle=True)

# pure_bg için isim dosyası olup olmadığını kontrol et
# Eğer yoksa, pure_bg klasöründen doğrudan oku
bg_names_path = NPY_DIR / 'pure_bg_names_128.npy'
if bg_names_path.exists():
    bg_names = np.load(str(bg_names_path), allow_pickle=True)
    print(f"Pure BG isimleri yüklendi: {len(bg_names)} klip")
else:
    # İsim dosyası yoksa — Drive'daki klasör yapısından al
    bg_clip_dir = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/training/pure_background')
    if bg_clip_dir.exists():
        # Her klip klasörü: "20210115/clip_5_1234" gibi
        bg_names_list = sorted([d.parent.name + '/' + d.name for d in bg_clip_dir.iterdir() if d.is_dir()])
        # Eğer alt klasör yapısı farklıysa (yyyymmdd/clip_xxx)
        if len(bg_names_list) == 0:
            # Tek seviye klasör: "20210115_clip_5_1234" veya doğrudan klasör adları
            bg_names_list = sorted([d.name for d in bg_clip_dir.iterdir() if d.is_dir()])
        bg_names = np.array(bg_names_list)
        print(f"Pure BG isimleri Drive'dan okundu: {len(bg_names)} klip")
        # Kaydet (bir daha okumamak için)
        np.save(str(bg_names_path), bg_names)
    else:
        bg_names = None
        print("⚠️ Pure BG klip isimleri bulunamadı — fallback: rastgele split")

print(f"Pure BG: {bg_all.shape}")
print(f"Test: {test_all.shape} (N:{(test_labels==0).sum()} A:{(test_labels==1).sum()})")
print(f"Yükleme: {time.time()-t0:.1f}s")
print(f"RAM: {(bg_all.nbytes + test_all.nbytes)/1024**3:.2f}GB\n")

# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN/VAL SPLIT — GÜN BAZLI STRATİFİED
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("TRAIN/VAL SPLIT (Gün Bazlı Stratified)")
print("=" * 60)

def day_stratified_split(data, names, val_ratio=0.15, seed=42):
    """
    Ay bazında orantılı, gün bazında ayrık val split.

    Mantık:
    1. Klip isimlerinden ayı (yyyymm) ve günü (yyyymmdd) çıkar
    2. Her ay için: günleri listele, val_ratio kadar günü val'a ayır
    3. Aynı günün TÜM klipleri ya train'de ya val'da → data leakage yok

    Klip isim formatı beklentisi: "20210115/clip_5_1234" veya "20210115_clip_..."
    İlk 8 karakter = yyyymmdd
    """
    rng = random.Random(seed)

    # Gün → klip indeksleri mapping
    day_to_indices = defaultdict(list)
    month_to_days = defaultdict(set)

    for i, name in enumerate(names):
        # İlk 8 karakter: yyyymmdd
        name_str = str(name)
        day_str = name_str[:8]      # "20210115"
        month_str = day_str[:6]     # "202101"
        day_to_indices[day_str].append(i)
        month_to_days[month_str].add(day_str)

    # Her ay için günleri sırala
    val_indices = []
    train_indices = []

    print(f"\n  Ay bazlı dağılım:")
    for month in sorted(month_to_days.keys()):
        days = sorted(month_to_days[month])
        n_days = len(days)
        n_val_days = max(1, round(n_days * val_ratio))

        # Günleri karıştır ve val günlerini seç
        shuffled_days = days.copy()
        rng.shuffle(shuffled_days)
        val_days = set(shuffled_days[:n_val_days])
        train_days = set(shuffled_days[n_val_days:])

        month_val_clips = 0
        month_train_clips = 0
        for d in days:
            if d in val_days:
                val_indices.extend(day_to_indices[d])
                month_val_clips += len(day_to_indices[d])
            else:
                train_indices.extend(day_to_indices[d])
                month_train_clips += len(day_to_indices[d])

        print(f"    {month}: {n_days} gün → Train:{len(train_days)} ({month_train_clips} klip) "
              f"Val:{len(val_days)} ({month_val_clips} klip)")

    return np.array(train_indices), np.array(val_indices)


if bg_names is not None and len(bg_names) == len(bg_all):
    train_idx, val_idx = day_stratified_split(bg_all, bg_names, CONFIG['val_split'], SEED)
    train_data = bg_all[train_idx]
    val_data = bg_all[val_idx]
    actual_val_ratio = len(val_idx) / len(bg_all)
    print(f"\n  ✅ Gün bazlı split başarılı")
    print(f"  Train: {len(train_data)} | Val: {len(val_data)} | Oran: %{actual_val_ratio*100:.1f}")
    print(f"  ⚠️ Aynı gün klipleri train/val'a KARIŞMAZ (data leakage yok)")
else:
    # Fallback: rastgele split (isimler yoksa)
    print("  ⚠️ Klip isimleri eşleşmedi — rastgele split kullanılıyor")
    n_val = int(len(bg_all) * CONFIG['val_split'])
    perm = list(range(len(bg_all))); random.shuffle(perm)
    val_data = bg_all[perm[:n_val]]
    train_data = bg_all[perm[n_val:]]

del bg_all; gc.collect()
print(f"\n  Final — Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_all)}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# AUGMENTATION
# ═══════════════════════════════════════════════════════════════════════════════

class AugmentedDataset(Dataset):
    """
    Augmentation:
      - Yatay flip (%50)
      - Brightness shift (±10%)
      - Gaussian noise (sigma=0.02)
    """
    def __init__(self, data, augment=True):
        self.data = data
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        clip = self.data[idx].copy()  # [16, 128, 128]

        if self.augment:
            if random.random() > 0.5:
                clip = clip[:, :, ::-1].copy()

            shift = random.uniform(-0.1, 0.1)
            clip = np.clip(clip + shift, 0.0, 1.0)

            noise = np.random.normal(0, 0.02, clip.shape).astype(np.float32)
            clip = np.clip(clip + noise, 0.0, 1.0)

        tensor = torch.tensor(clip).unsqueeze(1)  # [16, 1, 128, 128]
        return tensor


class TestDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data; self.labels = labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]).unsqueeze(1), self.labels[idx]


train_dl = DataLoader(
    AugmentedDataset(train_data, augment=True),
    batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)
val_dl = DataLoader(
    AugmentedDataset(val_data, augment=False),
    batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True,
)
test_dl = DataLoader(
    TestDataset(test_all, test_labels),
    batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

print(f"DataLoader: batch={CONFIG['batch_size']} → "
      f"Train:{len(train_dl)} iter/epoch | Val:{len(val_dl)} | Test:{len(test_dl)}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL — CR-AE (Asıl boyut)
# ═══════════════════════════════════════════════════════════════════════════════

class CRAE(nn.Module):
    """
    Convolutional Recurrent Autoencoder

    Encoder: Conv2D 1→32→64→128→256, stride 2 (128→8)
    Project: Linear (256*8*8=16384) → 256
    LSTM: 2 katman, hidden=256
    Unproject: Linear 256 → 16384
    Decoder: ConvT2d 256→128→64→32→1, stride 2 (8→128)
    """
    def __init__(self, cfg):
        super().__init__()
        f = cfg['encoder_filters']  # [32, 64, 128, 256]
        self.spatial = f[-1] * 8 * 8  # 256*8*8 = 16384 (128→64→32→16→8)
        lstm_h = cfg['lstm_hidden']
        drop = cfg['dropout']

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, f[0], 3, stride=2, padding=1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0], f[1], 3, stride=2, padding=1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1], f[2], 3, stride=2, padding=1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2], f[3], 3, stride=2, padding=1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )

        # Projection
        self.proj = nn.Sequential(
            nn.Linear(self.spatial, lstm_h),
            nn.LeakyReLU(0.2),
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=lstm_h,
            hidden_size=lstm_h,
            num_layers=cfg['lstm_layers'],
            batch_first=True,
            dropout=drop if cfg['lstm_layers'] > 1 else 0,
        )

        # Unprojection
        self.unproj = nn.Sequential(
            nn.Linear(lstm_h, self.spatial),
            nn.LeakyReLU(0.2),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3], f[2], 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2], f[1], 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1], f[0], 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0], 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )

    def forward(self, clip):
        B, T, C, H, W = clip.shape

        # Encode
        x = self.encoder(clip.view(B*T, C, H, W))     # [B*T, 256, 8, 8]
        x = self.proj(x.view(B*T, -1)).view(B, T, -1) # [B, T, 256]

        # LSTM
        x, _ = self.lstm(x)                            # [B, T, 256]

        # Decode
        x = self.unproj(x.reshape(B*T, -1))            # [B*T, 16384]
        x = x.view(B*T, 256, 8, 8)
        x = self.decoder(x)                             # [B*T, 1, 128, 128]

        return x.view(B, T, 1, H, W)


model = CRAE(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params (~{n_params*4/1024/1024:.1f}MB)")

# Model'i compile et (PyTorch 2.0+ hız artışı)
if hasattr(torch, 'compile'):
    try:
        model = torch.compile(model)
        print("torch.compile aktif ✓")
    except:
        print("torch.compile başarısız, normal devam")

print()

# ═══════════════════════════════════════════════════════════════════════════════
# EĞİTİM
# ═══════════════════════════════════════════════════════════════════════════════

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=CONFIG['min_lr']
)

print("=" * 60)
print(f"EĞİTİM | Ep:{CONFIG['epochs']} B:{CONFIG['batch_size']} LR:{CONFIG['lr']}")
print(f"Early stopping patience: {CONFIG['patience']}")
print("=" * 60)

tl_h = []; vl_h = []; lr_h = []
best_val = float('inf')
patience_counter = 0
t_start = time.time()

for ep in range(CONFIG['epochs']):
    # ── Train ──
    model.train()
    el = nb = 0
    for batch in train_dl:
        clips = batch.to(device)
        optimizer.zero_grad()
        out = model(clips)
        loss = criterion(out, clips)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        el += loss.item(); nb += 1

    tl = el / max(nb, 1)
    tl_h.append(tl)

    # ── Val ──
    model.eval()
    vl = nv = 0
    with torch.no_grad():
        for batch in val_dl:
            clips = batch.to(device)
            vl += criterion(model(clips), clips).item(); nv += 1
    vl /= max(nv, 1)
    vl_h.append(vl)

    lr_now = optimizer.param_groups[0]['lr']
    lr_h.append(lr_now)
    scheduler.step()

    # Checkpoint
    if vl < best_val:
        best_val = vl
        patience_counter = 0
        torch.save({
            'epoch': ep,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': vl,
            'config': CONFIG,
        }, str(SAVE_DIR / 'crae_best.pth'))
    else:
        patience_counter += 1

    elapsed = time.time() - t_start
    eta = elapsed / (ep+1) * (CONFIG['epochs']-ep-1)
    print(f"  Ep{ep+1:>3}/{CONFIG['epochs']} | T:{tl:.6f} V:{vl:.6f} | "
          f"LR:{lr_now:.6f} | Best:{best_val:.6f} | "
          f"Pat:{patience_counter}/{CONFIG['patience']} | "
          f"{elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)

    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\n  ⚠️ Early stopping at epoch {ep+1}")
        break

    # Her 10 epoch'ta Drive'a kaydet
    if (ep+1) % 10 == 0:
        torch.save(model.state_dict(), str(SAVE_DIR / f'crae_ep{ep+1}.pth'))

total_time = time.time() - t_start
print(f"\nEğitim: {total_time/60:.1f}dk ({total_time/3600:.1f}saat)")

# ═══════════════════════════════════════════════════════════════════════════════
# EĞİTİM GRAFİKLERİ
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(tl_h, 'b-', label='Train', alpha=0.7)
axes[0].plot(vl_h, 'r-', label='Val', alpha=0.7)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title('Eğitim Eğrisi'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_h, 'g-')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')
axes[1].set_title('LR Schedule'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
# v2 FIX: Önce her iki yere kaydet, sonra close
plt.savefig('/content/training.png', dpi=150)
plt.savefig(str(SAVE_DIR / 'training_curve.png'), dpi=150)
plt.close()
display(IPImage(filename='/content/training.png'))

# ═══════════════════════════════════════════════════════════════════════════════
# RECONSTRUCTION ÖRNEKLERİ
# ═══════════════════════════════════════════════════════════════════════════════

print("\nReconstruction örnekleri:")
checkpoint = torch.load(str(SAVE_DIR / 'crae_best.pth'))
model.load_state_dict(checkpoint['model_state'])
model.eval()

# 4 val + 2 test normal + 2 test anomali
n_examples = 4
sample_val = torch.tensor(val_data[:n_examples]).unsqueeze(2).to(device)

test_n_idx = [i for i, l in enumerate(test_labels) if l == 0][:2]
test_a_idx = [i for i, l in enumerate(test_labels) if l == 1][:2]
sample_test_n = torch.tensor(test_all[test_n_idx]).unsqueeze(2).to(device)
sample_test_a = torch.tensor(test_all[test_a_idx]).unsqueeze(2).to(device)

with torch.no_grad():
    recon_val = model(sample_val)
    recon_tn = model(sample_test_n)
    recon_ta = model(sample_test_a)

fig, axes = plt.subplots(8, 4, figsize=(16, 32))
fig.suptitle('Reconstruction (Orig → Recon → Diff)', fontsize=16, y=0.995)

row = 0
for title, orig, recon in [
    ('Val', sample_val, recon_val),
    ('Test Normal', sample_test_n, recon_tn),
    ('Test Anomali', sample_test_a, recon_ta),
]:
    n = orig.shape[0]
    for i in range(n):
        o = orig[i, 8, 0].cpu().numpy()
        r = recon[i, 8, 0].cpu().numpy()
        d = np.abs(o - r)
        mse = np.mean((o - r)**2)

        axes[row, 0].imshow(o, cmap='gray', vmin=0, vmax=1)
        axes[row, 0].set_title(f'{title} {i+1} Orig'); axes[row, 0].axis('off')
        axes[row, 1].imshow(r, cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title(f'Recon'); axes[row, 1].axis('off')
        axes[row, 2].imshow(d, cmap='hot', vmin=0, vmax=0.3)
        axes[row, 2].set_title(f'Diff (MSE:{mse:.5f})'); axes[row, 2].axis('off')
        axes[row, 3].hist(d.flatten(), bins=50, color='red', alpha=0.7)
        axes[row, 3].set_title('Error Dist'); axes[row, 3].set_xlim(0, 0.5)
        row += 1

plt.tight_layout()
# v2 FIX: Önce her iki yere kaydet, sonra close
plt.savefig('/content/recon.png', dpi=100)
plt.savefig(str(SAVE_DIR / 'reconstruction.png'), dpi=100)
plt.close()
display(IPImage(filename='/content/recon.png'))

# ═══════════════════════════════════════════════════════════════════════════════
# TEST — ANOMALİ TESPİTİ
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("TEST — ANOMALİ TESPİTİ")
print("=" * 60)

model.eval()
scores = []; labels = []
with torch.no_grad():
    for clips, lbls in test_dl:
        clips = clips.to(device)
        out = model(clips)
        for i in range(clips.shape[0]):
            mse = torch.mean((clips[i] - out[i])**2).item()
            scores.append(mse)
            labels.append(lbls[i].item())

scores = np.array(scores); labels = np.array(labels)
ns = scores[labels==0]; als = scores[labels==1]

print(f"Normal MSE:  {ns.mean():.6f} (±{ns.std():.6f}) [min:{ns.min():.6f} max:{ns.max():.6f}]")
print(f"Anomali MSE: {als.mean():.6f} (±{als.std():.6f}) [min:{als.min():.6f} max:{als.max():.6f}]")
print(f"Oran: {als.mean()/ns.mean():.2f}x")
print(f"Ayrım: {(als.mean()-ns.mean())/ns.std():.2f} sigma")

from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              f1_score, confusion_matrix, classification_report)

auc = roc_auc_score(labels, scores)
print(f"\n*** AUC-ROC: {auc:.4f} ***")

# Optimal threshold
precisions, recalls, thresholds_pr = precision_recall_curve(labels, scores)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

preds = (scores >= best_threshold).astype(int)
cm = confusion_matrix(labels, preds)
tn, fp, fn, tp = cm.ravel()

print(f"\nOptimal Threshold: {best_threshold:.6f}")
print(f"F1: {best_f1:.4f}")
print(f"Precision: {tp/(tp+fp+1e-8):.4f}")
print(f"Recall: {tp/(tp+fn+1e-8):.4f}")
print(f"Specificity: {tn/(tn+fp+1e-8):.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN:{tn} FP:{fp}")
print(f"  FN:{fn} TP:{tp}")
print(f"\n{classification_report(labels, preds, target_names=['Normal', 'Anomali'])}")

# ═══════════════════════════════════════════════════════════════════════════════
# v3 EK METRİK 1: EER (Equal Error Rate)
# ═══════════════════════════════════════════════════════════════════════════════
# EER = FPR ile FNR'nin kesiştiği nokta (güvenlik sistemlerinde standart metrik)
# Düşük EER = daha iyi (ideal: 0.0)

fpr_roc, tpr_roc, thresholds_roc = roc_curve(labels, scores)
fnr_roc = 1 - tpr_roc
# EER: FPR ≈ FNR olan noktayı bul (interpolasyon ile)
eer_idx = np.nanargmin(np.abs(fpr_roc - fnr_roc))
eer = float((fpr_roc[eer_idx] + fnr_roc[eer_idx]) / 2)
eer_threshold = float(thresholds_roc[eer_idx]) if eer_idx < len(thresholds_roc) else float('nan')

print(f"{'='*60}")
print(f"EER (Equal Error Rate)")
print(f"{'='*60}")
print(f"  EER              : {eer:.4f} ({eer*100:.2f}%)")
print(f"  EER Threshold    : {eer_threshold:.6f}")
print(f"  (FPR={fpr_roc[eer_idx]:.4f}, FNR={fnr_roc[eer_idx]:.4f} bu noktada)")

# ═══════════════════════════════════════════════════════════════════════════════
# v3 EK METRİK 2: SEASONAL ROBUSTNESS SCORE
# ═══════════════════════════════════════════════════════════════════════════════
# Eğitim: Kış 2021 (Ocak-Mart) → Test: Yaz 2020-2021 (Mayıs-Ağustos, Nisan)
# Her test ayı için ayrı AUC hesaplayarak mevsimsel dayanıklılığı ölç

test_names_path = NPY_DIR / 'test_names_128.npy'
if test_names_path.exists():
    test_names = np.load(str(test_names_path), allow_pickle=True)

    # Ay bazlı grupla
    month_data = defaultdict(lambda: {'scores': [], 'labels': []})
    for i, name in enumerate(test_names):
        month_str = str(name)[:6]  # "202005", "202006", etc.
        month_data[month_str]['scores'].append(scores[i])
        month_data[month_str]['labels'].append(labels[i])

    print(f"\n{'='*60}")
    print(f"SEASONAL ROBUSTNESS SCORE")
    print(f"{'='*60}")
    print(f"  Eğitim mevsimi  : Kış 2021 (202101-202103)")
    print(f"  Test mevsimi    : Yaz 2020-2021")
    print(f"  {'Ay':<10} {'N':>5} {'A':>5} {'AUC':>8} {'Durum'}")
    print(f"  {'-'*45}")

    monthly_aucs = {}
    for month in sorted(month_data.keys()):
        m_scores = np.array(month_data[month]['scores'])
        m_labels = np.array(month_data[month]['labels'])
        n_normal = (m_labels == 0).sum()
        n_anomaly = (m_labels == 1).sum()

        if n_anomaly > 0 and n_normal > 0:
            m_auc = roc_auc_score(m_labels, m_scores)
            monthly_aucs[month] = m_auc
            status = "✅" if m_auc >= 0.80 else "⚠️" if m_auc >= 0.65 else "❌"
            print(f"  {month:<10} {n_normal:>5} {n_anomaly:>5} {m_auc:>8.4f} {status}")
        else:
            print(f"  {month:<10} {n_normal:>5} {n_anomaly:>5} {'N/A':>8} (tek sınıf)")

    if monthly_aucs:
        auc_values = list(monthly_aucs.values())
        seasonal_robustness = {
            'overall_auc': float(auc),
            'monthly_aucs': {k: float(v) for k, v in monthly_aucs.items()},
            'mean_monthly_auc': float(np.mean(auc_values)),
            'std_monthly_auc': float(np.std(auc_values)),
            'min_monthly_auc': float(np.min(auc_values)),
            'max_monthly_auc': float(np.max(auc_values)),
            'drift_spread': float(np.max(auc_values) - np.min(auc_values)),
        }
        print(f"\n  Ortalama AUC    : {seasonal_robustness['mean_monthly_auc']:.4f} ± {seasonal_robustness['std_monthly_auc']:.4f}")
        print(f"  Min/Max AUC     : {seasonal_robustness['min_monthly_auc']:.4f} / {seasonal_robustness['max_monthly_auc']:.4f}")
        print(f"  Drift Yayılımı  : {seasonal_robustness['drift_spread']:.4f}")
        print(f"  (Düşük yayılım = mevsime dayanıklı model)")
    else:
        seasonal_robustness = None
        print("  ⚠️ Ay bazlı AUC hesaplanamadı")
else:
    seasonal_robustness = None
    print("\n  ⚠️ test_names_128.npy bulunamadı — seasonal score atlandı")

# ═══════════════════════════════════════════════════════════════════════════════
# v3 EK METRİK 3: INFERENCE LATENCY
# ═══════════════════════════════════════════════════════════════════════════════
# Tez hedefi: < 30ms per frame
# Ölçüm: GPU warm-up + 50 klip üzerinde ortalama süre

print(f"\n{'='*60}")
print(f"INFERENCE LATENCY")
print(f"{'='*60}")

model.eval()
# Warm-up (ilk inference yavaştır — CUDA kernel başlatma)
dummy = torch.randn(1, 16, 1, 128, 128).to(device)
with torch.no_grad():
    for _ in range(5):
        _ = model(dummy)
if device.type == 'cuda':
    torch.cuda.synchronize()

# Gerçek ölçüm — 50 klip tek tek
n_latency_test = 50
latencies_clip = []
latencies_frame = []

sample_clips = torch.tensor(test_all[:n_latency_test]).unsqueeze(2).to(device)

with torch.no_grad():
    for i in range(n_latency_test):
        single_clip = sample_clips[i:i+1]  # [1, 16, 1, 128, 128]
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0_inf = time.time()
        _ = model(single_clip)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t1_inf = time.time()

        clip_ms = (t1_inf - t0_inf) * 1000
        frame_ms = clip_ms / 16  # 16 frame per clip
        latencies_clip.append(clip_ms)
        latencies_frame.append(frame_ms)

latencies_clip = np.array(latencies_clip)
latencies_frame = np.array(latencies_frame)

print(f"  Klip başına     : {latencies_clip.mean():.2f}ms ± {latencies_clip.std():.2f}ms")
print(f"  Frame başına    : {latencies_frame.mean():.2f}ms ± {latencies_frame.std():.2f}ms")
print(f"  Throughput      : {1000/latencies_frame.mean():.1f} FPS")
target_met = latencies_frame.mean() < 30
print(f"  Tez hedefi      : < 30ms/frame → {'✅ BAŞARILI' if target_met else '❌ AŞILDI'}")
print(f"  (A100 GPU üzerinde — edge device'da farklı olacaktır)")

inference_latency = {
    'clip_ms_mean': float(latencies_clip.mean()),
    'clip_ms_std': float(latencies_clip.std()),
    'frame_ms_mean': float(latencies_frame.mean()),
    'frame_ms_std': float(latencies_frame.std()),
    'throughput_fps': float(1000/latencies_frame.mean()),
    'target_30ms_met': bool(target_met),
    'device': str(device),
    'gpu_name': torch.cuda.get_device_name() if device.type == 'cuda' else 'CPU',
}

# ═══════════════════════════════════════════════════════════════════════════════
# TEST GRAFİKLERİ
# ═══════════════════════════════════════════════════════════════════════════════

fpr, tpr, _ = roc_curve(labels, scores)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ROC
axes[0,0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc:.4f}')
axes[0,0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR')
axes[0,0].set_title('ROC Eğrisi'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Score dağılımı
axes[0,1].hist(ns, bins=40, alpha=0.6, label=f'Normal (n={len(ns)})', color='green')
axes[0,1].hist(als, bins=40, alpha=0.6, label=f'Anomali (n={len(als)})', color='red')
axes[0,1].axvline(best_threshold, color='black', ls='--', lw=2, label=f'Threshold={best_threshold:.5f}')
axes[0,1].set_xlabel('Reconstruction Error (MSE)'); axes[0,1].set_ylabel('Klip Sayısı')
axes[0,1].set_title('Score Dağılımı'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Precision-Recall-F1
axes[1,0].plot(thresholds_pr, precisions[:-1], 'b-', label='Precision')
axes[1,0].plot(thresholds_pr, recalls[:-1], 'r-', label='Recall')
axes[1,0].plot(thresholds_pr, f1_scores[:-1], 'g-', lw=2, label='F1')
axes[1,0].axvline(best_threshold, color='black', ls='--')
axes[1,0].set_xlabel('Threshold'); axes[1,0].set_title('Precision / Recall / F1')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# Confusion matrix
im = axes[1,1].imshow(cm, cmap='Blues')
axes[1,1].set_xticks([0,1]); axes[1,1].set_yticks([0,1])
axes[1,1].set_xticklabels(['Normal', 'Anomali'])
axes[1,1].set_yticklabels(['Normal', 'Anomali'])
axes[1,1].set_xlabel('Predicted'); axes[1,1].set_ylabel('Actual')
axes[1,1].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[1,1].text(j, i, f'{cm[i,j]}', ha='center', va='center', fontsize=20,
                       color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.tight_layout()
# v2 FIX: Önce her iki yere kaydet, sonra close
plt.savefig('/content/test_results.png', dpi=150)
plt.savefig(str(SAVE_DIR / 'test_results.png'), dpi=150)
plt.close()
display(IPImage(filename='/content/test_results.png'))

# ═══════════════════════════════════════════════════════════════════════════════
# SONUÇLARI KAYDET
# ═══════════════════════════════════════════════════════════════════════════════

results = {
    'version': 'v3',
    'changes': [
        'Day-stratified val split (no data leakage)',
        'Batch size 16→32',
        'plt.savefig ordering fix',
        'weight_decay in CONFIG',
        'EER metric added',
        'Seasonal Robustness Score added',
        'Inference Latency measurement added',
    ],
    'config': CONFIG,
    'n_params': n_params,
    'train_clips': len(train_data),
    'val_clips': len(val_data),
    'test_clips': len(test_all),
    'test_normal': int((test_labels==0).sum()),
    'test_anomaly': int((test_labels==1).sum()),
    'best_val_loss': float(best_val),
    'best_epoch': int(checkpoint['epoch']) + 1,
    'total_epochs': len(tl_h),
    'training_time_minutes': round(total_time/60, 1),
    'normal_mse_mean': float(ns.mean()),
    'normal_mse_std': float(ns.std()),
    'anomaly_mse_mean': float(als.mean()),
    'anomaly_mse_std': float(als.std()),
    'separation_ratio': float(als.mean()/ns.mean()),
    'separation_sigma': float((als.mean()-ns.mean())/ns.std()),
    'auc_roc': float(auc),
    'eer': float(eer),
    'eer_threshold': float(eer_threshold),
    'best_f1': float(best_f1),
    'best_threshold': float(best_threshold),
    'precision': float(tp/(tp+fp+1e-8)),
    'recall': float(tp/(tp+fn+1e-8)),
    'specificity': float(tn/(tn+fp+1e-8)),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'seasonal_robustness': seasonal_robustness,
    'inference_latency': inference_latency,
    'train_losses': [float(x) for x in tl_h],
    'val_losses': [float(x) for x in vl_h],
}

with open(str(SAVE_DIR / 'crae_results_v3.json'), 'w') as f:
    json.dump(results, f, indent=2)

# ═══════════════════════════════════════════════════════════════════════════════
# ÖZET
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("CR-AE ASIL MODEL — SONUÇ (v3)")
print("=" * 60)
print(f"  Params         : {n_params:,}")
print(f"  Epochs         : {len(tl_h)} (best: ep{checkpoint['epoch']+1})")
print(f"  Eğitim süresi  : {total_time/60:.1f}dk")
print(f"  Best val loss  : {best_val:.6f}")
print(f"  Normal MSE     : {ns.mean():.6f} ± {ns.std():.6f}")
print(f"  Anomali MSE    : {als.mean():.6f} ± {als.std():.6f}")
print(f"  Fark           : {als.mean()/ns.mean():.2f}x ({(als.mean()-ns.mean())/ns.std():.1f}σ)")
print(f"  ─────────────────────────────")
print(f"  AUC-ROC        : {auc:.4f}")
print(f"  EER            : {eer:.4f} ({eer*100:.2f}%)")
print(f"  F1             : {best_f1:.4f}")
print(f"  Precision      : {tp/(tp+fp+1e-8):.4f}")
print(f"  Recall         : {tp/(tp+fn+1e-8):.4f}")
print(f"  Specificity    : {tn/(tn+fp+1e-8):.4f}")
print(f"  ─────────────────────────────")
if seasonal_robustness:
    print(f"  Seasonal AUC   : {seasonal_robustness['mean_monthly_auc']:.4f} ± {seasonal_robustness['std_monthly_auc']:.4f}")
    print(f"  Drift Spread   : {seasonal_robustness['drift_spread']:.4f}")
print(f"  Latency/frame  : {inference_latency['frame_ms_mean']:.2f}ms ({'✅' if inference_latency['target_30ms_met'] else '❌'} <30ms)")
print(f"  Throughput     : {inference_latency['throughput_fps']:.0f} FPS")
print(f"  ─────────────────────────────")
print(f"  Val split      : Gün bazlı stratified (leakage-free)")
print(f"  Batch size     : {CONFIG['batch_size']}")
print(f"  Model          : {SAVE_DIR / 'crae_best.pth'}")
print(f"  Sonuçlar       : {SAVE_DIR / 'crae_results_v3.json'}")
print(f"  Grafikler      : {SAVE_DIR}")
print("=" * 60)

if auc >= 0.85:
    print("\n✅ Çok iyi! 3D-CNN aşamasına geçilebilir.")
elif auc >= 0.70:
    print("\n⚠️ İyi ama iyileştirilebilir. Hiperparametre ayarı dene.")
else:
    print("\n❌ Düşük. Mimari/veri değişikliği gerekebilir.")

In [ ]:
"""
MODEL HATA AYIKLAMA VE GÖRSELLEŞTİRME (V4 - KESİN DİZİN YOLU)
============================================================
Google Colab — A100

Düzeltme:
- Orijinal .jpg dosyalarının kök dizini tam paylaşılan klasör ağacına
  (Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final)
  göre güncellendi.
"""

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib
matplotlib.use('Agg') # Colab'de UI sorunlarını önlemek için
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob
from sklearn.metrics import precision_recall_curve
from IPython.display import display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

# Proje Dizini Tanımlamaları
ARCHIVE_DIR = Path('/content/drive/MyDrive/archive')
NPY_DIR = ARCHIVE_DIR / 'npy_cache'
MODEL_DIR = ARCHIVE_DIR / 'models'

# YENİ VE KESİN DİZİN YOLU:
TESTING_FINAL_DIR = ARCHIVE_DIR / 'Data_Subset_Autoencoders_Anomaly_Detectors' / 'testing' / 'testing_final'

# ═══════════════════════════════════════════════════════════════════════════════
# 1. VERİ YÜKLEME
# ═══════════════════════════════════════════════════════════════════════════════
print("Veriler NPY önbelleğinden yükleniyor...")
test_raw = np.load(str(NPY_DIR / 'test_64.npy')).astype(np.float32)
test_labels = np.load(str(NPY_DIR / 'test_labels_64.npy'))
test_names = np.load(str(NPY_DIR / 'test_names_64.npy'), allow_pickle=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. MODEL MİMARİSİ (BASELINE)
# ═══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg['encoder_filters']
        fs = cfg.get('frame_size', 128)
        final_s = fs // 16
        self.spatial = f[-1] * final_s * final_s
        self.final_s = final_s
        self.last_f = f[-1]
        lstm_h = cfg['lstm_hidden']
        drop = cfg['dropout']

        self.encoder = nn.Sequential(
            nn.Conv2d(1, f[0], 3, 2, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0], f[1], 3, 2, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1], f[2], 3, 2, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2], f[3], 3, 2, 1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm = nn.LSTM(lstm_h, lstm_h, num_layers=cfg['lstm_layers'],
                            batch_first=True, dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3], f[2], 3, 2, 1, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2], f[1], 3, 2, 1, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1], f[0], 3, 2, 1, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0], 1, 3, 2, 1, 1), nn.Sigmoid(),
        )

    def forward(self, clip):
        B, T, C, H, W = clip.shape
        x = self.encoder(clip.view(B*T, C, H, W))
        x = self.proj(x.view(B*T, -1)).view(B, T, -1)
        x, _ = self.lstm(x)
        x = self.unproj(x.reshape(B*T, -1))
        x = x.view(B*T, self.last_f, self.final_s, self.final_s)
        x = self.decoder(x)
        return x.view(B, T, 1, H, W)

# ═══════════════════════════════════════════════════════════════════════════════
# 3. ÇIKARIM (INFERENCE) VE VERİ TOPLAMA
# ═══════════════════════════════════════════════════════════════════════════════
print("Model diskten yükleniyor...")
checkpoint = torch.load(str(MODEL_DIR / 'crae_best.pth'), map_location=device)
config = checkpoint['config']
model = CRAE(config).to(device)
cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model_state'].items()}
model.load_state_dict(cleaned_state)
model.eval()
print("Model yüklendi. Çıkarım (Inference) başlatılıyor...\n")

dl = DataLoader(TensorDataset(torch.tensor(test_raw).unsqueeze(2), torch.tensor(test_labels)), batch_size=16)

all_scores = []
all_inputs = []      # NPY'den gelen 64x64 klipler (Yedek olarak tutulur)
all_model_ins = []   # Modele giren 128x128 klipler
all_recons = []      # Modelin çıktısı 128x128 klipler

with torch.no_grad():
    for clips, labels_batch in dl:
        clips = clips.to(device)
        B, T, C, H, W = clips.shape

        expected_size = model.final_s * 16
        if H != expected_size or W != expected_size:
            clips_resized = F.interpolate(
                clips.view(B * T, C, H, W),
                size=(expected_size, expected_size),
                mode='bilinear',
                align_corners=False
            ).view(B, T, C, expected_size, expected_size)
        else:
            clips_resized = clips

        out = model(clips_resized)

        for i in range(B):
            score = torch.mean((clips_resized[i] - out[i])**2).item()
            all_scores.append(score)

            all_inputs.append(clips[i].cpu().numpy().squeeze())
            all_model_ins.append(clips_resized[i].cpu().numpy().squeeze())
            all_recons.append(out[i].cpu().numpy().squeeze())

all_scores = np.array(all_scores)

# Eşik Değerini (Threshold) Bulma
prec, rec, thr = precision_recall_curve(test_labels, all_scores)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
best_idx = np.argmax(f1s)
best_threshold = thr[best_idx]

print(f"--- ANALİZ SONUÇLARI ---")
print(f"Optimum Anomali Eşiği (Threshold): {best_threshold:.5f}")

preds = (all_scores >= best_threshold).astype(int)

fp_indices = np.where((preds == 1) & (test_labels == 0))[0]
fn_indices = np.where((preds == 0) & (test_labels == 1))[0]
tn_indices = np.where((preds == 0) & (test_labels == 0))[0]

print(f"Yanlış Alarmlar (Normal Ama Anomali Denmiş): {len(fp_indices)} adet")
print(f"Kaçırılan Anomaliler (Anomali Ama Normal Denmiş): {len(fn_indices)} adet")
print(f"Doğru Bilinen Normaller: {len(tn_indices)} adet\n")
print("="*80)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. GÖRSELLEŞTİRME FONKSİYONU
# ═══════════════════════════════════════════════════════════════════════════════
def plot_clip_error(idx, category_title):
    # clip_name örneğin '20210429/normal/clip_0_0992' formatında olmalıdır.
    clip_name = str(test_names[idx])
    score = all_scores[idx]

    clip_in_64 = all_inputs[idx]
    clip_model_in = all_model_ins[idx]
    clip_recon = all_recons[idx]

    # Hatanın en yüksek olduğu frame'i bul
    mse_per_frame = np.mean((clip_model_in - clip_recon)**2, axis=(1, 2))
    worst_t = np.argmax(mse_per_frame)

    img_model_in = clip_model_in[worst_t]
    img_recon = clip_recon[worst_t]
    error_map = (img_model_in - img_recon)**2

    # --- Orijinal Dosyayı Diskten Okuma ---
    frame_filename = f"frame_{worst_t+1:03d}.jpg"

    # Yeni ve kesin dizin yoluna göre tam dosya yolu:
    search_pattern = str(TESTING_FINAL_DIR / clip_name / frame_filename)

    # Yine de alt klasör kırılımlarında bir farklılık varsa diye glob ile jokerli arama
    found_files = glob.glob(str(TESTING_FINAL_DIR / '**' / clip_name / frame_filename), recursive=True)

    img_in_raw = None
    raw_title = "Orijinal Görsel (Drive'dan)"

    try:
        if len(found_files) > 0:
            raw_img_path = found_files[0]
            img_in_raw = mpimg.imread(raw_img_path)
        else:
            # Doğrudan path'i deneyelim
            img_in_raw = mpimg.imread(search_pattern)
    except Exception as e:
        print(f"HATA: Orijinal JPG bulunamadı! Aranan yol: {search_pattern}")
        print(f"Sistem Hatası: {e}")
        # Hata durumunda 64x64 numpy array'e geri dön (kodun çökmesini önlemek için)
        img_in_raw = clip_in_64[worst_t]
        raw_title = "GÖRSEL BULUNAMADI (64x64 Yedek)"

    # --- Kopyalanabilir Metin Çıktısı ---
    print(f"[{category_title}]")
    print(f"Dosya Yolu: {clip_name}")
    print(f"Hatalı Frame: {worst_t+1}/16 | Skor: {score:.5f} (Eşik: {best_threshold:.5f})")

    # --- Görsel Çıktı ---
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.patch.set_facecolor('white')

    # Orijinal Görüntü
    axes[0].imshow(img_in_raw, cmap='gray')
    axes[0].set_title(raw_title, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_model_in, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Modele Giren (128x128 Interpolated)", fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(img_recon, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title("Modelin Çizdiği (Reconstructed)", fontweight='bold')
    axes[2].axis('off')

    # Hata Haritası
    im = axes[3].imshow(error_map, cmap='inferno')
    axes[3].set_title("Hata Haritası (Isı Farkı)", fontweight='bold')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    # Ana Başlık
    fig.suptitle(f"{category_title}", fontsize=14, fontweight='bold', color='darkred')

    plt.tight_layout()
    display(fig)
    plt.close(fig)
    print("-" * 80 + "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. LİSTELEME
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "▼"*80)
print("YANLIŞ ALARMLAR (FALSE POSITIVES)")
print("Aslında her şey normal, ancak model 'Burada Anomali var' demiş.")
print("▼"*80 + "\n")
for idx in fp_indices:
    plot_clip_error(idx, "NORMAL Klipti AMA Model ANOMALİ Dedi (Yanlış Alarm)")

print("\n" + "▼"*80)
print("KAÇIRILAN ANOMALİLER (FALSE NEGATIVES)")
print("Aslında anomali var (örneğin insan), ancak model fark edemeyip 'Normal' demiş.")
print("▼"*80 + "\n")
for idx in fn_indices:
    plot_clip_error(idx, "ANOMALİ Klipti AMA Model NORMAL Dedi (Kaçırılan Anomali)")

print("\n" + "▼"*80)
print("REFERANS İÇİN: BAZI DOĞRU BİLİNEN NORMALLER (TRUE NEGATIVES)")
print("Hem klipler normal, hem de model 'Normal' diyerek doğru bilmiş.")
print("▼"*80 + "\n")
# Tüm doğru bilinenleri basmak çok uzun süreceği için rastgele 5 adet seçiyoruz.
sample_tn = np.random.choice(tn_indices, min(5, len(tn_indices)), replace=False)
for idx in sample_tn:
    plot_clip_error(idx, "NORMAL Klipti VE Model Doğru Bir Şekilde NORMAL Dedi")

print("\nBÜTÜN KLİPLERİN ANALİZİ TAMAMLANDI.")

In [ ]:
"""
crae_full_evaluation.py
========================
1. Winter + Summer TAR → RAM → test_all.tar → Drive
2. Winter → Summer → All sırayla test (TAR'dan oku, 128x128 resize, NPY RAM'de tut)
3. Her set için: AUC-ROC, F1, Precision, Recall, Confusion Matrix, grafikler
4. Recall 90+ eşiği için ayrı confusion matrix (3 set yan yana)
5. Grafikler + NPY'ler locale/RAM'de — Drive'a YAZILMAZ (onay beklenir)

Düzeltmeler:
- frame sıralama: hem frame000.jpg hem frame_000.jpg formatı desteklenir
- ALL TAR prefix: winter/ ve summer/ alt klasörleri doğru tanınır
"""

from google.colab import drive
drive.mount('/content/drive')

import torch, tarfile, shutil, subprocess, time, warnings
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score)
from IPython.display import display, Image as IPImage
import cv2
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ══════════════════════════════════════════════════════════════════════════════
#  YOLLAR
# ══════════════════════════════════════════════════════════════════════════════
ARCHIVE    = Path('/content/drive/MyDrive/archive')
NPY_DIR    = ARCHIVE / 'npy_cache'
MODEL_PATH = ARCHIVE / 'models/crae_best.pth'
TAR_WINTER = ARCHIVE / 'test_winter.tar'
TAR_SUMMER = ARCHIVE / 'test_summer.tar'
TAR_ALL    = ARCHIVE / 'test_all.tar'

TMP        = Path('/tmp/crae_eval')
TMP.mkdir(parents=True, exist_ok=True)

LOCAL_OUT  = Path('/content/eval_results')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

FRAME_SIZE = 128
BATCH_SIZE = 32
N_WORKERS  = 32

npy_cache_ram = {}

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL MİMARİSİ
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f       = cfg['encoder_filters']
        fs      = cfg.get('frame_size', 128)
        final_s = fs // 16
        self.spatial = f[-1] * final_s * final_s
        self.final_s = final_s
        self.last_f  = f[-1]
        lstm_h  = cfg['lstm_hidden']
        drop    = cfg['dropout']

        self.encoder = nn.Sequential(
            nn.Conv2d(1,    f[0], 3, 2, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0], f[1], 3, 2, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1], f[2], 3, 2, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2], f[3], 3, 2, 1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm   = nn.LSTM(lstm_h, lstm_h, num_layers=cfg['lstm_layers'],
                              batch_first=True,
                              dropout=drop if cfg['lstm_layers'] > 1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3], f[2], 3, 2, 1, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2], f[1], 3, 2, 1, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1], f[0], 3, 2, 1, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0], 1,    3, 2, 1, 1), nn.Sigmoid(),
        )

    def forward(self, clip):
        B, T, C, H, W = clip.shape
        x = self.encoder(clip.view(B*T, C, H, W))
        x = self.proj(x.view(B*T, -1)).view(B, T, -1)
        x, _ = self.lstm(x)
        x = self.unproj(x.reshape(B*T, -1))
        x = x.view(B*T, self.last_f, self.final_s, self.final_s)
        return self.decoder(x).view(B, T, 1, H, W)

# ══════════════════════════════════════════════════════════════════════════════
#  ADIM 1 — test_all.tar
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ADIM 1: test_all.tar")
print("="*60)

if TAR_ALL.exists():
    print(f"  Zaten var: {TAR_ALL}")
else:
    TMP_MERGE = TMP / 'merge'
    TMP_MERGE.mkdir(parents=True, exist_ok=True)
    for lbl, tp in [('winter', TAR_WINTER), ('summer', TAR_SUMMER)]:
        print(f"  {lbl} aciliyor...")
        t0 = time.time()
        subprocess.run(f"tar -xf '{tp}' -C '{TMP_MERGE}'", shell=True, check=True)
        print(f"    {time.time()-t0:.0f}sn")
    TMP_ALL_TAR = TMP / 'test_all.tar'
    print("  all TAR olusturuluyor...")
    t0 = time.time()
    subprocess.run(f"tar -cf '{TMP_ALL_TAR}' -C '{TMP_MERGE}' .", shell=True, check=True)
    print(f"    TAR: {time.time()-t0:.0f}sn")
    print("  Drive'a yaziliyor...")
    t0 = time.time()
    subprocess.run(f"cp '{TMP_ALL_TAR}' '{TAR_ALL}'", shell=True, check=True)
    print(f"    CP: {time.time()-t0:.0f}sn  ({TAR_ALL.stat().st_size/1e6:.0f} MB)")
    shutil.rmtree(str(TMP_MERGE))

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL YÜKLEME
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("MODEL YUKLENIYOR")
print("="*60)
checkpoint    = torch.load(str(MODEL_PATH), map_location=device)
config        = checkpoint['config']
model         = CRAE(config).to(device)
cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model_state'].items()}
model.load_state_dict(cleaned_state)
model.eval()
print(f"  Yuklendi  epoch={checkpoint.get('epoch','?')}  config={config}")

# ══════════════════════════════════════════════════════════════════════════════
#  VERİ YÜKLEME
#  DÜZELTİLDİ: frame sıralama sayısal (frame_1 < frame_2 < ... < frame_16)
#              her iki format desteklenir: frame000.jpg / frame_001.jpg
# ══════════════════════════════════════════════════════════════════════════════
def load_clip(clip_dir: Path):
    frames = sorted(
        clip_dir.glob("frame*.jpg"),
        key=lambda p: int(''.join(filter(str.isdigit, p.stem)))
    )
    if len(frames) < 16:
        return None
    imgs = []
    for fp in frames[:16]:
        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_LINEAR)
        imgs.append(img.astype(np.float16) / 255.0)
    return np.stack(imgs)   # (16, 128, 128) float16

def load_from_tar(tar_path: Path, tmp_dir: Path):
    if tmp_dir.exists():
        shutil.rmtree(str(tmp_dir))
    tmp_dir.mkdir(parents=True, exist_ok=True)

    print(f"  TAR aciliyor: {tar_path.name}")
    t0 = time.time()
    subprocess.run(f"tar -xf '{tar_path}' -C '{tmp_dir}'", shell=True, check=True)
    print(f"    Acma: {time.time()-t0:.0f}sn")

    # DÜZELTİLDİ: rglob ile tüm derinliklerde ara
    # winter/YYYYMMDD/normal/clip_xxx  veya  YYYYMMDD/normal/clip_xxx
    # parent.name = normal|anomaly, parent.parent.name = YYYYMMDD (8 haneli)
    clip_list = []
    for p in sorted(tmp_dir.rglob('*')):
        if (p.is_dir()
                and p.parent.name in ('normal', 'anomaly')
                and p.parent.parent.name.isdigit()
                and len(p.parent.parent.name) == 8):
            label = 1 if p.parent.name == 'anomaly' else 0
            clip_list.append((p, label))

    print(f"  {len(clip_list)} klip bulundu, {N_WORKERS} thread ile yukleniyor...")
    results = [None] * len(clip_list)

    def _load(idx):
        arr = load_clip(clip_list[idx][0])
        return idx, arr, clip_list[idx][1], clip_list[idx][0].name

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(_load, i): i for i in range(len(clip_list))}
        done = 0
        for f in as_completed(futs):
            results[futs[f]] = f.result()
            done += 1
            if done % 100 == 0:
                print(f"    {done}/{len(clip_list)}")

    valid = [(r[1], r[2], r[3]) for r in results if r[1] is not None]
    skipped = len(clip_list) - len(valid)
    if skipped:
        print(f"    [UYARI] {skipped} klip okunamadi (16'dan az frame veya bozuk)")

    clips_np  = np.stack([v[0] for v in valid]).astype(np.float16)
    labels_np = np.array([v[1] for v in valid], dtype=np.int8)
    names_np  = np.array([v[2] for v in valid])
    print(f"    Yukleme: {time.time()-t0:.0f}sn  "
          f"Normal:{(labels_np==0).sum()}  Anomali:{(labels_np==1).sum()}  "
          f"RAM:{clips_np.nbytes/1e6:.0f}MB")
    return clips_np, labels_np, names_np

# ══════════════════════════════════════════════════════════════════════════════
#  INFERENCE
# ══════════════════════════════════════════════════════════════════════════════
def run_inference(clips_np, labels_np):
    tensor_clips  = torch.tensor(clips_np.astype(np.float32)).unsqueeze(2)
    tensor_labels = torch.tensor(labels_np.astype(np.int64))
    dl = DataLoader(TensorDataset(tensor_clips, tensor_labels),
                    batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    expected = model.final_s * 16
    scores, lbls = [], []

    with torch.no_grad():
        for clips_b, labels_b in dl:
            clips_b = clips_b.to(device, non_blocking=True)
            B, T, C, H, W = clips_b.shape
            if H != expected or W != expected:
                clips_b = F.interpolate(
                    clips_b.view(B*T, C, H, W),
                    size=(expected, expected), mode='bilinear', align_corners=False
                ).view(B, T, C, expected, expected)
            out  = model(clips_b)
            mse  = torch.mean((clips_b - out)**2, dim=[1,2,3,4])
            scores.extend(mse.cpu().numpy().tolist())
            lbls.extend(labels_b.numpy().tolist())

    return np.array(scores), np.array(lbls)

# ══════════════════════════════════════════════════════════════════════════════
#  METRİKLER
# ══════════════════════════════════════════════════════════════════════════════
def compute_metrics(scores, labels, tag):
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thresh_roc = roc_curve(labels, scores)
    fnr   = 1 - tpr
    eer_i = np.nanargmin(np.abs(fpr - fnr))
    eer   = float((fpr[eer_i] + fnr[eer_i]) / 2)

    prec_c, rec_c, thresh_pr = precision_recall_curve(labels, scores)
    f1s    = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    best_i = np.argmax(f1s)
    opt_thresh = thresh_pr[best_i]

    preds = (scores >= opt_thresh).astype(int)
    cm    = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    rec90_thresh = rec90_cm = None
    for t in np.sort(thresh_roc)[::-1]:
        p = (scores >= t).astype(int)
        if recall_score(labels, p, zero_division=0) >= 0.90:
            rec90_thresh = t
            rec90_cm = confusion_matrix(labels, p)
            break

    normal_s  = scores[labels == 0]
    anomaly_s = scores[labels == 1]

    return dict(
        tag=tag, auc=auc, eer=eer,
        best_f1=f1s[best_i], opt_thresh=opt_thresh,
        prec=precision_score(labels, preds, zero_division=0),
        rec=recall_score(labels, preds, zero_division=0),
        tn=tn, fp=fp, fn=fn, tp=tp,
        fpr=fpr, tpr=tpr, eer_i=eer_i,
        prec_c=prec_c, rec_c=rec_c,
        normal_s=normal_s, anomaly_s=anomaly_s,
        sep_ratio=anomaly_s.mean()/(normal_s.mean()+1e-8),
        rec90_thresh=rec90_thresh, rec90_cm=rec90_cm,
        scores=scores, labels=labels,
        n_normal=int((labels==0).sum()), n_anomaly=int((labels==1).sum())
    )

# ══════════════════════════════════════════════════════════════════════════════
#  GRAFİKLER
# ══════════════════════════════════════════════════════════════════════════════
def plot_set(m):
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle(f'CR-AE — {m["tag"]}  '
                 f'(Normal:{m["n_normal"]}  Anomali:{m["n_anomaly"]})',
                 fontsize=17, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(m['fpr'], m['tpr'], 'b-', lw=2, label=f"AUC={m['auc']:.4f}")
    ax.plot([0,1],[0,1],'k--', alpha=0.3)
    ax.plot(m['fpr'][m['eer_i']], m['tpr'][m['eer_i']], 'ro', ms=8,
            label=f"EER={m['eer']:.4f}")
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Eğrisi')
    ax.legend(); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    ax.plot(m['rec_c'], m['prec_c'], 'g-', lw=2)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Eğrisi  (Best F1={m["best_f1"]:.4f})')
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    ax.hist(m['normal_s'],  bins=40, alpha=0.6, color='steelblue', label='Normal')
    ax.hist(m['anomaly_s'], bins=40, alpha=0.6, color='tomato',    label='Anomali')
    ax.axvline(m['opt_thresh'], color='k', ls='--', lw=1.5,
               label=f"Esik={m['opt_thresh']:.4f}")
    ax.set_xlabel('MSE'); ax.set_ylabel('Frekans')
    ax.set_title('Score Dagilimi'); ax.legend(); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 0])
    cm_arr = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    ax.imshow(cm_arr, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred Normal','Pred Anomali'])
    ax.set_yticklabels(['Gercek Normal','Gercek Anomali'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm_arr[i,j]), ha='center', va='center',
                    fontsize=14, fontweight='bold',
                    color='white' if cm_arr[i,j]>cm_arr.max()/2 else 'black')
    ax.set_title(f'CM  (Esik={m["opt_thresh"]:.4f})')

    ax = fig.add_subplot(gs[1, 1])
    mnames = ['AUC','F1','Precision','Recall','1-EER']
    mvals  = [m['auc'], m['best_f1'], m['prec'], m['rec'], 1-m['eer']]
    bars = ax.bar(mnames, mvals,
                  color=['royalblue','mediumseagreen','darkorange','crimson','purple'],
                  alpha=0.8)
    ax.set_ylim(0, 1.15); ax.set_ylabel('Deger'); ax.set_title('Metrik Ozeti')
    for bar, val in zip(bars, mvals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02, f'{val:.3f}',
                ha='center', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    ax = fig.add_subplot(gs[1, 2])
    ax.axis('off')
    txt  = f"{'='*36}\n{m['tag']} OZET\n{'='*36}\n\n"
    txt += f"AUC        : {m['auc']:.4f}\n"
    txt += f"EER        : {m['eer']:.4f}\n"
    txt += f"Best F1    : {m['best_f1']:.4f}\n"
    txt += f"Precision  : {m['prec']:.4f}\n"
    txt += f"Recall     : {m['rec']:.4f}\n\n"
    txt += f"Normal MSE : {m['normal_s'].mean():.5f}\n"
    txt += f"Anomali MSE: {m['anomaly_s'].mean():.5f}\n"
    txt += f"Ayrim      : {m['sep_ratio']:.2f}x\n\n"
    txt += f"TP:{m['tp']:>4}  FP:{m['fp']:>4}\n"
    txt += f"FN:{m['fn']:>4}  TN:{m['tn']:>4}\n\n"
    txt += f"Normal :{m['n_normal']:>4}  Anomali:{m['n_anomaly']:>4}"
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.9))

    save = LOCAL_OUT / f'eval_{m["tag"].lower()}.png'
    plt.savefig(save, dpi=150, bbox_inches='tight')
    plt.close()
    display(IPImage(filename=str(save)))
    print(f"  Grafik: {save}")

def plot_recall90(results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Recall >= 90% Esigi — Confusion Matrix Karsilastirmasi',
                 fontsize=15, fontweight='bold')
    for ax, m in zip(axes, results):
        if m['rec90_cm'] is None:
            ax.text(0.5,0.5,'Recall 90+\nulasilamadi',
                    ha='center',va='center',transform=ax.transAxes,fontsize=13)
            ax.set_title(m['tag']); continue
        cm_arr = m['rec90_cm']
        ax.imshow(cm_arr, cmap='Oranges')
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(['Pred Normal','Pred Anomali'])
        ax.set_yticklabels(['Gercek Normal','Gercek Anomali'])
        for i in range(2):
            for j in range(2):
                ax.text(j,i,str(cm_arr[i,j]),ha='center',va='center',
                        fontsize=14,fontweight='bold',
                        color='white' if cm_arr[i,j]>cm_arr.max()/2 else 'black')
        tn,fp,fn,tp = cm_arr.ravel()
        rec = tp/(tp+fn+1e-8)
        ax.set_title(f'{m["tag"]}\nEsik={m["rec90_thresh"]:.4f}  Recall={rec:.3f}')
    plt.tight_layout()
    save = LOCAL_OUT / 'recall90_cm.png'
    plt.savefig(save, dpi=150, bbox_inches='tight')
    plt.close()
    display(IPImage(filename=str(save)))
    print(f"  Recall90 CM: {save}")

# ══════════════════════════════════════════════════════════════════════════════
#  ANA DÖNGÜ
# ══════════════════════════════════════════════════════════════════════════════
datasets = [
    ('WINTER', TAR_WINTER, 'test_winter_new_128.npy', 'test_winter_new_labels_128.npy', 'test_winter_new_names_128.npy'),
    ('SUMMER', TAR_SUMMER, 'test_summer_128.npy',     'test_summer_labels_128.npy',     'test_summer_names_128.npy'),
    ('ALL',    TAR_ALL,    'test_all_128.npy',         'test_all_labels_128.npy',         'test_all_names_128.npy'),
]

all_results = []

for tag, tar_path, npy_clips, npy_labels, npy_names in datasets:
    print("\n" + "="*60)
    print(f"TEST: {tag}")
    print("="*60)

    tmp_dir = TMP / tag.lower()
    clips_np, labels_np, names_np = load_from_tar(tar_path, tmp_dir)

    npy_cache_ram[tag] = {
        'clips':  clips_np,
        'labels': labels_np,
        'names':  names_np,
        'fnames': (npy_clips, npy_labels, npy_names)
    }

    print(f"  Inference basliyor...")
    t0 = time.time()
    scores, lbls = run_inference(clips_np, labels_np)
    print(f"  Inference: {time.time()-t0:.0f}sn")

    metrics = compute_metrics(scores, lbls, tag)
    all_results.append(metrics)
    plot_set(metrics)

    shutil.rmtree(str(tmp_dir))
    print(f"  /tmp/{tag.lower()} temizlendi")

# ══════════════════════════════════════════════════════════════════════════════
#  RECALL 90+ KARŞILAŞTIRMA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("RECALL 90+ CM KARSILASTIRMASI")
print("="*60)
plot_recall90(all_results)

# ══════════════════════════════════════════════════════════════════════════════
#  KARŞILAŞTIRMA TABLOSU
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("KARSILASTIRMA TABLOSU")
print("="*60)
print(f"{'Set':<8} {'N':>5} {'A':>5} {'AUC':>7} {'EER':>7} "
      f"{'F1':>7} {'Prec':>7} {'Rec':>7} {'Sep':>6}")
print("-"*65)
for m in all_results:
    print(f"{m['tag']:<8} {m['n_normal']:>5} {m['n_anomaly']:>5} "
          f"{m['auc']:>7.4f} {m['eer']:>7.4f} {m['best_f1']:>7.4f} "
          f"{m['prec']:>7.4f} {m['rec']:>7.4f} {m['sep_ratio']:>6.2f}x")

print(f"\nGrafikler: {LOCAL_OUT}")
print("NPY'ler RAM'de — Drive'a yazmak icin asagidaki blogu calistir.\n")

# ══════════════════════════════════════════════════════════════════════════════
#  ONAY SONRASI DRIVE'A YAZMA
# ══════════════════════════════════════════════════════════════════════════════
def save_to_drive():
    print("Drive'a kaydediliyor...")
    for f in LOCAL_OUT.glob('*.png'):
        dst = ARCHIVE / 'models' / f.name
        shutil.copy(str(f), str(dst))
        print(f"  Grafik: {dst}")
    for tag, data in npy_cache_ram.items():
        fn_clips, fn_labels, fn_names = data['fnames']
        np.save(str(NPY_DIR / fn_clips),  data['clips'])
        np.save(str(NPY_DIR / fn_labels), data['labels'])
        np.save(str(NPY_DIR / fn_names),  data['names'])
        print(f"  {tag} NPY -> {fn_clips}, {fn_labels}, {fn_names}")
    print("Tamamlandi!")

# save_to_drive()   # ← ONAY SONRASI BU SATIRI AÇ VE ÇALIŞTIR

In [ ]:
import shutil
from pathlib import Path

ARCHIVE   = Path('/content/drive/MyDrive/archive')
LOCAL_OUT = Path('/content/eval_results')

# test_results klasörü oluştur
TEST_RESULTS = ARCHIVE / 'test_results'
TEST_RESULTS.mkdir(parents=True, exist_ok=True)

# Grafikleri kaydet
for f in sorted(LOCAL_OUT.glob('*.png')):
    dst = TEST_RESULTS / f.name
    shutil.copy(str(f), str(dst))
    print(f"  {f.name} → {dst}")

print(f"\nTamamlandı! {TEST_RESULTS}")

In [ ]:
"""
crae_error_map_roi_v2.py
=========================
all_results + npy_cache_ram'den direkt al (Drive I/O yok)
WINTER / SUMMER / ALL'dan:
  - 3 en yüksek MSE anomali (net insan)
  - 3 en düşük MSE normal  (net arka plan)
Her set benzersiz klip — ALL'da winter/summer'da seçilenler tekrar alınmaz
Çıktı: /content/eval_results/error_maps/ + Drive'a kopyalanır
"""

import torch, cv2, shutil
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ARCHIVE   = Path('/content/drive/MyDrive/archive')
OUT_DIR   = Path('/content/eval_results/error_maps')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAME_SIZE = 128
FRAME_IDX  = 8   # 16 frame'in ortası
N_SELECT   = 3   # her kategoriden kaç klip

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg['encoder_filters']
        fs = cfg.get('frame_size', 128)
        final_s = fs // 16
        self.spatial = f[-1] * final_s * final_s
        self.final_s = final_s
        self.last_f  = f[-1]
        lstm_h = cfg['lstm_hidden']
        drop   = cfg['dropout']
        self.encoder = nn.Sequential(
            nn.Conv2d(1,    f[0], 3, 2, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0], f[1], 3, 2, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1], f[2], 3, 2, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2], f[3], 3, 2, 1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm   = nn.LSTM(lstm_h, lstm_h, num_layers=cfg['lstm_layers'],
                              batch_first=True,
                              dropout=drop if cfg['lstm_layers'] > 1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3], f[2], 3, 2, 1, 1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2], f[1], 3, 2, 1, 1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1], f[0], 3, 2, 1, 1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0], 1,    3, 2, 1, 1), nn.Sigmoid(),
        )

    def forward(self, clip):
        B, T, C, H, W = clip.shape
        x = self.encoder(clip.view(B*T, C, H, W))
        x = self.proj(x.view(B*T, -1)).view(B, T, -1)
        x, _ = self.lstm(x)
        x = self.unproj(x.reshape(B*T, -1))
        x = x.view(B*T, self.last_f, self.final_s, self.final_s)
        return self.decoder(x).view(B, T, 1, H, W)

checkpoint    = torch.load(str(ARCHIVE / 'models/crae_best.pth'), map_location=device)
model         = CRAE(checkpoint['config']).to(device)
cleaned       = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model_state'].items()}
model.load_state_dict(cleaned)
model.eval()
print(f"Model yüklendi (epoch={checkpoint.get('epoch','?')})")

# ══════════════════════════════════════════════════════════════════════════════
#  KLİP SEÇİMİ — all_results + npy_cache_ram'den (Drive I/O yok)
# ══════════════════════════════════════════════════════════════════════════════
def select_from_results(result, cache, n=3, label=1, highest=True, exclude_names=None):
    """
    label=1 → anomali, label=0 → normal
    highest=True → en yüksek MSE (net anomali)
    highest=False → en düşük MSE  (net normal)
    exclude_names → daha önce seçilen klip isimleri (tekrar alma)
    """
    exclude_names = exclude_names or set()
    scores = result['scores']
    labels = result['labels']
    names  = cache['names']
    clips  = cache['clips']  # float16

    # İlgili label indeksleri
    idx_pool = [i for i in range(len(labels))
                if labels[i] == label and names[i] not in exclude_names]

    # MSE'ye göre sırala
    idx_pool.sort(key=lambda i: scores[i], reverse=highest)

    selected = []
    for i in idx_pool[:n]:
        selected.append({
            'name':  names[i],
            'score': scores[i],
            'clip':  clips[i].astype(np.float32),  # (16,128,128)
            'label': 'anomaly' if label == 1 else 'normal',
        })
    return selected

# WINTER
w_anomaly = select_from_results(all_results[0], npy_cache_ram['WINTER'],
                                 n=N_SELECT, label=1, highest=True)
w_normal  = select_from_results(all_results[0], npy_cache_ram['WINTER'],
                                 n=N_SELECT, label=0, highest=False)

# SUMMER
s_anomaly = select_from_results(all_results[1], npy_cache_ram['SUMMER'],
                                 n=N_SELECT, label=1, highest=True)
s_normal  = select_from_results(all_results[1], npy_cache_ram['SUMMER'],
                                 n=N_SELECT, label=0, highest=False)

# ALL — daha önce seçilenler hariç
used_names = {c['name'] for c in w_anomaly + w_normal + s_anomaly + s_normal}
a_anomaly  = select_from_results(all_results[2], npy_cache_ram['ALL'],
                                  n=N_SELECT, label=1, highest=True,
                                  exclude_names=used_names)
used_names.update(c['name'] for c in a_anomaly)
a_normal   = select_from_results(all_results[2], npy_cache_ram['ALL'],
                                  n=N_SELECT, label=0, highest=False,
                                  exclude_names=used_names)

print("\nSeçilen klipler:")
for tag, group in [('WINTER anomali', w_anomaly), ('WINTER normal', w_normal),
                   ('SUMMER anomali', s_anomaly), ('SUMMER normal', s_normal),
                   ('ALL anomali',    a_anomaly), ('ALL normal',    a_normal)]:
    print(f"  {tag}:")
    for c in group:
        print(f"    {c['name']}  MSE={c['score']:.5f}")

# ══════════════════════════════════════════════════════════════════════════════
#  INFERENCE + ERROR MAP
# ══════════════════════════════════════════════════════════════════════════════
def get_error_map(clip_np, frame_idx=FRAME_IDX):
    t = torch.tensor(clip_np).unsqueeze(0).unsqueeze(2).to(device)  # (1,16,1,H,W)
    with torch.no_grad():
        out = model(t)
    orig  = t[0, frame_idx, 0].cpu().numpy()
    recon = out[0, frame_idx, 0].cpu().numpy()
    err   = np.abs(orig - recon)
    # Tüm frame MSE
    frame_mses = [torch.mean((t[0,i]-out[0,i])**2).item() for i in range(16)]
    return orig, recon, err, frame_mses

def find_roi(err_map, pct=95):
    tv = np.percentile(err_map, pct)
    binary = (err_map >= tv).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [cv2.boundingRect(c) for c in cnts if cv2.contourArea(c) > 20]
    return boxes, tv

# ══════════════════════════════════════════════════════════════════════════════
#  GÖRSELLEŞTİRME
# ══════════════════════════════════════════════════════════════════════════════
def plot_clip(clip_info, season, idx):
    name  = clip_info['name']
    score = clip_info['score']
    label = clip_info['label']
    clip  = clip_info['clip']

    orig, recon, err, frame_mses = get_error_map(clip)
    boxes, thresh_val = find_roi(err)

    color = 'red' if label == 'anomaly' else 'limegreen'
    lbl_tr = 'ANOMALİ' if label == 'anomaly' else 'NORMAL'

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        f'{season.upper()} — {lbl_tr} | {name}\nClip MSE: {score:.5f}',
        fontsize=13, fontweight='bold', color=color
    )

    # 1. Orijinal
    axes[0,0].imshow(orig, cmap='gray', vmin=0, vmax=1)
    axes[0,0].set_title('Orijinal Frame', fontsize=11); axes[0,0].axis('off')

    # 2. Reconstructed
    axes[0,1].imshow(recon, cmap='gray', vmin=0, vmax=1)
    axes[0,1].set_title('Reconstructed Frame', fontsize=11); axes[0,1].axis('off')

    # 3. Error map (hot)
    im = axes[0,2].imshow(err, cmap='hot', vmin=0, vmax=0.15)
    axes[0,2].set_title(f'Error Map | Threshold={thresh_val:.4f}', fontsize=11)
    axes[0,2].axis('off')
    plt.colorbar(im, ax=axes[0,2], fraction=0.046)

    # 4. ROI overlay
    axes[1,0].imshow(orig, cmap='gray', vmin=0, vmax=1)
    for (x, y, w, h) in boxes:
        axes[1,0].add_patch(patches.Rectangle(
            (x,y), w, h, linewidth=2.5, edgecolor=color, facecolor='none'))
    axes[1,0].set_title(f'ROI ({len(boxes)} bölge) — {lbl_tr}',
                         fontsize=11, color=color)
    axes[1,0].axis('off')

    # 5. Jet heatmap
    jet = cv2.applyColorMap(
        (err * 255 / 0.15).clip(0,255).astype(np.uint8), cv2.COLORMAP_JET)
    axes[1,1].imshow(cv2.cvtColor(jet, cv2.COLOR_BGR2RGB))
    axes[1,1].set_title('Error Heatmap (Jet)', fontsize=11)
    axes[1,1].axis('off')

    # 6. Frame MSE eğrisi
    axes[1,2].plot(range(16), frame_mses, 'b-o', markersize=4)
    axes[1,2].axvline(FRAME_IDX, color='red', ls='--', lw=1.5,
                       label=f'Frame {FRAME_IDX}')
    axes[1,2].set_xlabel('Frame'); axes[1,2].set_ylabel('MSE')
    axes[1,2].set_title('Frame Bazlı MSE'); axes[1,2].legend(fontsize=9)
    axes[1,2].grid(alpha=0.3)

    plt.tight_layout()
    fname = f'{season}_{label}_{idx+1:02d}_{name}.png'
    save  = OUT_DIR / fname
    plt.savefig(str(save), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Kaydedildi: {fname}")
    return save

# ══════════════════════════════════════════════════════════════════════════════
#  TÜMÜNÜ ÇALIŞTIR
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("GÖRSELLEŞTİRME BAŞLIYOR")
print("="*60)

saved = []
for season, groups in [
    ('winter', [w_anomaly, w_normal]),
    ('summer', [s_anomaly, s_normal]),
    ('all',    [a_anomaly, a_normal]),
]:
    print(f"\n  {season.upper()}")
    for group in groups:
        for idx, clip_info in enumerate(group):
            f = plot_clip(clip_info, season, idx)
            saved.append(f)

# ── Karşılaştırma grid (6×3: tüm setler, her satır bir set+label) ─────────
print("\nKarşılaştırma grafiği oluşturuluyor...")

groups_ordered = [
    ('Winter Anomali', w_anomaly, 'red'),
    ('Winter Normal',  w_normal,  'limegreen'),
    ('Summer Anomali', s_anomaly, 'red'),
    ('Summer Normal',  s_normal,  'limegreen'),
    ('ALL Anomali',    a_anomaly, 'red'),
    ('ALL Normal',     a_normal,  'limegreen'),
]

fig, axes = plt.subplots(6, 3, figsize=(18, 30))
fig.suptitle('CR-AE Error Map + ROI — Winter / Summer / ALL Karşılaştırması',
             fontsize=14, fontweight='bold')

for row, (rlabel, group, rcolor) in enumerate(groups_ordered):
    for col, clip_info in enumerate(group):
        orig, recon, err, _ = get_error_map(clip_info['clip'])
        boxes, _ = find_roi(err)
        ax = axes[row, col]
        ax.imshow(orig, cmap='gray', vmin=0, vmax=1)
        for (x, y, w, h) in boxes:
            ax.add_patch(patches.Rectangle(
                (x,y), w, h, linewidth=2, edgecolor=rcolor, facecolor='none'))
        ax.set_title(f"{clip_info['name']}\nMSE={clip_info['score']:.5f}",
                      fontsize=7)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(rlabel, fontsize=9, color=rcolor, fontweight='bold')

plt.tight_layout()
comp = OUT_DIR / 'comparison_all_sets.png'
plt.savefig(str(comp), dpi=150, bbox_inches='tight')
plt.close()
print(f"  Karşılaştırma: {comp.name}")

# ── Drive'a kopyala ───────────────────────────────────────────────────────
print("\nDrive'a kopyalanıyor...")
DRIVE_OUT = ARCHIVE / 'test_results_CRAE/error_maps'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for f in list(OUT_DIR.glob('*.png')):
    shutil.copy(str(f), str(DRIVE_OUT / f.name))
    print(f"  {f.name}")

print(f"\n{'='*60}")
print(f"TAMAMLANDI — {len(saved)+1} görsel")
print(f"  Locale : {OUT_DIR}")
print(f"  Drive  : {DRIVE_OUT}")
print(f"{'='*60}")

## ***ÖNCESİNDE KISLA EĞİTİM YAPILDI, KIS TESTİ:0.85+ YAZ TESTİ:0.90+ AUC ELDE EDİLDİ***

In [ ]:
"""
build_all_tars.py
==================
4 TAR olusturur:
1. pure_bg_winter.tar   — training/pure_background/YYYYMMDD/clip_xxx/
2. pure_bg_summer.tar   — training/pure_background_summer/YYYYMMDD/clip_xxx/
3. test_winter.tar      — testing/testing_final_winter/YYYYMMDD/normal|anomaly/clip_xxx/
4. test_summer.tar      — testing/testing_final_summer/YYYYMMDD/normal|anomaly/clip_xxx/

Cikti: archive/all_final_tars_crae/
Yontem: Drive -> /tmp paralel kopyala -> tar -> Drive
"""

import shutil, time, subprocess, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

ARCHIVE  = Path('/content/drive/MyDrive/archive')
BASE_AE  = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors'
OUT_DIR  = ARCHIVE / 'all_final_tars_crae'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TMP      = Path('/tmp/tar_build')
TMP.mkdir(parents=True, exist_ok=True)

WORKERS  = 128

DATASETS = [
    {
        'name'   : 'pure_bg_winter',
        'src'    : BASE_AE / 'training/pure_background',
        'tar'    : OUT_DIR / 'pure_bg_winter.tar',
        'has_labels': False,   # YYYYMMDD/clip_xxx/
    },
    {
        'name'   : 'pure_bg_summer',
        'src'    : BASE_AE / 'training/pure_background_summer',
        'tar'    : OUT_DIR / 'pure_bg_summer.tar',
        'has_labels': False,
    },
    {
        'name'   : 'test_winter',
        'src'    : BASE_AE / 'testing/testing_final_winter',
        'tar'    : OUT_DIR / 'test_winter.tar',
        'has_labels': True,    # YYYYMMDD/normal|anomaly/clip_xxx/
    },
    {
        'name'   : 'test_summer',
        'src'    : BASE_AE / 'testing/testing_final_summer',
        'tar'    : OUT_DIR / 'test_summer.tar',
        'has_labels': True,
    },
]

# ══════════════════════════════════════════════════════════════════════════════
#  YARDIMCI
# ══════════════════════════════════════════════════════════════════════════════
def copy_clip(args):
    src_clip, dst_clip = args
    dst_clip.mkdir(parents=True, exist_ok=True)
    for f in sorted(src_clip.glob("frame*.jpg")):
        dst = dst_clip / f.name
        if not dst.exists():
            shutil.copy2(str(f), str(dst))

def collect_copy_jobs(src_root, tmp_root, has_labels):
    """
    has_labels=False: src/YYYYMMDD/clip_xxx  -> tmp/YYYYMMDD/clip_xxx
    has_labels=True : src/YYYYMMDD/label/clip -> tmp/YYYYMMDD/label/clip
    """
    jobs = []
    for day_dir in sorted(src_root.iterdir()):
        if not day_dir.is_dir(): continue
        if not day_dir.name.isdigit(): continue
        if has_labels:
            for lbl_dir in sorted(day_dir.iterdir()):
                if not lbl_dir.is_dir(): continue
                for clip_dir in sorted(lbl_dir.iterdir()):
                    if not clip_dir.is_dir(): continue
                    dst = tmp_root / day_dir.name / lbl_dir.name / clip_dir.name
                    jobs.append((clip_dir, dst))
        else:
            for clip_dir in sorted(day_dir.iterdir()):
                if not clip_dir.is_dir(): continue
                dst = tmp_root / day_dir.name / clip_dir.name
                jobs.append((clip_dir, dst))
    return jobs

# ══════════════════════════════════════════════════════════════════════════════
#  HER DATASET
# ══════════════════════════════════════════════════════════════════════════════
total_start = time.time()

for ds in DATASETS:
    name       = ds['name']
    src        = ds['src']
    tar_path   = ds['tar']
    has_labels = ds['has_labels']
    tmp_src    = TMP / name

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    # ── ADIM 1: Drive → /tmp paralel kopyala ──────────────────────────────────
    jobs = collect_copy_jobs(src, tmp_src, has_labels)
    existing_frames = len(list(tmp_src.rglob('*.jpg'))) if tmp_src.exists() else 0
    total_frames    = sum(len(list(j[0].glob('frame*.jpg'))) for j in jobs)

    if existing_frames >= total_frames * 0.95:
        print(f"  Zaten locale: {existing_frames} frame")
    else:
        print(f"  Kopyalanıyor: {len(jobs)} klip, ~{total_frames} frame...")
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(tqdm(ex.map(copy_clip, jobs),
                      total=len(jobs), desc=f"  {name}"))
        print(f"  Kopyalama: {time.time()-t0:.0f}sn")

    # ── ADIM 2: /tmp → TAR ────────────────────────────────────────────────────
    tmp_tar = TMP / f"{name}.tar"
    print(f"  TAR olusturuluyor...")
    t0 = time.time()
    subprocess.run(
        f"tar -cf '{tmp_tar}' -C '{tmp_src.parent}' '{tmp_src.name}'",
        shell=True, check=True
    )
    size_mb = tmp_tar.stat().st_size / 1e6
    print(f"  TAR: {time.time()-t0:.0f}sn  ({size_mb:.0f}MB)")

    # ── ADIM 3: TAR → Drive ───────────────────────────────────────────────────
    print(f"  Drive'a yaziliyor...")
    t0 = time.time()
    if tar_path.exists():
        tar_path.unlink()
    subprocess.run(f"cp '{tmp_tar}' '{tar_path}'", shell=True, check=True)
    print(f"  Yazildi: {time.time()-t0:.0f}sn  -> {tar_path.name}")

    # ── /tmp temizle ──────────────────────────────────────────────────────────
    shutil.rmtree(str(tmp_src), ignore_errors=True)
    tmp_tar.unlink(missing_ok=True)
    gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  OZET
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"TAMAMLANDI  ({(time.time()-total_start)/60:.1f}dk)")
print(f"{'='*60}")
for ds in DATASETS:
    p = ds['tar']
    if p.exists():
        print(f"  {p.name:<30} {p.stat().st_size/1e6:>8.0f} MB")

shutil.rmtree(str(TMP), ignore_errors=True)
gc.collect()

In [ ]:
"""
build_npy_cache.py
===================
4 TAR'dan 128x128 NPY cache olusturur:
1. pure_bg_winter_128.npy
2. pure_bg_summer_128.npy
3. test_winter_128.npy  + labels + names
4. test_summer_128.npy  + labels + names

Cikti: archive/final_npycache/
GPU RAM cache: gpu_cache dict
"""

import subprocess, shutil, time, gc, cv2
import numpy as np
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

ARCHIVE    = Path('/content/drive/MyDrive/archive')
TAR_DIR    = ARCHIVE / 'all_final_tars_crae'
OUT_DIR    = ARCHIVE / 'final_npycache'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TMP        = Path('/tmp/npy_build')
TMP.mkdir(parents=True, exist_ok=True)

WORKERS    = 128
BATCH      = 64
FRAME_SIZE = 128
CLIP_LEN   = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

gpu_cache = {}

# ══════════════════════════════════════════════════════════════════════════════
#  YARDIMCI
# ══════════════════════════════════════════════════════════════════════════════
def find_actual_root(tmp_dst):
    """TAR icinde ekstra klasor olabilir, YYYYMMDD icerenin parent'ini bul."""
    for p in sorted(tmp_dst.rglob('*')):
        if p.is_dir() and p.name.isdigit() and len(p.name) == 8:
            return p.parent
    return tmp_dst

def read_clip(clip_dir):
    frames = sorted(clip_dir.glob("frame*.jpg"),
                    key=lambda p: int(''.join(filter(str.isdigit, p.stem))))
    if len(frames) < CLIP_LEN: return None
    imgs = []
    for f in frames[:CLIP_LEN]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))
        imgs.append(img.astype(np.float32) / 255.0)
    return np.stack(imgs)

def read_batch(clip_dirs):
    results = [None] * len(clip_dirs)
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(read_clip, c): i for i, c in enumerate(clip_dirs)}
        for fut in as_completed(futs):
            results[futs[fut]] = fut.result()
    return results

def load_all_clips(root, has_labels):
    clip_dirs = []
    labels    = []
    names     = []
    day_names = []

    for day_dir in sorted(root.iterdir()):
        if not day_dir.is_dir() or not day_dir.name.isdigit(): continue
        if has_labels:
            for lbl_dir in sorted(day_dir.iterdir()):
                if not lbl_dir.is_dir(): continue
                lbl_val = 0 if lbl_dir.name == 'normal' else 1
                for clip_dir in sorted(lbl_dir.iterdir()):
                    if not clip_dir.is_dir(): continue
                    clip_dirs.append(clip_dir)
                    labels.append(lbl_val)
                    names.append(clip_dir.name)
                    day_names.append(day_dir.name)
        else:
            for clip_dir in sorted(day_dir.iterdir()):
                if not clip_dir.is_dir(): continue
                clip_dirs.append(clip_dir)
                labels.append(0)
                names.append(clip_dir.name)
                day_names.append(day_dir.name)

    print(f"  Toplam klip: {len(clip_dirs)}")
    if has_labels:
        n = sum(1 for l in labels if l == 0)
        a = sum(1 for l in labels if l == 1)
        print(f"  Normal: {n}  Anomali: {a}")

    all_clips = []
    valid_idx = []
    for i in tqdm(range(0, len(clip_dirs), BATCH), desc="  Batch"):
        batch   = clip_dirs[i:i+BATCH]
        results = read_batch(batch)
        for j, arr in enumerate(results):
            if arr is not None:
                all_clips.append(arr)
                valid_idx.append(i+j)

    if not all_clips:
        raise ValueError("Hic klip okunamadi! Klasor yapisi kontrol edilmeli.")

    clips_np   = np.stack(all_clips).astype(np.float16)
    labels_np  = np.array([labels[i]    for i in valid_idx], dtype=np.int8)
    names_np   = np.array([names[i]     for i in valid_idx], dtype=object)
    days_np    = np.array([day_names[i] for i in valid_idx], dtype=object)

    return clips_np, labels_np, names_np, days_np

# ══════════════════════════════════════════════════════════════════════════════
#  DATASET TANIMLARI
# ══════════════════════════════════════════════════════════════════════════════
DATASETS = [
    {
        'name'       : 'pure_bg_winter',
        'tar'        : TAR_DIR / 'pure_bg_winter.tar',
        'has_labels' : False,
        'npy_clips'  : OUT_DIR / 'pure_bg_winter_128.npy',
        'npy_labels' : None,
        'npy_names'  : OUT_DIR / 'pure_bg_winter_names.npy',
    },
    {
        'name'       : 'pure_bg_summer',
        'tar'        : TAR_DIR / 'pure_bg_summer.tar',
        'has_labels' : False,
        'npy_clips'  : OUT_DIR / 'pure_bg_summer_128.npy',
        'npy_labels' : None,
        'npy_names'  : OUT_DIR / 'pure_bg_summer_names.npy',
    },
    {
        'name'       : 'test_winter',
        'tar'        : TAR_DIR / 'test_winter.tar',
        'has_labels' : True,
        'npy_clips'  : OUT_DIR / 'test_winter_128.npy',
        'npy_labels' : OUT_DIR / 'test_winter_labels.npy',
        'npy_names'  : OUT_DIR / 'test_winter_names.npy',
    },
    {
        'name'       : 'test_summer',
        'tar'        : TAR_DIR / 'test_summer.tar',
        'has_labels' : True,
        'npy_clips'  : OUT_DIR / 'test_summer_128.npy',
        'npy_labels' : OUT_DIR / 'test_summer_labels.npy',
        'npy_names'  : OUT_DIR / 'test_summer_names.npy',
    },
]

# ══════════════════════════════════════════════════════════════════════════════
#  ANA DONGU
# ══════════════════════════════════════════════════════════════════════════════
total_start = time.time()

for ds in DATASETS:
    name       = ds['name']
    tar_path   = ds['tar']
    has_labels = ds['has_labels']
    tmp_dst    = TMP / name

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    # NPY zaten varsa yukle
    if ds['npy_clips'].exists():
        print(f"  NPY zaten var, yukleniyor...")
        t0     = time.time()
        clips  = np.load(str(ds['npy_clips']))
        labels = np.load(str(ds['npy_labels'])) if ds['npy_labels'] and ds['npy_labels'].exists() else None
        names  = np.load(str(ds['npy_names']), allow_pickle=True)
        print(f"  {clips.shape}  dtype={clips.dtype}  ({time.time()-t0:.0f}sn)")
    else:
        # ── TAR → /tmp ────────────────────────────────────────────────────────
        if tmp_dst.exists() and len(list(tmp_dst.rglob('*.jpg'))) > 100:
            print(f"  /tmp hazir")
        else:
            tmp_dst.mkdir(parents=True, exist_ok=True)
            print(f"  TAR aciliyor: {tar_path.name}...")
            t0 = time.time()
            subprocess.run(f"tar -xf '{tar_path}' -C '{tmp_dst}'",
                           shell=True, check=True)
            print(f"  TAR: {time.time()-t0:.0f}sn")

        # Gercek root'u bul
        actual_root = find_actual_root(tmp_dst)
        print(f"  Root: {actual_root.relative_to(TMP)}")

        # ── Frame'leri oku ────────────────────────────────────────────────────
        t0 = time.time()
        clips, labels, names, day_names = load_all_clips(actual_root, has_labels)
        print(f"  Okuma: {time.time()-t0:.0f}sn")
        print(f"  Shape: {clips.shape}  dtype={clips.dtype}")

        # ── NPY kaydet ────────────────────────────────────────────────────────
        print(f"  NPY kaydediliyor...")
        t0 = time.time()
        np.save(str(ds['npy_clips']), clips)
        np.save(str(ds['npy_names']), names)
        if has_labels and ds['npy_labels']:
            np.save(str(ds['npy_labels']), labels)
        print(f"  Kaydedildi: {time.time()-t0:.0f}sn")

        shutil.rmtree(str(tmp_dst), ignore_errors=True)
        gc.collect()

    # ── GPU RAM cache ─────────────────────────────────────────────────────────
    print(f"  GPU RAM'e yukleniyor...")
    t0 = time.time()
    gpu_tensor = torch.tensor(
        clips.astype(np.float32), dtype=torch.float16).to(device)
    gpu_cache[name] = {
        'clips' : gpu_tensor,
        'labels': torch.tensor(labels.astype(np.int8)).to(device) if labels is not None else None,
        'names' : names,
    }
    mb = gpu_tensor.element_size() * gpu_tensor.nelement() / 1e6
    print(f"  GPU: {mb:.0f}MB  ({time.time()-t0:.0f}sn)")
    del clips
    gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  OZET
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"TAMAMLANDI  ({(time.time()-total_start)/60:.1f}dk)")
print(f"{'='*60}")

print(f"\nNPY dosyalari:")
for f in sorted(OUT_DIR.glob('*.npy')):
    print(f"  {f.name:<45} {f.stat().st_size/1e6:>8.0f} MB")

print(f"\nGPU RAM cache:")
for k, v in gpu_cache.items():
    shape = tuple(v['clips'].shape)
    mb    = v['clips'].element_size()*v['clips'].nelement()/1e6
    print(f"  {k:<28} {str(shape):<28} {mb:>8.0f} MB")

if device.type == 'cuda':
    alloc = torch.cuda.memory_allocated()/1e6
    total = torch.cuda.get_device_properties(0).total_memory/1e6
    print(f"\nGPU RAM: {alloc:.0f}MB / {total:.0f}MB")

shutil.rmtree(str(TMP), ignore_errors=True)
gc.collect()
print("\nHazir! gpu_cache kullanima hazir.")

In [ ]:
"""
crae_model1_winter_only.py
===========================
Model 1: Sadece Pure BG Winter ile egitim
Test: Winter / Summer / All
Cikti: models/crae_winter_only.pth
       diff_technic_results/model1_winter_only_results.json
"""

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, random, gc, json
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Image as IPImage
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              f1_score, confusion_matrix, classification_report)

# ══════════════════════════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

MODEL_NAME = 'winter_only'
NPY_DIR    = Path('/content/drive/MyDrive/archive/final_npycache')
SAVE_DIR   = Path('/content/drive/MyDrive/archive/models')
RES_DIR    = Path('/content/drive/MyDrive/archive/diff_technic_results')
SAVE_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

CONFIG = {
    'frame_size'     : 128,
    'frames_per_clip': 16,
    'batch_size'     : 32,
    'val_split'      : 0.15,
    'epochs'         : 100,
    'lr'             : 1e-3,
    'min_lr'         : 1e-6,
    'lstm_hidden'    : 256,
    'lstm_layers'    : 2,
    'dropout'        : 0.3,
    'patience'       : 15,
    'weight_decay'   : 1e-5,
    'encoder_filters': [32, 64, 128, 256],
}

# ══════════════════════════════════════════════════════════════════════════════
#  VERİ YUKLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("VERI YUKLEME")
print("="*60)
t0 = time.time()

# Egitim
bg_clips = np.load(str(NPY_DIR/'pure_bg_winter_128.npy')).astype(np.float32)
bg_names = np.load(str(NPY_DIR/'pure_bg_winter_names.npy'), allow_pickle=True)

# Test setleri
tw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)
tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))
tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)

ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)
ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))
ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)

# All = winter + summer
all_clips  = np.concatenate([tw_clips,  ts_clips])
all_labels = np.concatenate([tw_labels, ts_labels])
all_names  = np.concatenate([tw_names,  ts_names])

print(f"Pure BG Winter : {bg_clips.shape}")
print(f"Test Winter    : {tw_clips.shape} (N:{(tw_labels==0).sum()} A:{(tw_labels==1).sum()})")
print(f"Test Summer    : {ts_clips.shape} (N:{(ts_labels==0).sum()} A:{(ts_labels==1).sum()})")
print(f"Test All       : {all_clips.shape} (N:{(all_labels==0).sum()} A:{(all_labels==1).sum()})")
print(f"Yukleme: {time.time()-t0:.1f}sn")

# ══════════════════════════════════════════════════════════════════════════════
#  GUN BAZLI SPLIT
# ══════════════════════════════════════════════════════════════════════════════
def day_stratified_split(data, names, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    day_to_idx  = defaultdict(list)
    month_to_days = defaultdict(set)
    for i, name in enumerate(names):
        d = str(name)[:8]; m = d[:6]
        day_to_idx[d].append(i)
        month_to_days[m].add(d)
    val_idx = []; tr_idx = []
    for month in sorted(month_to_days):
        days = sorted(month_to_days[month])
        n_val = max(1, round(len(days)*val_ratio))
        shuf  = days[:]; rng.shuffle(shuf)
        vd    = set(shuf[:n_val])
        for d in days:
            (val_idx if d in vd else tr_idx).extend(day_to_idx[d])
    return np.array(tr_idx), np.array(val_idx)

tr_idx, val_idx = day_stratified_split(bg_clips, bg_names, CONFIG['val_split'], SEED)
train_data = bg_clips[tr_idx]
val_data   = bg_clips[val_idx]
print(f"\nTrain: {len(train_data)}  Val: {len(val_data)}")
del bg_clips; gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════════════════════
class AugDataset(Dataset):
    def __init__(self, data, aug=True):
        self.data = data; self.aug = aug
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        clip = self.data[i].copy()
        if self.aug:
            if random.random() > 0.5: clip = clip[:,:,::-1].copy()
            clip = np.clip(clip + random.uniform(-0.1,0.1), 0, 1)
            clip = np.clip(clip + np.random.normal(0,0.02,clip.shape).astype(np.float32), 0, 1)
        return torch.tensor(clip).unsqueeze(1)

class TestDS(Dataset):
    def __init__(self, d, l): self.d=d; self.l=l
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

train_dl = DataLoader(AugDataset(train_data, True),  CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(AugDataset(val_data,   False), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
tw_dl    = DataLoader(TestDS(tw_clips,  tw_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
ts_dl    = DataLoader(TestDS(ts_clips,  ts_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
all_dl   = DataLoader(TestDS(all_clips, all_labels), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg['encoder_filters']
        self.spatial = f[-1]*8*8
        lstm_h = cfg['lstm_hidden']
        drop   = cfg['dropout']
        self.encoder = nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm   = nn.LSTM(lstm_h, lstm_h, cfg['lstm_layers'], batch_first=True,
                              dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W = x.shape
        e = self.encoder(x.view(B*T,C,H,W))
        e = self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_ = self.lstm(e)
        d = self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)
        return self.decoder(d).view(B,T,1,H,W)

model = CRAE(CONFIG).to(device)
if hasattr(torch,'compile'):
    try: model = torch.compile(model); print("torch.compile aktif")
    except: pass
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,}")

# ══════════════════════════════════════════════════════════════════════════════
#  EGITIM
# ══════════════════════════════════════════════════════════════════════════════
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CONFIG['min_lr'])

best_val = float('inf'); pat = 0
tl_h=[]; vl_h=[]; lr_h=[]
t_start = time.time()
save_path = SAVE_DIR / f'crae_{MODEL_NAME}.pth'

print(f"\n{'='*60}\nEGITIM — {MODEL_NAME}\n{'='*60}")
for ep in range(CONFIG['epochs']):
    model.train()
    el=nb=0
    for batch in train_dl:
        clips=batch.to(device); optimizer.zero_grad()
        loss=criterion(model(clips),clips); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); el+=loss.item(); nb+=1
    tl=el/max(nb,1); tl_h.append(tl)

    model.eval()
    vl=nv=0
    with torch.no_grad():
        for batch in val_dl:
            clips=batch.to(device); vl+=criterion(model(clips),clips).item(); nv+=1
    vl/=max(nv,1); vl_h.append(vl)
    lr_now=optimizer.param_groups[0]['lr']; lr_h.append(lr_now)
    scheduler.step()

    if vl < best_val:
        best_val=vl; pat=0
        torch.save({'epoch':ep,'model_state':model.state_dict(),
                    'val_loss':vl,'config':CONFIG,'model_name':MODEL_NAME}, str(save_path))
    else: pat+=1

    elapsed=time.time()-t_start
    eta=elapsed/(ep+1)*(CONFIG['epochs']-ep-1)
    print(f"  Ep{ep+1:>3} T:{tl:.6f} V:{vl:.6f} Best:{best_val:.6f} "
          f"Pat:{pat}/{CONFIG['patience']} {elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)
    if pat>=CONFIG['patience']:
        print(f"  Early stopping ep{ep+1}"); break

total_time = time.time()-t_start
print(f"\nEgitim: {total_time/60:.1f}dk")

# Egitim grafigi
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(tl_h,'b-',label='Train',alpha=0.7); axes[0].plot(vl_h,'r-',label='Val',alpha=0.7)
axes[0].set_title(f'Egitim Egrisi — {MODEL_NAME}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(lr_h,'g-'); axes[1].set_title('LR Schedule'); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/training_{MODEL_NAME}.png',dpi=150)
plt.savefig(str(RES_DIR/f'training_{MODEL_NAME}.png'),dpi=150); plt.close()
display(IPImage(f'/content/training_{MODEL_NAME}.png'))

# ══════════════════════════════════════════════════════════════════════════════
#  TEST FONKSİYONU
# ══════════════════════════════════════════════════════════════════════════════
ck = torch.load(str(save_path)); model.load_state_dict(ck['model_state']); model.eval()

def run_test(dl, tag):
    scores=[]; labs=[]
    with torch.no_grad():
        for clips,lbls in dl:
            clips=clips.to(device); out=model(clips)
            for i in range(clips.shape[0]):
                scores.append(torch.mean((clips[i]-out[i])**2).item())
                labs.append(lbls[i].item())
    scores=np.array(scores); labs=np.array(labs)
    ns=scores[labs==0]; als=scores[labs==1]
    auc=roc_auc_score(labs,scores)
    prec,rec,thr=precision_recall_curve(labs,scores)
    f1s=2*prec*rec/(prec+rec+1e-8)
    opt_t=thr[np.argmax(f1s)]
    preds=(scores>=opt_t).astype(int)
    cm=confusion_matrix(labs,preds); tn,fp,fn,tp=cm.ravel()
    fpr_r,tpr_r,thr_r=roc_curve(labs,scores)
    fnr_r=1-tpr_r
    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))
    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)

    print(f"\n  {'='*50}")
    print(f"  {tag}")
    print(f"  {'='*50}")
    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")
    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")
    print(f"  AUC        : {auc:.4f}")
    print(f"  EER        : {eer:.4f} ({eer*100:.2f}%)")
    print(f"  F1         : {f1s.max():.4f}")
    print(f"  Precision  : {tp/(tp+fp+1e-8):.4f}")
    print(f"  Recall     : {tp/(tp+fn+1e-8):.4f}")
    print(f"  Threshold  : {opt_t:.6f}")
    print(f"  TP:{tp} FP:{fp} FN:{fn} TN:{tn}")

    # ROC grafigi
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
    axes[0].set_title(f'ROC — {tag}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')
    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')
    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')
    axes[1].set_title(f'Score Dagilimi — {tag}'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
    plt.tight_layout()
    fname=f'{MODEL_NAME}_{tag.lower().replace(" ","_")}'
    plt.savefig(f'/content/{fname}.png',dpi=150)
    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=150); plt.close()

    return {
        'tag':tag,'auc':float(auc),'eer':float(eer),
        'f1':float(f1s.max()),'threshold':float(opt_t),
        'precision':float(tp/(tp+fp+1e-8)),'recall':float(tp/(tp+fn+1e-8)),
        'specificity':float(tn/(tn+fp+1e-8)),
        'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),
        'n_normal':int((labs==0).sum()),'n_anomaly':int((labs==1).sum()),
        'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),
    }

print(f"\n{'='*60}\nTEST\n{'='*60}")
r_winter = run_test(tw_dl,  'WINTER')
r_summer = run_test(ts_dl,  'SUMMER')
r_all    = run_test(all_dl, 'ALL')

# ══════════════════════════════════════════════════════════════════════════════
#  KAYDET
# ══════════════════════════════════════════════════════════════════════════════
results = {
    'model_name'   : MODEL_NAME,
    'training_data': 'pure_bg_winter',
    'config'       : CONFIG,
    'n_params'     : n_params,
    'train_clips'  : len(train_data),
    'val_clips'    : len(val_data),
    'best_val_loss': float(best_val),
    'best_epoch'   : int(ck['epoch'])+1,
    'total_epochs' : len(tl_h),
    'training_time_min': round(total_time/60,1),
    'test_winter'  : r_winter,
    'test_summer'  : r_summer,
    'test_all'     : r_all,
    'train_losses' : [float(x) for x in tl_h],
    'val_losses'   : [float(x) for x in vl_h],
}
out_json = RES_DIR/f'model1_{MODEL_NAME}_results.json'
with open(str(out_json),'w') as f: json.dump(results,f,indent=2)

print(f"\n{'='*60}")
print(f"OZET — Model 1: {MODEL_NAME}")
print(f"{'='*60}")
print(f"  {'Test Seti':<12} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print(f"  {'-'*55}")
for r in [r_winter, r_summer, r_all]:
    print(f"  {r['tag']:<12} {r['auc']:>8.4f} {r['eer']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")
print(f"\n  Model : {save_path}")
print(f"  JSON  : {out_json}")
print("="*60)

gc.collect()
print("\nTamamlandi!")

In [ ]:
"""
crae_model2_summer_only.py
===========================
Model 2: Sadece Pure BG Summer ile egitim
Test: Winter / Summer / All
Cikti: models/crae_summer_only.pth
       diff_technic_results/model2_summer_only_results.json
"""

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, random, gc, json
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Image as IPImage
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              confusion_matrix)

# ══════════════════════════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

MODEL_NAME = 'summer_only'
NPY_DIR    = Path('/content/drive/MyDrive/archive/final_npycache')
SAVE_DIR   = Path('/content/drive/MyDrive/archive/models')
RES_DIR    = Path('/content/drive/MyDrive/archive/diff_technic_results')
SAVE_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

CONFIG = {
    'frame_size'     : 128,
    'frames_per_clip': 16,
    'batch_size'     : 32,
    'val_split'      : 0.15,
    'epochs'         : 100,
    'lr'             : 1e-3,
    'min_lr'         : 1e-6,
    'lstm_hidden'    : 256,
    'lstm_layers'    : 2,
    'dropout'        : 0.3,
    'patience'       : 15,
    'weight_decay'   : 1e-5,
    'encoder_filters': [32, 64, 128, 256],
}

# ══════════════════════════════════════════════════════════════════════════════
#  VERI YUKLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("VERI YUKLEME")
print("="*60)
t0 = time.time()

bg_clips = np.load(str(NPY_DIR/'pure_bg_summer_128.npy')).astype(np.float32)
bg_names = np.load(str(NPY_DIR/'pure_bg_summer_names.npy'), allow_pickle=True)

tw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)
tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))
tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)

ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)
ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))
ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)

all_clips  = np.concatenate([tw_clips,  ts_clips])
all_labels = np.concatenate([tw_labels, ts_labels])
all_names  = np.concatenate([tw_names,  ts_names])

print(f"Pure BG Summer : {bg_clips.shape}")
print(f"Test Winter    : {tw_clips.shape} (N:{(tw_labels==0).sum()} A:{(tw_labels==1).sum()})")
print(f"Test Summer    : {ts_clips.shape} (N:{(ts_labels==0).sum()} A:{(ts_labels==1).sum()})")
print(f"Test All       : {all_clips.shape} (N:{(all_labels==0).sum()} A:{(all_labels==1).sum()})")
print(f"Yukleme: {time.time()-t0:.1f}sn")

# ══════════════════════════════════════════════════════════════════════════════
#  GUN BAZLI SPLIT
# ══════════════════════════════════════════════════════════════════════════════
def day_stratified_split(data, names, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    day_to_idx = defaultdict(list); month_to_days = defaultdict(set)
    for i, name in enumerate(names):
        d=str(name)[:8]; m=d[:6]
        day_to_idx[d].append(i); month_to_days[m].add(d)
    val_idx=[]; tr_idx=[]
    for month in sorted(month_to_days):
        days=sorted(month_to_days[month])
        n_val=max(1,round(len(days)*val_ratio))
        shuf=days[:]; rng.shuffle(shuf); vd=set(shuf[:n_val])
        for d in days:
            (val_idx if d in vd else tr_idx).extend(day_to_idx[d])
    return np.array(tr_idx), np.array(val_idx)

tr_idx, val_idx = day_stratified_split(bg_clips, bg_names, CONFIG['val_split'], SEED)
train_data = bg_clips[tr_idx]; val_data = bg_clips[val_idx]
print(f"\nTrain: {len(train_data)}  Val: {len(val_data)}")
del bg_clips; gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════════════════════
class AugDataset(Dataset):
    def __init__(self, data, aug=True):
        self.data=data; self.aug=aug
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        clip=self.data[i].copy()
        if self.aug:
            if random.random()>0.5: clip=clip[:,:,::-1].copy()
            clip=np.clip(clip+random.uniform(-0.1,0.1),0,1)
            clip=np.clip(clip+np.random.normal(0,0.02,clip.shape).astype(np.float32),0,1)
        return torch.tensor(clip).unsqueeze(1)

class TestDS(Dataset):
    def __init__(self, d, l): self.d=d; self.l=l
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

train_dl = DataLoader(AugDataset(train_data,True),  CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(AugDataset(val_data,  False), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
tw_dl    = DataLoader(TestDS(tw_clips,  tw_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
ts_dl    = DataLoader(TestDS(ts_clips,  ts_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
all_dl   = DataLoader(TestDS(all_clips, all_labels), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8
        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']
        self.encoder=nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h), nn.LeakyReLU(0.2))
        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,
                            dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial), nn.LeakyReLU(0.2))
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W=x.shape
        e=self.encoder(x.view(B*T,C,H,W))
        e=self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_=self.lstm(e)
        d=self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)
        return self.decoder(d).view(B,T,1,H,W)

model=CRAE(CONFIG).to(device)
if hasattr(torch,'compile'):
    try: model=torch.compile(model); print("torch.compile aktif")
    except: pass
n_params=sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,}")

# ══════════════════════════════════════════════════════════════════════════════
#  EGITIM
# ══════════════════════════════════════════════════════════════════════════════
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler=optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=2,eta_min=CONFIG['min_lr'])

best_val=float('inf'); pat=0; tl_h=[]; vl_h=[]; lr_h=[]
t_start=time.time()
save_path=SAVE_DIR/f'crae_{MODEL_NAME}.pth'

print(f"\n{'='*60}\nEGITIM — {MODEL_NAME}\n{'='*60}")
for ep in range(CONFIG['epochs']):
    model.train(); el=nb=0
    for batch in train_dl:
        clips=batch.to(device); optimizer.zero_grad()
        loss=criterion(model(clips),clips); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); el+=loss.item(); nb+=1
    tl=el/max(nb,1); tl_h.append(tl)

    model.eval(); vl=nv=0
    with torch.no_grad():
        for batch in val_dl:
            clips=batch.to(device); vl+=criterion(model(clips),clips).item(); nv+=1
    vl/=max(nv,1); vl_h.append(vl)
    lr_now=optimizer.param_groups[0]['lr']; lr_h.append(lr_now)
    scheduler.step()

    if vl<best_val:
        best_val=vl; pat=0
        torch.save({'epoch':ep,'model_state':model.state_dict(),
                    'val_loss':vl,'config':CONFIG,'model_name':MODEL_NAME}, str(save_path))
    else: pat+=1

    elapsed=time.time()-t_start; eta=elapsed/(ep+1)*(CONFIG['epochs']-ep-1)
    print(f"  Ep{ep+1:>3} T:{tl:.6f} V:{vl:.6f} Best:{best_val:.6f} "
          f"Pat:{pat}/{CONFIG['patience']} {elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)
    if pat>=CONFIG['patience']: print(f"  Early stopping ep{ep+1}"); break

total_time=time.time()-t_start
print(f"\nEgitim: {total_time/60:.1f}dk")

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(tl_h,'b-',label='Train',alpha=0.7); axes[0].plot(vl_h,'r-',label='Val',alpha=0.7)
axes[0].set_title(f'Egitim Egrisi — {MODEL_NAME}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(lr_h,'g-'); axes[1].set_title('LR Schedule'); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/training_{MODEL_NAME}.png',dpi=150)
plt.savefig(str(RES_DIR/f'training_{MODEL_NAME}.png'),dpi=150); plt.close()
display(IPImage(f'/content/training_{MODEL_NAME}.png'))

# ══════════════════════════════════════════════════════════════════════════════
#  TEST
# ══════════════════════════════════════════════════════════════════════════════
ck=torch.load(str(save_path)); model.load_state_dict(ck['model_state']); model.eval()

def run_test(dl, tag):
    scores=[]; labs=[]
    with torch.no_grad():
        for clips,lbls in dl:
            clips=clips.to(device); out=model(clips)
            for i in range(clips.shape[0]):
                scores.append(torch.mean((clips[i]-out[i])**2).item())
                labs.append(lbls[i].item())
    scores=np.array(scores); labs=np.array(labs)
    ns=scores[labs==0]; als=scores[labs==1]
    auc=roc_auc_score(labs,scores)
    prec,rec,thr=precision_recall_curve(labs,scores)
    f1s=2*prec*rec/(prec+rec+1e-8); opt_t=thr[np.argmax(f1s)]
    preds=(scores>=opt_t).astype(int)
    cm=confusion_matrix(labs,preds); tn,fp,fn,tp=cm.ravel()
    fpr_r,tpr_r,thr_r=roc_curve(labs,scores); fnr_r=1-tpr_r
    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))
    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)

    print(f"\n  {'='*50}\n  {tag}\n  {'='*50}")
    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")
    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")
    print(f"  AUC: {auc:.4f}  EER: {eer:.4f}  F1: {f1s.max():.4f}")
    print(f"  Prec: {tp/(tp+fp+1e-8):.4f}  Rec: {tp/(tp+fn+1e-8):.4f}  Thr: {opt_t:.6f}")
    print(f"  TP:{tp} FP:{fp} FN:{fn} TN:{tn}")

    fig,axes=plt.subplots(1,2,figsize=(12,5))
    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
    axes[0].set_title(f'ROC — {tag}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')
    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')
    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')
    axes[1].set_title(f'Score Dagilimi — {tag}'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
    plt.tight_layout()
    fname=f'{MODEL_NAME}_{tag.lower().replace(" ","_")}'
    plt.savefig(f'/content/{fname}.png',dpi=150)
    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=150); plt.close()

    return {'tag':tag,'auc':float(auc),'eer':float(eer),'f1':float(f1s.max()),
            'threshold':float(opt_t),'precision':float(tp/(tp+fp+1e-8)),
            'recall':float(tp/(tp+fn+1e-8)),'specificity':float(tn/(tn+fp+1e-8)),
            'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),
            'n_normal':int((labs==0).sum()),'n_anomaly':int((labs==1).sum()),
            'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn)}

print(f"\n{'='*60}\nTEST\n{'='*60}")
r_winter=run_test(tw_dl,  'WINTER')
r_summer=run_test(ts_dl,  'SUMMER')
r_all   =run_test(all_dl, 'ALL')

results={'model_name':MODEL_NAME,'training_data':'pure_bg_summer','config':CONFIG,
         'n_params':n_params,'train_clips':len(train_data),'val_clips':len(val_data),
         'best_val_loss':float(best_val),'best_epoch':int(ck['epoch'])+1,
         'total_epochs':len(tl_h),'training_time_min':round(total_time/60,1),
         'test_winter':r_winter,'test_summer':r_summer,'test_all':r_all,
         'train_losses':[float(x) for x in tl_h],'val_losses':[float(x) for x in vl_h]}
out_json=RES_DIR/f'model2_{MODEL_NAME}_results.json'
with open(str(out_json),'w') as f: json.dump(results,f,indent=2)

print(f"\n{'='*60}\nOZET — Model 2: {MODEL_NAME}\n{'='*60}")
print(f"  {'Test Seti':<12} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print(f"  {'-'*55}")
for r in [r_winter,r_summer,r_all]:
    print(f"  {r['tag']:<12} {r['auc']:>8.4f} {r['eer']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")
print(f"\n  Model: {save_path}\n  JSON : {out_json}")
gc.collect(); print("\nTamamlandi!")

In [ ]:
import torch, torch.nn as nn
import numpy as np, json
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NPY_DIR  = Path('/content/drive/MyDrive/archive/final_npycache')
RES_DIR  = Path('/content/drive/MyDrive/archive/diff_technic_results')
SAVE_DIR = Path('/content/drive/MyDrive/archive/models/cc')

# ── Model yükle (kış eğitimli) ────────────────────────────────────────────────
CONFIG = {
    'frame_size':128,'frames_per_clip':16,'batch_size':32,'val_split':0.15,
    'epochs':100,'lr':1e-3,'min_lr':1e-6,'lstm_hidden':256,'lstm_layers':2,
    'dropout':0.3,'patience':15,'weight_decay':1e-5,'encoder_filters':[32,64,128,256],
}

class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8
        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']
        self.encoder=nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1),nn.BatchNorm2d(f[3]),nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h),nn.LeakyReLU(0.2))
        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,
                            dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial),nn.LeakyReLU(0.2))
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1),nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W=x.shape
        e=self.encoder(x.view(B*T,C,H,W))
        e=self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_=self.lstm(e)
        d=self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)
        return self.decoder(d).view(B,T,1,H,W)

ck = torch.load(str(SAVE_DIR/'crae_winter_only.pth'), map_location=device)
model = CRAE(CONFIG).to(device)
state = {k.replace('_orig_mod.',''):v for k,v in ck['model_state'].items()}
model.load_state_dict(state)
model.eval()
print("Model yuklendi")

# ── Skorlama fonksiyonu ───────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class TestDS(Dataset):
    def __init__(self, d, l): self.d=d; self.l=l
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

def score_all(clips, labels, names):
    ds = TestDS(clips, labels)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
    scores = []
    with torch.no_grad():
        for batch_clips, _ in dl:
            batch_clips = batch_clips.to(device)
            out = model(batch_clips)
            for i in range(batch_clips.shape[0]):
                scores.append(torch.mean((batch_clips[i]-out[i])**2).item())
    return np.array(scores)

# ── Threshold hesapla (winter modelinin JSON'undan) ───────────────────────────
from sklearn.metrics import roc_auc_score, precision_recall_curve

def get_threshold(scores, labels):
    prec,rec,thr = precision_recall_curve(labels, scores)
    f1s = 2*prec*rec/(prec+rec+1e-8)
    return thr[np.argmax(f1s)]

# ══════════════════════════════════════════════════════════════════════════════
#  WINTER ANALİZİ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("WINTER HATA ANALİZİ")
print("="*60)

tw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)
tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))
tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)

tw_scores = score_all(tw_clips, tw_labels, tw_names)
tw_thr    = get_threshold(tw_scores, tw_labels)
tw_preds  = (tw_scores >= tw_thr).astype(int)
tw_auc    = roc_auc_score(tw_labels, tw_scores)

print(f"AUC: {tw_auc:.4f}  Threshold: {tw_thr:.6f}")

# Yanlış kararlar
fn_winter = []  # Kaçırılan anomali (label=1, pred=0)
fp_winter = []  # Yanlış alarm (label=0, pred=1)

for i, (name, label, pred, score) in enumerate(zip(tw_names, tw_labels, tw_preds, tw_scores)):
    day = str(name)[:8]
    if label == 1 and pred == 0:  # FN — kaçırıldı
        fn_winter.append({'clip': str(name), 'day': day, 'score': float(score),
                          'label': int(label), 'pred': int(pred), 'type': 'FN_missed'})
    elif label == 0 and pred == 1:  # FP — yanlış alarm
        fp_winter.append({'clip': str(name), 'day': day, 'score': float(score),
                          'label': int(label), 'pred': int(pred), 'type': 'FP_false_alarm'})

print(f"\nKacirilan anomali (FN): {len(fn_winter)}")
print(f"Yanlis alarm     (FP): {len(fp_winter)}")

# Gün bazlı özet
fn_days = defaultdict(int)
for r in fn_winter: fn_days[r['day']] += 1
fp_days = defaultdict(int)
for r in fp_winter: fp_days[r['day']] += 1

print(f"\n--- WINTER KACIRILAN ANOMALİLER (FN) ---")
print(f"{'Gun':<12} {'Adet':>5}  {'Klip isimleri'}")
print("-"*60)
day_fn = defaultdict(list)
for r in fn_winter: day_fn[r['day']].append(r['clip'])
for day in sorted(day_fn):
    clips_str = ', '.join(day_fn[day][:5])
    if len(day_fn[day]) > 5: clips_str += f' ... (+{len(day_fn[day])-5})'
    print(f"  {day:<12} {len(day_fn[day]):>5}  {clips_str}")

print(f"\n--- WINTER YANLIS ALARMLAR (FP) ---")
print(f"{'Gun':<12} {'Adet':>5}  {'Klip isimleri'}")
print("-"*60)
day_fp = defaultdict(list)
for r in fp_winter: day_fp[r['day']].append(r['clip'])
for day in sorted(day_fp):
    clips_str = ', '.join(day_fp[day][:5])
    if len(day_fp[day]) > 5: clips_str += f' ... (+{len(day_fp[day])-5})'
    print(f"  {day:<12} {len(day_fp[day]):>5}  {clips_str}")

# ══════════════════════════════════════════════════════════════════════════════
#  SUMMER ANALİZİ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("SUMMER HATA ANALİZİ")
print("="*60)

ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)
ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))
ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)

ts_scores = score_all(ts_clips, ts_labels, ts_names)
ts_thr    = get_threshold(ts_scores, ts_labels)
ts_preds  = (ts_scores >= ts_thr).astype(int)
ts_auc    = roc_auc_score(ts_labels, ts_scores)

print(f"AUC: {ts_auc:.4f}  Threshold: {ts_thr:.6f}")

fn_summer = []; fp_summer = []
for i, (name, label, pred, score) in enumerate(zip(ts_names, ts_labels, ts_preds, ts_scores)):
    day = str(name)[:8]
    if label == 1 and pred == 0:
        fn_summer.append({'clip':str(name),'day':day,'score':float(score),
                          'label':int(label),'pred':int(pred),'type':'FN_missed'})
    elif label == 0 and pred == 1:
        fp_summer.append({'clip':str(name),'day':day,'score':float(score),
                          'label':int(label),'pred':int(pred),'type':'FP_false_alarm'})

print(f"\nKacirilan anomali (FN): {len(fn_summer)}")
print(f"Yanlis alarm     (FP): {len(fp_summer)}")

print(f"\n--- SUMMER KACIRILAN ANOMALİLER (FN) ---")
print(f"{'Gun':<12} {'Adet':>5}  {'Klip isimleri'}")
print("-"*60)
day_fn_s = defaultdict(list)
for r in fn_summer: day_fn_s[r['day']].append(r['clip'])
for day in sorted(day_fn_s):
    clips_str = ', '.join(day_fn_s[day][:5])
    if len(day_fn_s[day]) > 5: clips_str += f' ... (+{len(day_fn_s[day])-5})'
    print(f"  {day:<12} {len(day_fn_s[day]):>5}  {clips_str}")

print(f"\n--- SUMMER YANLIS ALARMLAR (FP) ---")
print(f"{'Gun':<12} {'Adet':>5}  {'Klip isimleri'}")
print("-"*60)
day_fp_s = defaultdict(list)
for r in fp_summer: day_fp_s[r['day']].append(r['clip'])
for day in sorted(day_fp_s):
    clips_str = ', '.join(day_fp_s[day][:5])
    if len(day_fp_s[day]) > 5: clips_str += f' ... (+{len(day_fp_s[day])-5})'
    print(f"  {day:<12} {len(day_fp_s[day]):>5}  {clips_str}")

# ══════════════════════════════════════════════════════════════════════════════
#  JSON KAYDET + GRAFIK
# ══════════════════════════════════════════════════════════════════════════════
error_analysis = {
    'model': 'winter_only',
    'winter': {
        'auc': float(tw_auc), 'threshold': float(tw_thr),
        'fn_count': len(fn_winter), 'fp_count': len(fp_winter),
        'fn_by_day': {d: len(v) for d,v in day_fn.items()},
        'fp_by_day': {d: len(v) for d,v in day_fp.items()},
        'fn_clips': fn_winter, 'fp_clips': fp_winter,
    },
    'summer': {
        'auc': float(ts_auc), 'threshold': float(ts_thr),
        'fn_count': len(fn_summer), 'fp_count': len(fp_summer),
        'fn_by_day': {d: len(v) for d,v in day_fn_s.items()},
        'fp_by_day': {d: len(v) for d,v in day_fp_s.items()},
        'fn_clips': fn_summer, 'fp_clips': fp_summer,
    }
}
out_json = RES_DIR / 'winter_only_error_analysis.json'
with open(str(out_json), 'w') as f: json.dump(error_analysis, f, indent=2)
print(f"\nJSON: {out_json}")

# Grafik — gün bazlı FN/FP dağılımı
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Winter Only Model — Hata Analizi', fontsize=13, fontweight='bold')

for ax, day_dict, title, color in [
    (axes[0,0], day_fn,   'Winter FN (Kacirilan Anomali)', 'red'),
    (axes[0,1], day_fp,   'Winter FP (Yanlis Alarm)',      'orange'),
    (axes[1,0], day_fn_s, 'Summer FN (Kacirilan Anomali)', 'darkred'),
    (axes[1,1], day_fp_s, 'Summer FP (Yanlis Alarm)',      'darkorange'),
]:
    if day_dict:
        days = sorted(day_dict.keys())
        counts = [len(day_dict[d]) for d in days]
        ax.bar(range(len(days)), counts, color=color, alpha=0.8)
        ax.set_xticks(range(len(days)))
        ax.set_xticklabels(days, rotation=45, ha='right', fontsize=7)
        ax.set_title(f'{title} (toplam: {sum(counts)})')
        ax.set_ylabel('Klip Sayisi')
        ax.grid(axis='y', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Hata yok!', ha='center', va='center',
                transform=ax.transAxes, fontsize=14, color='green')
        ax.set_title(title)

plt.tight_layout()
plt.savefig('/content/error_analysis_winter_only.png', dpi=150)
plt.savefig(str(RES_DIR/'error_analysis_winter_only.png'), dpi=150)
plt.close()
display(IPImage('/content/error_analysis_winter_only.png'))
print("Tamamlandi!")

In [ ]:
"""
crae_error_analysis.py
=======================
Kis eğitimli + Yaz eğitimli model için:
- Winter test + Summer test hata analizi
- FN (kaçırılan anomali) + FP (yanlış alarm)
- Tam Drive yolu: testing_final_winter/YYYYMMDD/label/clip
- İki modelde ortak yanlış sınıflandırmalar
"""

import torch, torch.nn as nn
import numpy as np, json
from pathlib import Path
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, precision_recall_curve
from IPython.display import display, Image as IPImage

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

NPY_DIR   = Path('/content/drive/MyDrive/archive/final_npycache')
SAVE_DIR  = Path('/content/drive/MyDrive/archive/models')
RES_DIR   = Path('/content/drive/MyDrive/archive/diff_technic_results')
BASE_AE   = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing')
TEST_WIN  = BASE_AE / 'testing_final_winter'
TEST_SUM  = BASE_AE / 'testing_final_summer'
RES_DIR.mkdir(exist_ok=True)

CONFIG = {
    'frame_size':128,'frames_per_clip':16,'batch_size':32,'val_split':0.15,
    'epochs':100,'lr':1e-3,'min_lr':1e-6,'lstm_hidden':256,'lstm_layers':2,
    'dropout':0.3,'patience':15,'weight_decay':1e-5,'encoder_filters':[32,64,128,256],
}

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8
        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']
        self.encoder=nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1),nn.BatchNorm2d(f[3]),nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h),nn.LeakyReLU(0.2))
        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,
                            dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial),nn.LeakyReLU(0.2))
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1),nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W=x.shape
        e=self.encoder(x.view(B*T,C,H,W))
        e=self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_=self.lstm(e)
        d=self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)
        return self.decoder(d).view(B,T,1,H,W)

def load_model(path):
    ck    = torch.load(str(path), map_location=device)
    m     = CRAE(CONFIG).to(device)
    state = {k.replace('_orig_mod.',''):v for k,v in ck['model_state'].items()}
    m.load_state_dict(state); m.eval()
    return m

# ══════════════════════════════════════════════════════════════════════════════
#  NPY → clip_name : (day, label, drive_path) haritas
# ══════════════════════════════════════════════════════════════════════════════
def build_path_map(test_root):
    """
    test_root/YYYYMMDD/normal|anomaly/clip_xxx
    Returns: {clip_name: {'day':..., 'label':..., 'drive_path':...}}
    """
    path_map = {}
    for day_dir in sorted(test_root.iterdir()):
        if not day_dir.is_dir() or not day_dir.name.isdigit(): continue
        for lbl_dir in sorted(day_dir.iterdir()):
            if not lbl_dir.is_dir(): continue
            lbl_val = 0 if lbl_dir.name == 'normal' else 1
            for clip_dir in sorted(lbl_dir.iterdir()):
                if not clip_dir.is_dir(): continue
                path_map[clip_dir.name] = {
                    'day'       : day_dir.name,
                    'label'     : lbl_val,
                    'drive_path': str(clip_dir),
                }
    return path_map

print("Path haritalari olusturuluyor...")
win_map = build_path_map(TEST_WIN)
sum_map = build_path_map(TEST_SUM)
print(f"Winter: {len(win_map)} klip  Summer: {len(sum_map)} klip")

# ══════════════════════════════════════════════════════════════════════════════
#  SKORLAMA
# ══════════════════════════════════════════════════════════════════════════════
class TestDS(Dataset):
    def __init__(self, d, l): self.d=d; self.l=l
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

def score_clips(model, clips, labels):
    ds = TestDS(clips, labels)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
    scores = []
    with torch.no_grad():
        for bc,_ in dl:
            bc=bc.to(device); out=model(bc)
            for i in range(bc.shape[0]):
                scores.append(torch.mean((bc[i]-out[i])**2).item())
    return np.array(scores)

def get_optimal_threshold(scores, labels):
    prec,rec,thr = precision_recall_curve(labels, scores)
    f1s = 2*prec*rec/(prec+rec+1e-8)
    return float(thr[np.argmax(f1s)])

# ══════════════════════════════════════════════════════════════════════════════
#  HATA ANALİZİ FONKSİYONU
# ══════════════════════════════════════════════════════════════════════════════
def analyze_errors(scores, labels, names, path_map, tag):
    thr   = get_optimal_threshold(scores, labels)
    preds = (scores >= thr).astype(int)
    auc   = roc_auc_score(labels, scores)

    fn_list = []  # label=1, pred=0 — kaçırılan anomali
    fp_list = []  # label=0, pred=1 — yanlış alarm

    for name, label, pred, score in zip(names, labels, preds, scores):
        info = path_map.get(str(name), {})
        day  = info.get('day', 'unknown')
        dpath= info.get('drive_path', f'BULUNAMADI/{name}')
        entry = {
            'clip'       : str(name),
            'day'        : day,
            'score'      : round(float(score), 6),
            'threshold'  : round(float(thr), 6),
            'label'      : int(label),
            'pred'       : int(pred),
            'drive_path' : dpath,
        }
        if label == 1 and pred == 0:
            fn_list.append(entry)
        elif label == 0 and pred == 1:
            fp_list.append(entry)

    # Gün bazlı grupla
    fn_by_day = defaultdict(list)
    fp_by_day = defaultdict(list)
    for r in fn_list: fn_by_day[r['day']].append(r)
    for r in fp_list: fp_by_day[r['day']].append(r)

    print(f"\n{'='*65}")
    print(f"{tag}  AUC:{auc:.4f}  Thr:{thr:.6f}")
    print(f"{'='*65}")
    print(f"  Kacirilan anomali (FN): {len(fn_list)}")
    print(f"  Yanlis alarm     (FP): {len(fp_list)}")

    print(f"\n  --- FN: KACIRILAN ANOMALİLER ---")
    print(f"  {'Gun':<12} {'Klip':<20} {'Score':>10}  Drive Yolu")
    print(f"  {'-'*80}")
    for day in sorted(fn_by_day):
        for r in sorted(fn_by_day[day], key=lambda x: x['score']):
            print(f"  {r['day']:<12} {r['clip']:<20} {r['score']:>10.6f}  {r['drive_path']}")

    print(f"\n  --- FP: YANLIŞ ALARMLAR ---")
    print(f"  {'Gun':<12} {'Klip':<20} {'Score':>10}  Drive Yolu")
    print(f"  {'-'*80}")
    for day in sorted(fp_by_day):
        for r in sorted(fp_by_day[day], key=lambda x: x['score'], reverse=True):
            print(f"  {r['day']:<12} {r['clip']:<20} {r['score']:>10.6f}  {r['drive_path']}")

    return {
        'tag': tag, 'auc': float(auc), 'threshold': float(thr),
        'fn_count': len(fn_list), 'fp_count': len(fp_list),
        'fn': fn_list, 'fp': fp_list,
    }

# ══════════════════════════════════════════════════════════════════════════════
#  VERİ YUKLE
# ══════════════════════════════════════════════════════════════════════════════
print("\nVeri yukleniyor...")
tw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)
tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))
tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)
ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)
ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))
ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)
print(f"Winter: {tw_clips.shape}  Summer: {ts_clips.shape}")

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL 1: KIŞ EĞİTİMLİ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("MODEL 1: KIS EGİTİMLİ (crae_winter_only.pth)")
print("="*65)
m1 = load_model(SAVE_DIR/'crae_winter_only.pth')
print("Model yuklendi")

m1_tw_scores = score_clips(m1, tw_clips, tw_labels)
m1_ts_scores = score_clips(m1, ts_clips, ts_labels)

m1_win = analyze_errors(m1_tw_scores, tw_labels, tw_names, win_map, 'M1-WINTER')
m1_sum = analyze_errors(m1_ts_scores, ts_labels, ts_names, sum_map, 'M1-SUMMER')

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL 2: YAZ EĞİTİMLİ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("MODEL 2: YAZ EGİTİMLİ (crae_summer_only.pth)")
print("="*65)
m2 = load_model(SAVE_DIR/'crae_summer_only.pth')
print("Model yuklendi")

m2_tw_scores = score_clips(m2, tw_clips, tw_labels)
m2_ts_scores = score_clips(m2, ts_clips, ts_labels)

m2_win = analyze_errors(m2_tw_scores, tw_labels, tw_names, win_map, 'M2-WINTER')
m2_sum = analyze_errors(m2_ts_scores, ts_labels, ts_names, sum_map, 'M2-SUMMER')

# ══════════════════════════════════════════════════════════════════════════════
#  ORTAK YANLIŞ SINIFLANDIRMALAR
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("ORTAK YANLIŞ SINIFLANDIRMALAR")
print("="*65)

def find_common(res1, res2, error_type, tag):
    clips1 = {r['clip'] for r in res1[error_type]}
    clips2 = {r['clip'] for r in res2[error_type]}
    common = clips1 & clips2

    map1 = {r['clip']: r for r in res1[error_type]}

    print(f"\n  {tag} — Ortak {error_type.upper()} ({len(common)} klip):")
    if not common:
        print("  Ortak hata yok!")
        return []

    results = []
    print(f"  {'Klip':<20} {'Gun':<12} {'M1 Score':>10}  {'M2 Score':>10}  Drive Yolu")
    print(f"  {'-'*80}")
    for clip in sorted(common):
        r1 = map1[clip]
        r2 = {r['clip']:r for r in res2[error_type]}[clip]
        print(f"  {clip:<20} {r1['day']:<12} {r1['score']:>10.6f}  {r2['score']:>10.6f}  {r1['drive_path']}")
        results.append({
            'clip': clip, 'day': r1['day'],
            'm1_score': r1['score'], 'm2_score': r2['score'],
            'drive_path': r1['drive_path']
        })
    return results

common_win_fn = find_common(m1_win, m2_win, 'fn', 'WINTER')
common_win_fp = find_common(m1_win, m2_win, 'fp', 'WINTER')
common_sum_fn = find_common(m1_sum, m2_sum, 'fn', 'SUMMER')
common_sum_fp = find_common(m1_sum, m2_sum, 'fp', 'SUMMER')

# ══════════════════════════════════════════════════════════════════════════════
#  GRAFİK
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('Hata Analizi — M1(Kis) vs M2(Yaz)', fontsize=13, fontweight='bold')

pairs = [
    (axes[0,0], m1_win['fn'], 'M1 Winter FN', 'red'),
    (axes[0,1], m1_win['fp'], 'M1 Winter FP', 'orange'),
    (axes[0,2], m1_sum['fn'], 'M1 Summer FN', 'darkred'),
    (axes[0,3], m1_sum['fp'], 'M1 Summer FP', 'darkorange'),
    (axes[1,0], m2_win['fn'], 'M2 Winter FN', 'blue'),
    (axes[1,1], m2_win['fp'], 'M2 Winter FP', 'steelblue'),
    (axes[1,2], m2_sum['fn'], 'M2 Summer FN', 'darkblue'),
    (axes[1,3], m2_sum['fp'], 'M2 Summer FP', 'cornflowerblue'),
]

for ax, items, title, color in pairs:
    if items:
        day_cnt = defaultdict(int)
        for r in items: day_cnt[r['day']] += 1
        days   = sorted(day_cnt.keys())
        counts = [day_cnt[d] for d in days]
        ax.bar(range(len(days)), counts, color=color, alpha=0.8)
        ax.set_xticks(range(len(days)))
        ax.set_xticklabels(days, rotation=45, ha='right', fontsize=6)
        ax.set_title(f'{title}\n(toplam: {len(items)})', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Hata yok!', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='green')
        ax.set_title(title, fontsize=9)
    ax.set_ylabel('Klip sayisi')

plt.tight_layout()
plt.savefig('/content/error_analysis_both_models.png', dpi=130, bbox_inches='tight')
plt.savefig(str(RES_DIR/'error_analysis_both_models.png'), dpi=130, bbox_inches='tight')
plt.close()
display(IPImage('/content/error_analysis_both_models.png'))

# ══════════════════════════════════════════════════════════════════════════════
#  JSON KAYDET
# ══════════════════════════════════════════════════════════════════════════════
output = {
    'model1_winter': m1_win,
    'model1_summer': m1_sum,
    'model2_winter': m2_win,
    'model2_summer': m2_sum,
    'common_errors': {
        'winter_fn': common_win_fn,
        'winter_fp': common_win_fp,
        'summer_fn': common_sum_fn,
        'summer_fp': common_sum_fp,
    }
}
out_json = RES_DIR / 'error_analysis_both_models.json'
with open(str(out_json), 'w') as f: json.dump(output, f, indent=2)

print(f"\n{'='*65}")
print(f"OZET")
print(f"{'='*65}")
print(f"  {'':25} {'FN':>6} {'FP':>6} {'AUC':>8}")
print(f"  {'-'*50}")
for tag, res in [('M1 Winter', m1_win), ('M1 Summer', m1_sum),
                  ('M2 Winter', m2_win), ('M2 Summer', m2_sum)]:
    print(f"  {tag:<25} {res['fn_count']:>6} {res['fp_count']:>6} {res['auc']:>8.4f}")

print(f"\n  Ortak hatalar:")
print(f"  Winter FN: {len(common_win_fn)}  Winter FP: {len(common_win_fp)}")
print(f"  Summer FN: {len(common_sum_fn)}  Summer FP: {len(common_sum_fp)}")
print(f"\n  JSON: {out_json}")
print(f"  PNG : {RES_DIR/'error_analysis_both_models.png'}")
print("\nTamamlandi!")

In [ ]:
import torch, torch.nn as nn
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, precision_recall_curve,
                              roc_curve, confusion_matrix, classification_report)
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TEST_WIN = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final_winter')
TEST_SUM = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final_summer')
SAVE_DIR = Path('/content/drive/MyDrive/archive/models')
RES_DIR  = Path('/content/drive/MyDrive/archive/diff_technic_results')

WORKERS    = 128
BATCH      = 64
FRAME_SIZE = 128
CLIP_LEN   = 16

# ── Model ─────────────────────────────────────────────────────────────────────
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8
        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']
        self.encoder=nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1),nn.BatchNorm2d(f[3]),nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h),nn.LeakyReLU(0.2))
        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,
                            dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial),nn.LeakyReLU(0.2))
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1),nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W=x.shape
        e=self.encoder(x.view(B*T,C,H,W))
        e=self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_=self.lstm(e)
        d=self.unproj(e.reshape(B*T,-1)).view(B*T,self.spatial//64,8,8)
        return self.decoder(d).view(B,T,1,H,W)

def load_model(path):
    ck=torch.load(str(path),map_location=device)
    cfg=ck.get('config',{'encoder_filters':[32,64,128,256],
                          'lstm_hidden':256,'lstm_layers':2,'dropout':0.3})
    m=CRAE(cfg).to(device)
    state={k.replace('_orig_mod.',''):v for k,v in ck['model_state'].items()}
    m.load_state_dict(state); m.eval()
    return m, cfg

# ── Veri yükleme ──────────────────────────────────────────────────────────────
def read_clip(clip_dir):
    frames=sorted(clip_dir.glob("frame*.jpg"),
                  key=lambda p: int(''.join(filter(str.isdigit,p.stem))))
    if len(frames)<CLIP_LEN: return None
    imgs=[]
    for fr in frames[:CLIP_LEN]:
        img=cv2.imread(str(fr),cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        img=cv2.resize(img,(FRAME_SIZE,FRAME_SIZE))
        imgs.append(img.astype(np.float32)/255.0)
    return np.stack(imgs)

def load_test_set(test_root, tag):
    clip_dirs=[]; labels=[]; names=[]
    for day_dir in sorted(test_root.iterdir()):
        if not day_dir.is_dir() or not day_dir.name.isdigit(): continue
        for lbl_dir in sorted(day_dir.iterdir()):
            if not lbl_dir.is_dir(): continue
            lbl_name=lbl_dir.name.lower()
            if 'anomal' in lbl_name:   lbl_val=1
            elif 'normal' in lbl_name: lbl_val=0
            else: continue
            for clip_dir in sorted(lbl_dir.iterdir()):
                if not clip_dir.is_dir(): continue
                clip_dirs.append(clip_dir)
                labels.append(lbl_val)
                names.append(clip_dir.name)

    n_norm=sum(1 for l in labels if l==0)
    n_anom=sum(1 for l in labels if l==1)
    print(f"  {tag}: {len(clip_dirs)} klip  Normal:{n_norm}  Anomali:{n_anom}")

    clips_list=[None]*len(clip_dirs)
    for i in tqdm(range(0,len(clip_dirs),BATCH),desc=f"  {tag}"):
        batch=clip_dirs[i:i+BATCH]
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futs={ex.submit(read_clip,d):j for j,d in enumerate(batch)}
            for fut in as_completed(futs):
                clips_list[i+futs[fut]]=fut.result()

    valid=[(c,l,n) for c,l,n in zip(clips_list,labels,names) if c is not None]
    clips=np.stack([v[0] for v in valid]).astype(np.float32)
    labs =np.array([v[1] for v in valid])
    nms  =np.array([v[2] for v in valid])
    return clips, labs, nms

class TestDS(Dataset):
    def __init__(self,d,l): self.d=d; self.l=l
    def __len__(self): return len(self.d)
    def __getitem__(self,i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]

# ── Değerlendirme ─────────────────────────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import json

def evaluate_full(model, clips, labels, tag, model_name):
    ds=TestDS(clips,labels)
    dl=DataLoader(ds,batch_size=64,shuffle=False,num_workers=2)
    scores=[]
    with torch.no_grad():
        for bc,_ in dl:
            bc=bc.to(device); out=model(bc)
            for i in range(bc.shape[0]):
                scores.append(torch.mean((bc[i]-out[i])**2).item())
    scores=np.array(scores)

    auc=roc_auc_score(labels,scores)
    prec,rec,thr=precision_recall_curve(labels,scores)
    f1s=2*prec*rec/(prec+rec+1e-8)
    opt_t=thr[np.argmax(f1s)]
    preds=(scores>=opt_t).astype(int)
    cm=confusion_matrix(labels,preds)
    tn,fp,fn,tp=cm.ravel()
    fpr_r,tpr_r,thr_r=roc_curve(labels,scores)
    fnr_r=1-tpr_r
    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))
    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)
    ns=scores[labels==0]; als=scores[labels==1]

    print(f"\n  {'='*55}")
    print(f"  {model_name} — {tag}")
    print(f"  {'='*55}")
    print(f"  AUC        : {auc:.4f}")
    print(f"  EER        : {eer:.4f} ({eer*100:.2f}%)")
    print(f"  F1         : {f1s.max():.4f}")
    print(f"  Precision  : {tp/(tp+fp+1e-8):.4f}")
    print(f"  Recall     : {tp/(tp+fn+1e-8):.4f}")
    print(f"  Specificity: {tn/(tn+fp+1e-8):.4f}")
    print(f"  Threshold  : {opt_t:.6f}")
    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")
    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")
    print(f"\n  Confusion Matrix:")
    print(f"              Pred_N  Pred_A")
    print(f"  Actual_N    {tn:>6}  {fp:>6}")
    print(f"  Actual_A    {fn:>6}  {tp:>6}")
    print(f"\n{classification_report(labels,preds,target_names=['Normal','Anomali'],digits=4)}")

    # Grafik
    fig,axes=plt.subplots(1,3,figsize=(18,5))
    fig.suptitle(f'{model_name} — {tag}',fontsize=12,fontweight='bold')

    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
    axes[0].set_title('ROC'); axes[0].legend(); axes[0].grid(True,alpha=0.3)

    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')
    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')
    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')
    axes[1].set_title('Score Dagilimi'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

    im=axes[2].imshow(cm,cmap='Blues')
    axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
    axes[2].set_xticklabels(['Normal','Anomali'])
    axes[2].set_yticklabels(['Normal','Anomali'])
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
    axes[2].set_title('Confusion Matrix')
    for i in range(2):
        for j in range(2):
            axes[2].text(j,i,f'{cm[i,j]}',ha='center',va='center',
                         fontsize=16,color='white' if cm[i,j]>cm.max()/2 else 'black')
    plt.tight_layout()
    fname=f'{model_name}_{tag.lower()}_final'
    plt.savefig(f'/content/{fname}.png',dpi=130)
    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=130); plt.close()
    display(IPImage(f'/content/{fname}.png'))

    return {
        'tag':tag,'model':model_name,'auc':float(auc),'eer':float(eer),
        'f1':float(f1s.max()),'threshold':float(opt_t),
        'precision':float(tp/(tp+fp+1e-8)),'recall':float(tp/(tp+fn+1e-8)),
        'specificity':float(tn/(tn+fp+1e-8)),
        'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),
        'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),
        'n_normal':int((labels==0).sum()),'n_anomaly':int((labels==1).sum()),
    }

# ══════════════════════════════════════════════════════════════════════════════
print("Veri yukleniyor...")
tw_clips,tw_labels,tw_names = load_test_set(TEST_WIN,"Winter")
ts_clips,ts_labels,ts_names = load_test_set(TEST_SUM,"Summer")
all_clips  = np.concatenate([tw_clips,ts_clips])
all_labels = np.concatenate([tw_labels,ts_labels])

# Model 1
print("\n" + "="*60)
print("MODEL 1: crae_winter_only.pth")
print("="*60)
m1,_ = load_model(SAVE_DIR/'crae_winter_only.pth')
r_m1_w   = evaluate_full(m1, tw_clips,  tw_labels,  'WINTER', 'M1_WinterOnly')
r_m1_s   = evaluate_full(m1, ts_clips,  ts_labels,  'SUMMER', 'M1_WinterOnly')
r_m1_all = evaluate_full(m1, all_clips, all_labels, 'ALL',    'M1_WinterOnly')
del m1; gc.collect(); torch.cuda.empty_cache()

# Model 2
print("\n" + "="*60)
print("MODEL 2: crae_summer_only.pth")
print("="*60)
m2,_ = load_model(SAVE_DIR/'crae_summer_only.pth')
r_m2_w   = evaluate_full(m2, tw_clips,  tw_labels,  'WINTER', 'M2_SummerOnly')
r_m2_s   = evaluate_full(m2, ts_clips,  ts_labels,  'SUMMER', 'M2_SummerOnly')
r_m2_all = evaluate_full(m2, all_clips, all_labels, 'ALL',    'M2_SummerOnly')
del m2; gc.collect(); torch.cuda.empty_cache()

# Özet
print(f"\n{'='*65}")
print("FINAL OZET")
print(f"{'='*65}")
print(f"  {'':20} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print(f"  {'-'*55}")
for r in [r_m1_w,r_m1_s,r_m1_all,r_m2_w,r_m2_s,r_m2_all]:
    label=f"{r['model']}_{r['tag']}"
    print(f"  {label:<20} {r['auc']:>8.4f} {r['eer']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")

# JSON
all_results = {
    'm1_winter':r_m1_w,'m1_summer':r_m1_s,'m1_all':r_m1_all,
    'm2_winter':r_m2_w,'m2_summer':r_m2_s,'m2_all':r_m2_all,
}
with open(str(RES_DIR/'final_model_results.json'),'w') as f:
    json.dump(all_results,f,indent=2)
print(f"\nJSON: {RES_DIR/'final_model_results.json'}")
print("Tamamlandi!")

In [ ]:
"""
CR-AE Test — NPY'siz, doğrudan Drive'dan
==========================================
Preprocess: build_npy_cache.py ile BİREBİR AYNI
  - cv2 grayscale + /255
  - (B, T, 1, H, W) input
  - skor = mean MSE (tüm T,C,H,W)
  - LeakyReLU(0.2), Dropout2d encoder'da
"""

import os, cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)
import warnings; warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# AYARLAR
# ─────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE   = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")
MODELS = {
    "winter": BASE / "models/crae_winter_only.pth",
    "summer": BASE / "models/crae_summer_only.pth",
}
TESTS = {
    "winter": BASE / "testing/testing_final_winter",
    "summer": BASE / "testing/testing_final_summer",
}
FRAME_SIZE      = 128
CLIP_LEN        = 16
BATCH_SIZE      = 64
WORKERS         = 32
ANOMALY_CLASSES = {"anomaly", "trespassing", "loitering", "object_abandonment"}

print(f"Device: {DEVICE}")

# ─────────────────────────────────────────────
# MODEL  (build_npy_cache'deki CRAE ile birebir)
# ─────────────────────────────────────────────
CONFIG = {
    "encoder_filters": [32, 64, 128, 256],
    "lstm_hidden": 256, "lstm_layers": 2,
    "dropout": 0.3, "frame_size": 128,
}

class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg["encoder_filters"]
        self.spatial = f[-1] * 8 * 8
        lstm_h = cfg["lstm_hidden"]
        drop   = cfg["dropout"]

        self.encoder = nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm   = nn.LSTM(lstm_h, lstm_h, cfg["lstm_layers"], batch_first=True,
                              dropout=drop if cfg["lstm_layers"] > 1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        e = self.encoder(x.view(B*T, C, H, W))
        e = self.proj(e.view(B*T, -1)).view(B, T, -1)
        e, _ = self.lstm(e)
        d = self.unproj(e.reshape(B*T, -1)).view(B*T, 256, 8, 8)
        return self.decoder(d).view(B, T, 1, H, W)


def load_model(path):
    ck    = torch.load(str(path), map_location=DEVICE)
    cfg   = ck.get("config", CONFIG)
    state = {k.replace("_orig_mod.", ""): v for k, v in ck["model_state"].items()}
    m = CRAE(cfg).to(DEVICE)
    m.load_state_dict(state, strict=True)
    m.eval()
    print(f"  ✓ {path.name}")
    return m

# ─────────────────────────────────────────────
# VERİ YÜKLEME  (build_npy_cache ile aynı)
# ─────────────────────────────────────────────
def read_clip(clip_dir):
    """cv2 grayscale, /255, (T,H,W) — build_npy_cache.read_clip ile aynı"""
    files = sorted(
        Path(clip_dir).glob("frame*.jpg"),
        key=lambda p: int("".join(filter(str.isdigit, p.stem)))
    )
    if not files:  # jpg yoksa diğer uzantıları dene
        files = sorted(Path(clip_dir).iterdir(),
                       key=lambda p: p.name)
        files = [f for f in files if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}]

    if len(files) < CLIP_LEN:
        return None

    imgs = []
    for f in files[:CLIP_LEN]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))
        imgs.append(img.astype(np.float32) / 255.0)
    return np.stack(imgs)   # (T, H, W)


def load_test_dir(root):
    """root/GUN/sinif/klip/ → clips(N,T,H,W), labels(N,), names(N,)"""
    tasks = []
    for day in sorted(os.listdir(root)):
        dp = Path(root) / day
        if not dp.is_dir(): continue
        for cls in sorted(os.listdir(dp)):
            cp = dp / cls
            if not cp.is_dir(): continue
            label = 1 if cls.lower() in ANOMALY_CLASSES else 0
            for cd in sorted(os.listdir(cp)):
                full = cp / cd
                if full.is_dir():
                    tasks.append((str(full), label, cd))

    print(f"  {Path(root).name}: {len(tasks)} klip")
    clips, labels, names, errors = [], [], [], 0

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(read_clip, t[0]): t for t in tasks}
        done = 0
        for fut in as_completed(futs):
            path, label, name = futs[fut]
            clip = fut.result()
            done += 1
            if done % 50 == 0 or done == len(tasks):
                print(f"    {done}/{len(tasks)}...", end="\r")
            if clip is None:
                errors += 1; continue
            clips.append(clip); labels.append(label); names.append(name)

    print()
    la = np.array(labels)
    print(f"  ✓ {len(clips)} klip  normal={int((la==0).sum())}  "
          f"anomali={int((la==1).sum())}  hata={errors}")
    return np.stack(clips), la, np.array(names, dtype=object)

# ─────────────────────────────────────────────
# SKOR  (build_npy_cache'deki score_all ile aynı)
# ─────────────────────────────────────────────
@torch.no_grad()
def score_all(model, clips_np):
    """mean MSE tüm T,C,H,W — orijinal score_all ile aynı"""
    scores, recons = [], []
    for i in range(0, len(clips_np), BATCH_SIZE):
        # (B,T,H,W) → (B,T,1,H,W)
        b = torch.tensor(clips_np[i:i+BATCH_SIZE],
                         dtype=torch.float32).unsqueeze(2).to(DEVICE)
        out = model(b)
        for j in range(b.shape[0]):
            scores.append(torch.mean((b[j] - out[j]) ** 2).item())
        recons.append(out.squeeze(2).cpu().numpy())   # (B,T,H,W) görsel için
    return np.array(scores), np.concatenate(recons, axis=0)

# ─────────────────────────────────────────────
# GÖRSEL  (6 panel)
# ─────────────────────────────────────────────
def plot_dashboard(labels, scores, model_name, test_name):
    import matplotlib.gridspec as gridspec

    n_n = int((labels==0).sum()); n_a = int((labels==1).sum())

    fpr_a, tpr_a, thr_a = roc_curve(labels, scores)
    roc_auc = auc(fpr_a, tpr_a)

    fnr = 1 - tpr_a
    eer_i = np.argmin(np.abs(fpr_a - fnr))
    eer   = (fpr_a[eer_i] + fnr[eer_i]) / 2

    # Youden eşiği
    j_i   = np.argmax(tpr_a - fpr_a)
    thresh = thr_a[j_i]
    preds  = (scores >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
    f1   = 2*prec*rec/(prec+rec+1e-9)

    pp, pr, _ = precision_recall_curve(labels, scores)
    pf1 = 2*pp*pr/(pp+pr+1e-9)
    best_f1 = pf1.max()

    nm = scores[labels==0].mean(); am = scores[labels==1].mean()

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"CR-AE — {model_name.upper()}  →  {test_name}  "
                 f"(Normal:{n_n}  Anomali:{n_a})",
                 fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, hspace=0.38, wspace=0.35)

    ax1 = fig.add_subplot(gs[0,0])
    ax1.plot(fpr_a, tpr_a, "#1f77b4", lw=2, label=f"AUC={roc_auc:.4f}")
    ax1.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.5)
    ax1.scatter([fpr_a[eer_i]],[tpr_a[eer_i]], color="red", s=60,
                zorder=5, label=f"EER={eer:.4f}")
    ax1.set(xlabel="FPR", ylabel="TPR", title="ROC Eğrisi", xlim=[0,1], ylim=[0,1])
    ax1.legend(fontsize=8); ax1.grid(alpha=0.25)

    ax2 = fig.add_subplot(gs[0,1])
    ax2.plot(pr, pp, "#2ca02c", lw=2, label=f"Best F1={best_f1:.4f}")
    ax2.set(xlabel="Recall", ylabel="Precision", title="PR Eğrisi", xlim=[0,1], ylim=[0,1])
    ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

    ax3 = fig.add_subplot(gs[0,2])
    bins = np.linspace(scores.min(), np.percentile(scores,99), 40)
    ax3.hist(scores[labels==0], bins=bins, alpha=0.6, color="#5b9bd5", label="Normal")
    ax3.hist(scores[labels==1], bins=bins, alpha=0.6, color="#ed7d31", label="Anomali")
    ax3.axvline(thresh, color="black", lw=1.5, ls="--", label=f"Esik={thresh:.5f}")
    ax3.set(xlabel="MSE", ylabel="Frekans", title="Score Dağılımı")
    ax3.legend(fontsize=8); ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[1,0])
    cm_a = np.array([[tn,fp],[fn,tp]])
    ax4.imshow(cm_a, cmap="Blues")
    for (r,c),v in np.ndenumerate(cm_a):
        ax4.text(c, r, str(int(v)), ha="center", va="center", fontsize=14,
                 fontweight="bold",
                 color="white" if cm_a[r,c]>cm_a.max()*0.6 else "black")
    ax4.set_xticks([0,1]); ax4.set_yticks([0,1])
    ax4.set_xticklabels(["Pred Normal","Pred Anomali"], fontsize=8)
    ax4.set_yticklabels(["Gerçek Normal","Gerçek Anomali"], fontsize=8)
    ax4.set_title(f"CM (Youden eşik={thresh:.4f})", fontsize=10)

    ax5 = fig.add_subplot(gs[1,1])
    mdict = {"AUC":roc_auc,"F1":f1,"Precision":prec,"Recall":rec,"1-EER":1-eer}
    bars = ax5.bar(mdict.keys(), mdict.values(),
                   color=["#4472c4","#ed7d31","#a5a5a5","#ffc000","#5b9bd5"])
    for bar,val in zip(bars, mdict.values()):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax5.set_ylim([0,1.15]); ax5.set_title("Metrik Özeti"); ax5.grid(axis="y",alpha=0.25)

    ax6 = fig.add_subplot(gs[1,2])
    ax6.axis("off")
    ax6.text(0.05, 0.95,
        f"{'='*34}\n  {model_name.upper()} → {test_name}\n{'='*34}\n"
        f"AUC       : {roc_auc:.4f}\n"
        f"EER       : {eer:.4f}\n"
        f"Best F1   : {best_f1:.4f}\n"
        f"Precision : {prec:.4f}\n"
        f"Recall    : {rec:.4f}\n\n"
        f"Normal MSE: {nm:.5f}\nAnomali MSE:{am:.5f}\n"
        f"Ayrım     : {am/(nm+1e-9):.2f}x\n\n"
        f"TP:{tp:<4} FP:{fp:<4}\nFN:{fn:<4} TN:{tn:<4}\n\n"
        f"Normal:{n_n}  Anomali:{n_a}",
        transform=ax6.transAxes, fontsize=8.5, va="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", edgecolor="gray", alpha=0.9))
    plt.show()
    return fpr_a, tpr_a, thr_a, thresh

# ─────────────────────────────────────────────
# RECALL TABLOLARI
# ─────────────────────────────────────────────
def recall_tables(labels, scores, fpr_a, tpr_a, thr_a,
                  model_name, test_name, targets=(0.85, 0.90)):
    for target in targets:
        valid = np.where(tpr_a >= target)[0]
        if len(valid) == 0:
            print(f"  [UYARI] recall>={target} sağlanamadı"); continue
        idx    = valid[np.argmax(thr_a[valid])]
        thresh = thr_a[idx]
        preds  = (scores >= thresh).astype(int)
        tn,fp,fn,tp = confusion_matrix(labels, preds).ravel()
        prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
        f1   = 2*prec*rec/(prec+rec+1e-9)
        rows = [
            ["Threshold", f"{thresh:.6f}"], ["Recall",    f"{rec:.4f}"],
            ["Precision", f"{prec:.4f}"],   ["F1-Score",  f"{f1:.4f}"],
            ["Accuracy",  f"{(tp+tn)/(tp+tn+fp+fn+1e-9):.4f}"],
            ["FPR",       f"{fp/(fp+tn+1e-9):.4f}"],
            ["TNR",       f"{tn/(tn+fp+1e-9):.4f}"],
            ["FNR",       f"{fn/(fn+tp+1e-9):.4f}"],
            ["TP", str(tp)], ["FP", str(fp)], ["FN", str(fn)], ["TN", str(tn)],
        ]
        fig, ax = plt.subplots(figsize=(5, 4.2)); ax.axis("off")
        tbl = ax.table(cellText=rows, colLabels=["Metrik","Değer"],
                       cellLoc="center", loc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.3, 1.5)
        for j in range(2):
            tbl[(0,j)].set_facecolor("#2c5f8a")
            tbl[(0,j)].set_text_props(color="white", fontweight="bold")
        for i in range(1, len(rows)+1):
            for j in range(2):
                tbl[(i,j)].set_facecolor("#ddeeff" if i%2==0 else "white")
        ax.set_title(f"{model_name}  |  {test_name}\nRecall ≥ {target:.0%} eşiği",
                     fontsize=10, fontweight="bold", pad=12)
        plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
def main():
    print("\n── Modeller yükleniyor ──────────────────────")
    model_w = load_model(MODELS["winter"])
    model_s = load_model(MODELS["summer"])

    print("\n── Test verileri yükleniyor ─────────────────")
    clips_w, labels_w, names_w = load_test_dir(TESTS["winter"])
    clips_s, labels_s, names_s = load_test_dir(TESTS["summer"])

    results = {}

    for model_name, model in [("WINTER", model_w), ("SUMMER", model_s)]:
        print(f"\n{'█'*60}")
        print(f"  {model_name} MODEL")
        print(f"{'█'*60}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].plot([0,1],[0,1],"k--",lw=0.8,alpha=0.4)
        axes[0].set(xlabel="FPR", ylabel="TPR",
                    title=f"AUC-ROC — {model_name} Model",
                    xlim=[0,1], ylim=[0,1])

        for test_name, clips, labels, color in [
            ("Winter Test", clips_w, labels_w, "#1f4e8c"),
            ("Summer Test", clips_s, labels_s, "#a8321c"),
        ]:
            print(f"\n  → {test_name}")
            scores, recons = score_all(model, clips)
            fpr_a, tpr_a, thr_a = roc_curve(labels, scores)
            roc_auc = auc(fpr_a, tpr_a)
            j_i     = np.argmax(tpr_a - fpr_a)
            thresh  = thr_a[j_i]
            print(f"  AUC={roc_auc:.4f}  eşik={thresh:.6f}")

            # AUC figürü
            axes[0].plot(fpr_a, tpr_a, color=color, lw=2.5,
                         label=f"{test_name}  AUC={roc_auc:.4f}")
            axes[0].scatter([fpr_a[j_i]], [tpr_a[j_i]], color=color,
                            s=80, zorder=5, edgecolors="white", lw=0.8)
            axes[0].fill_between(fpr_a, tpr_a, alpha=0.07, color=color)

            # 6 panel dashboard
            plot_dashboard(labels, scores, model_name, test_name)
            # Recall tabloları
            recall_tables(labels, scores, fpr_a, tpr_a, thr_a,
                          model_name, test_name)

            results[(model_name, test_name)] = roc_auc

        axes[0].legend(loc="lower right", fontsize=9); axes[0].grid(alpha=0.25)
        axes[1].axis("off")
        plt.tight_layout(); plt.show()

    # Özet
    print("\n" + "="*55)
    print(f"  {'Model':<12} {'Test':<14} {'AUC':>8}")
    print(f"  {'-'*40}")
    for (mn, tn), v in results.items():
        print(f"  {mn:<12} {tn:<14} {v:>8.4f}")
    print("="*55)


if __name__ == "__main__":
    main()

In [ ]:
"""
Dataset & Model Analiz Raporu
==============================
Her iki model için:
- Eğitim config ve hiperparametreler
- Hangi günlerden kaç klip kullanıldı
- Eğitim/val split dağılımı
- Test seti kalite analizi (gün, sınıf, klip dağılımı)
- Score dağılımı istatistikleri
- Tüm çıktı metin tabanlı (print)
"""

import os, json
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict

BASE     = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")
NPY_DIR  = Path("/content/drive/MyDrive/archive/final_npycache")
RES_DIR  = BASE / "Testing_Results"
MODELS   = {
    "M1_WinterOnly": BASE / "models/crae_winter_only.pth",
    "M2_SummerOnly": BASE / "models/crae_summer_only.pth",
    "M3_Mixed":      BASE / "models/crae_mixed.pth",
}
TESTS = {
    "Winter": BASE / "testing/testing_final_winter",
    "Summer": BASE / "testing/testing_final_summer",
}
ANOMALY_CLASSES = {"anomaly", "trespassing", "loitering", "object_abandonment"}

SEP  = "=" * 70
SEP2 = "-" * 70

# ─────────────────────────────────────────────────────────────────
# YARDIMCI
# ─────────────────────────────────────────────────────────────────
def count_clips_in_dir(root, has_labels=True):
    """gün → {sinif: [klip_adları]} döndürür"""
    tree = defaultdict(lambda: defaultdict(list))
    root = Path(root)
    if not root.exists():
        return tree
    for day in sorted(os.listdir(root)):
        dp = root / day
        if not dp.is_dir(): continue
        if has_labels:
            for cls in sorted(os.listdir(dp)):
                cp = dp / cls
                if not cp.is_dir(): continue
                for cd in sorted(os.listdir(cp)):
                    if (cp / cd).is_dir():
                        tree[day][cls].append(cd)
        else:
            for cd in sorted(os.listdir(dp)):
                if (dp / cd).is_dir():
                    tree[day]["clips"].append(cd)
    return tree

def print_day_table(tree, has_labels=True):
    if has_labels:
        all_classes = sorted({c for d in tree.values() for c in d.keys()})
        header = f"  {'Gün':<12}" + "".join(f"  {c:<20}" for c in all_classes) + "  Toplam"
        print(header)
        print("  " + "-" * (len(header)-2))
        total_per_class = defaultdict(int)
        grand = 0
        for day in sorted(tree.keys()):
            row = f"  {day:<12}"
            day_total = 0
            for cls in all_classes:
                n = len(tree[day][cls])
                row += f"  {n:<20}"
                total_per_class[cls] += n
                day_total += n
            row += f"  {day_total}"
            grand += day_total
            print(row)
        print("  " + "-" * (len(header)-2))
        row = f"  {'TOPLAM':<12}"
        for cls in all_classes:
            row += f"  {total_per_class[cls]:<20}"
        row += f"  {grand}"
        print(row)
        return dict(total_per_class), grand
    else:
        print(f"  {'Gün':<12}  {'Klip Sayısı':>12}")
        print("  " + "-" * 28)
        total = 0
        for day in sorted(tree.keys()):
            n = len(tree[day]["clips"])
            print(f"  {day:<12}  {n:>12}")
            total += n
        print("  " + "-" * 28)
        print(f"  {'TOPLAM':<12}  {total:>12}")
        return total

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 1: MODEL CONFIG
# ─────────────────────────────────────────────────────────────────
print(SEP)
print("  BÖLÜM 1 — MODEL KONFİGÜRASYONLARI")
print(SEP)

for model_name, model_path in MODELS.items():
    print(f"\n  [{model_name}]")
    print(SEP2)
    ck = torch.load(str(model_path), map_location="cpu")
    cfg = ck.get("config", {})
    for k, v in cfg.items():
        print(f"    {k:<22} : {v}")
    print(f"    {'epoch (kayıt)':<22} : {ck.get('epoch', 'N/A')}")
    print(f"    {'val_loss (kayıt)':<22} : {ck.get('val_loss', 'N/A'):.6f}"
          if isinstance(ck.get('val_loss'), float) else
          f"    {'val_loss':<22} : {ck.get('val_loss', 'N/A')}")
    print(f"    {'model_name (kayıt)':<22} : {ck.get('model_name', 'N/A')}")

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 2: EĞİTİM VERİSİ DAĞILIMI (NPY'den)
# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 2 — EĞİTİM VERİSİ DAĞILIMI")
print(SEP)

train_sets = {
    "M1_WinterOnly — pure_bg_winter": NPY_DIR / "pure_bg_winter_names.npy",
    "M2_SummerOnly — pure_bg_summer": NPY_DIR / "pure_bg_summer_names.npy",
}

for label, npy_path in train_sets.items():
    print(f"\n  [{label}]")
    print(SEP2)
    if not Path(npy_path).exists():
        print(f"  NPY bulunamadı: {npy_path}")
        continue
    names = np.load(str(npy_path), allow_pickle=True)
    print(f"  Toplam klip: {len(names)}")

    day_counts = defaultdict(int)
    for name in names:
        # name formatı: clip_XXX veya YYYYMMDD/clip_XXX
        parts = str(name).split("/")
        day = parts[0] if len(parts) > 1 and parts[0].isdigit() else "bilinmiyor"
        day_counts[day] += 1

    print(f"\n  {'Gün':<12}  {'Klip Sayısı':>12}  {'Oran':>8}")
    print("  " + "-" * 36)
    for day in sorted(day_counts.keys()):
        n = day_counts[day]
        print(f"  {day:<12}  {n:>12}  {n/len(names)*100:>7.1f}%")
    print("  " + "-" * 36)
    print(f"  {'TOPLAM':<12}  {len(names):>12}  {'100.0%':>8}")

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 3: TEST VERİSİ DAĞILIMI (Drive'dan)
# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 3 — TEST VERİSİ DAĞILIMI (Drive'dan)")
print(SEP)

for test_name, test_path in TESTS.items():
    print(f"\n  [Test: {test_name}]")
    print(SEP2)
    tree = count_clips_in_dir(test_path, has_labels=True)
    if not tree:
        print(f"  Klasör bulunamadı: {test_path}")
        continue
    per_class, grand = print_day_table(tree, has_labels=True)
    print(f"\n  Özet:")
    print(f"    Toplam gün       : {len(tree)}")
    print(f"    Toplam klip      : {grand}")
    for cls, n in sorted(per_class.items()):
        print(f"    {cls:<20} : {n}  ({n/grand*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 4: TEST VERİSİ KALİTE ANALİZİ (klip başına frame sayısı)
# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 4 — KLİP KALİTE ANALİZİ (frame sayısı kontrolü)")
print(SEP)

for test_name, test_path in TESTS.items():
    print(f"\n  [Test: {test_name}]")
    print(SEP2)
    frame_counts = []
    empty_clips  = []
    short_clips  = []

    for day in sorted(os.listdir(test_path)):
        dp = Path(test_path) / day
        if not dp.is_dir(): continue
        for cls in os.listdir(dp):
            cp = dp / cls
            if not cp.is_dir(): continue
            for cd in os.listdir(cp):
                full = cp / cd
                if not full.is_dir(): continue
                frames = [f for f in os.listdir(full)
                          if f.lower().endswith((".jpg",".jpeg",".png"))]
                n = len(frames)
                frame_counts.append(n)
                if n == 0:
                    empty_clips.append(f"{day}/{cls}/{cd}")
                elif n < 16:
                    short_clips.append(f"{day}/{cls}/{cd} ({n} frame)")

    if frame_counts:
        arr = np.array(frame_counts)
        print(f"  Toplam klip incelendi : {len(arr)}")
        print(f"  Frame sayısı — min    : {arr.min()}")
        print(f"  Frame sayısı — max    : {arr.max()}")
        print(f"  Frame sayısı — ort    : {arr.mean():.1f}")
        print(f"  Frame sayısı — medyan : {np.median(arr):.0f}")
        print(f"  Tam 16 frame olan     : {int((arr==16).sum())}  ({(arr==16).sum()/len(arr)*100:.1f}%)")
        print(f"  Boş klip (<1 frame)   : {len(empty_clips)}")
        print(f"  Kısa klip (<16 frame) : {len(short_clips)}")
        if empty_clips:
            print(f"\n  Boş klipler:")
            for c in empty_clips[:10]:
                print(f"    {c}")
        if short_clips:
            print(f"\n  Kısa klipler (ilk 10):")
            for c in short_clips[:10]:
                print(f"    {c}")

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 5: SKOR İSTATİSTİKLERİ (JSON varsa)
# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 5 — MODEL PERFORMANS ÖZETİ (JSON'dan)")
print(SEP)

json_files = list(RES_DIR.glob("*.json")) if RES_DIR.exists() else []
if not json_files:
    print("  diff_technic_results/ altında JSON bulunamadı.")
else:
    for jf in sorted(json_files):
        print(f"\n  [{jf.name}]")
        print(SEP2)
        with open(jf) as f:
            data = json.load(f)
        for split in ["winter", "summer"]:
            if split not in data: continue
            d = data[split]
            print(f"  {split.upper()}:")
            print(f"    AUC       : {d.get('auc', 'N/A'):.4f}")
            print(f"    Threshold : {d.get('threshold', 'N/A'):.6f}")
            print(f"    FN (kaçırılan anomali)  : {d.get('fn_count', 'N/A')}")
            print(f"    FP (yanlış alarm)       : {d.get('fp_count', 'N/A')}")
            fn_days = d.get("fn_by_day", {})
            fp_days = d.get("fp_by_day", {})
            if fn_days:
                worst = sorted(fn_days.items(), key=lambda x: -x[1])[:3]
                print(f"    En çok FN olan günler  : "
                      f"{', '.join(f'{d}({n})' for d,n in worst)}")
            if fp_days:
                worst = sorted(fp_days.items(), key=lambda x: -x[1])[:3]
                print(f"    En çok FP olan günler  : "
                      f"{', '.join(f'{d}({n})' for d,n in worst)}")

# ─────────────────────────────────────────────────────────────────
# BÖLÜM 6: EĞİTİM vs TEST GÜN ÖRTÜŞMESİ
# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 6 — EĞİTİM / TEST GÜN ÖRTÜŞME KONTROLÜ")
print(SEP)

for model_label, npy_path in [
    ("M1_WinterOnly", NPY_DIR / "pure_bg_winter_names.npy"),
    ("M2_SummerOnly", NPY_DIR / "pure_bg_summer_names.npy"),
]:
    print(f"\n  [{model_label}]")
    print(SEP2)
    if not Path(npy_path).exists():
        print("  NPY bulunamadı"); continue

    names = np.load(str(npy_path), allow_pickle=True)
    train_days = set()
    for n in names:
        parts = str(n).split("/")
        if parts[0].isdigit():
            train_days.add(parts[0])

    for test_name, test_path in TESTS.items():
        test_days = set()
        for day in os.listdir(test_path):
            if (Path(test_path)/day).is_dir() and day.isdigit():
                test_days.add(day)

        overlap = train_days & test_days
        print(f"  Test={test_name}: eğitim {len(train_days)} gün, "
              f"test {len(test_days)} gün, "
              f"örtüşen={len(overlap)}")
        if overlap:
            print(f"    ⚠️  Örtüşen günler: {sorted(overlap)}")
        else:
            print(f"    ✓  Data leakage yok")

# ─────────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  RAPOR TAMAMLANDI")
print(SEP)

In [ ]:
"""
CR-AE Test — NPY'siz, doğrudan Drive'dan
==========================================
Preprocess: build_npy_cache.py ile BİREBİR AYNI
  - cv2 grayscale + /255
  - (B, T, 1, H, W) input
  - skor = mean MSE (tüm T,C,H,W)
  - LeakyReLU(0.2), Dropout2d encoder'da
"""

import os, cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)
import warnings; warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# AYARLAR
# ─────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE   = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")
MODELS = {
    "M1_WinterOnly": BASE / "models/crae_winter_only.pth",
    "M2_SummerOnly": BASE / "models/crae_summer_only.pth",
    "M3_Mixed":      BASE / "models/crae_mixed.pth",
}
TESTS = {
    "Winter": BASE / "testing/testing_final_winter",
    "Summer": BASE / "testing/testing_final_summer",
}
FRAME_SIZE      = 128
CLIP_LEN        = 16
BATCH_SIZE      = 64
WORKERS         = 32
ANOMALY_CLASSES = {"anomaly", "trespassing", "loitering", "object_abandonment"}

print(f"Device: {DEVICE}")

# ─────────────────────────────────────────────
# MODEL  (build_npy_cache'deki CRAE ile birebir)
# ─────────────────────────────────────────────
CONFIG = {
    "encoder_filters": [32, 64, 128, 256],
    "lstm_hidden": 256, "lstm_layers": 2,
    "dropout": 0.3, "frame_size": 128,
}

class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg["encoder_filters"]
        self.spatial = f[-1] * 8 * 8
        lstm_h = cfg["lstm_hidden"]
        drop   = cfg["dropout"]

        self.encoder = nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm   = nn.LSTM(lstm_h, lstm_h, cfg["lstm_layers"], batch_first=True,
                              dropout=drop if cfg["lstm_layers"] > 1 else 0)
        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        e = self.encoder(x.view(B*T, C, H, W))
        e = self.proj(e.view(B*T, -1)).view(B, T, -1)
        e, _ = self.lstm(e)
        d = self.unproj(e.reshape(B*T, -1)).view(B*T, 256, 8, 8)
        return self.decoder(d).view(B, T, 1, H, W)


def load_model(path):
    ck    = torch.load(str(path), map_location=DEVICE)
    cfg   = ck.get("config", CONFIG)
    state = {k.replace("_orig_mod.", ""): v for k, v in ck["model_state"].items()}
    m = CRAE(cfg).to(DEVICE)
    m.load_state_dict(state, strict=True)
    m.eval()
    print(f"  ✓ {path.name}")
    return m

# ─────────────────────────────────────────────
# VERİ YÜKLEME  (build_npy_cache ile aynı)
# ─────────────────────────────────────────────
def read_clip(clip_dir):
    """cv2 grayscale, /255, (T,H,W) — build_npy_cache.read_clip ile aynı"""
    files = sorted(
        Path(clip_dir).glob("frame*.jpg"),
        key=lambda p: int("".join(filter(str.isdigit, p.stem)))
    )
    if not files:  # jpg yoksa diğer uzantıları dene
        files = sorted(Path(clip_dir).iterdir(),
                       key=lambda p: p.name)
        files = [f for f in files if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}]

    if len(files) < CLIP_LEN:
        return None

    imgs = []
    for f in files[:CLIP_LEN]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))
        imgs.append(img.astype(np.float32) / 255.0)
    return np.stack(imgs)   # (T, H, W)


def load_test_dir(root):
    """root/GUN/sinif/klip/ → clips(N,T,H,W), labels(N,), names(N,)"""
    tasks = []
    for day in sorted(os.listdir(root)):
        dp = Path(root) / day
        if not dp.is_dir(): continue
        for cls in sorted(os.listdir(dp)):
            cp = dp / cls
            if not cp.is_dir(): continue
            label = 1 if cls.lower() in ANOMALY_CLASSES else 0
            for cd in sorted(os.listdir(cp)):
                full = cp / cd
                if full.is_dir():
                    tasks.append((str(full), label, cd))

    print(f"  {Path(root).name}: {len(tasks)} klip")
    clips, labels, names, errors = [], [], [], 0

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(read_clip, t[0]): t for t in tasks}
        done = 0
        for fut in as_completed(futs):
            path, label, name = futs[fut]
            clip = fut.result()
            done += 1
            if done % 50 == 0 or done == len(tasks):
                print(f"    {done}/{len(tasks)}...", end="\r")
            if clip is None:
                errors += 1; continue
            clips.append(clip); labels.append(label); names.append(name)

    print()
    la = np.array(labels)
    print(f"  ✓ {len(clips)} klip  normal={int((la==0).sum())}  "
          f"anomali={int((la==1).sum())}  hata={errors}")
    return np.stack(clips), la, np.array(names, dtype=object)

# ─────────────────────────────────────────────
# SKOR  (build_npy_cache'deki score_all ile aynı)
# ─────────────────────────────────────────────
@torch.no_grad()
def score_all(model, clips_np):
    """mean MSE tüm T,C,H,W — orijinal score_all ile aynı"""
    scores, recons = [], []
    for i in range(0, len(clips_np), BATCH_SIZE):
        # (B,T,H,W) → (B,T,1,H,W)
        b = torch.tensor(clips_np[i:i+BATCH_SIZE],
                         dtype=torch.float32).unsqueeze(2).to(DEVICE)
        out = model(b)
        for j in range(b.shape[0]):
            scores.append(torch.mean((b[j] - out[j]) ** 2).item())
        recons.append(out.squeeze(2).cpu().numpy())   # (B,T,H,W) görsel için
    return np.array(scores), np.concatenate(recons, axis=0)

# ─────────────────────────────────────────────
# GÖRSEL  (6 panel)
# ─────────────────────────────────────────────
def plot_dashboard(labels, scores, model_name, test_name):
    import matplotlib.gridspec as gridspec

    n_n = int((labels==0).sum()); n_a = int((labels==1).sum())

    fpr_a, tpr_a, thr_a = roc_curve(labels, scores)
    roc_auc = auc(fpr_a, tpr_a)

    fnr = 1 - tpr_a
    eer_i = np.argmin(np.abs(fpr_a - fnr))
    eer   = (fpr_a[eer_i] + fnr[eer_i]) / 2

    # Youden eşiği
    j_i   = np.argmax(tpr_a - fpr_a)
    thresh = thr_a[j_i]
    preds  = (scores >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
    f1   = 2*prec*rec/(prec+rec+1e-9)

    pp, pr, _ = precision_recall_curve(labels, scores)
    pf1 = 2*pp*pr/(pp+pr+1e-9)
    best_f1 = pf1.max()

    nm = scores[labels==0].mean(); am = scores[labels==1].mean()

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"CR-AE — {model_name.upper()}  →  {test_name}  "
                 f"(Normal:{n_n}  Anomali:{n_a})",
                 fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, hspace=0.38, wspace=0.35)

    ax1 = fig.add_subplot(gs[0,0])
    ax1.plot(fpr_a, tpr_a, "#1f77b4", lw=2, label=f"AUC={roc_auc:.4f}")
    ax1.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.5)
    ax1.scatter([fpr_a[eer_i]],[tpr_a[eer_i]], color="red", s=60,
                zorder=5, label=f"EER={eer:.4f}")
    ax1.set(xlabel="FPR", ylabel="TPR", title="ROC Eğrisi", xlim=[0,1], ylim=[0,1])
    ax1.legend(fontsize=8); ax1.grid(alpha=0.25)

    ax2 = fig.add_subplot(gs[0,1])
    ax2.plot(pr, pp, "#2ca02c", lw=2, label=f"Best F1={best_f1:.4f}")
    ax2.set(xlabel="Recall", ylabel="Precision", title="PR Eğrisi", xlim=[0,1], ylim=[0,1])
    ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

    ax3 = fig.add_subplot(gs[0,2])
    bins = np.linspace(scores.min(), np.percentile(scores,99), 40)
    ax3.hist(scores[labels==0], bins=bins, alpha=0.6, color="#5b9bd5", label="Normal")
    ax3.hist(scores[labels==1], bins=bins, alpha=0.6, color="#ed7d31", label="Anomali")
    ax3.axvline(thresh, color="black", lw=1.5, ls="--", label=f"Esik={thresh:.5f}")
    ax3.set(xlabel="MSE", ylabel="Frekans", title="Score Dağılımı")
    ax3.legend(fontsize=8); ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[1,0])
    cm_a = np.array([[tn,fp],[fn,tp]])
    ax4.imshow(cm_a, cmap="Blues")
    for (r,c),v in np.ndenumerate(cm_a):
        ax4.text(c, r, str(int(v)), ha="center", va="center", fontsize=14,
                 fontweight="bold",
                 color="white" if cm_a[r,c]>cm_a.max()*0.6 else "black")
    ax4.set_xticks([0,1]); ax4.set_yticks([0,1])
    ax4.set_xticklabels(["Pred Normal","Pred Anomali"], fontsize=8)
    ax4.set_yticklabels(["Gerçek Normal","Gerçek Anomali"], fontsize=8)
    ax4.set_title(f"CM (Youden eşik={thresh:.4f})", fontsize=10)

    ax5 = fig.add_subplot(gs[1,1])
    mdict = {"AUC":roc_auc,"F1":f1,"Precision":prec,"Recall":rec,"1-EER":1-eer}
    bars = ax5.bar(mdict.keys(), mdict.values(),
                   color=["#4472c4","#ed7d31","#a5a5a5","#ffc000","#5b9bd5"])
    for bar,val in zip(bars, mdict.values()):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax5.set_ylim([0,1.15]); ax5.set_title("Metrik Özeti"); ax5.grid(axis="y",alpha=0.25)

    ax6 = fig.add_subplot(gs[1,2])
    ax6.axis("off")
    ax6.text(0.05, 0.95,
        f"{'='*34}\n  {model_name.upper()} → {test_name}\n{'='*34}\n"
        f"AUC       : {roc_auc:.4f}\n"
        f"EER       : {eer:.4f}\n"
        f"Best F1   : {best_f1:.4f}\n"
        f"Precision : {prec:.4f}\n"
        f"Recall    : {rec:.4f}\n\n"
        f"Normal MSE: {nm:.5f}\nAnomali MSE:{am:.5f}\n"
        f"Ayrım     : {am/(nm+1e-9):.2f}x\n\n"
        f"TP:{tp:<4} FP:{fp:<4}\nFN:{fn:<4} TN:{tn:<4}\n\n"
        f"Normal:{n_n}  Anomali:{n_a}",
        transform=ax6.transAxes, fontsize=8.5, va="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", edgecolor="gray", alpha=0.9))
    plt.show()
    return fpr_a, tpr_a, thr_a, thresh

# ─────────────────────────────────────────────
# RECALL TABLOLARI
# ─────────────────────────────────────────────
def recall_tables(labels, scores, fpr_a, tpr_a, thr_a,
                  model_name, test_name, targets=(0.85, 0.90)):
    for target in targets:
        valid = np.where(tpr_a >= target)[0]
        if len(valid) == 0:
            print(f"  [UYARI] recall>={target} sağlanamadı"); continue
        idx    = valid[np.argmax(thr_a[valid])]
        thresh = thr_a[idx]
        preds  = (scores >= thresh).astype(int)
        tn,fp,fn,tp = confusion_matrix(labels, preds).ravel()
        prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
        f1   = 2*prec*rec/(prec+rec+1e-9)
        rows = [
            ["Threshold", f"{thresh:.6f}"], ["Recall",    f"{rec:.4f}"],
            ["Precision", f"{prec:.4f}"],   ["F1-Score",  f"{f1:.4f}"],
            ["Accuracy",  f"{(tp+tn)/(tp+tn+fp+fn+1e-9):.4f}"],
            ["FPR",       f"{fp/(fp+tn+1e-9):.4f}"],
            ["TNR",       f"{tn/(tn+fp+1e-9):.4f}"],
            ["FNR",       f"{fn/(fn+tp+1e-9):.4f}"],
            ["TP", str(tp)], ["FP", str(fp)], ["FN", str(fn)], ["TN", str(tn)],
        ]
        fig, ax = plt.subplots(figsize=(5, 4.2)); ax.axis("off")
        tbl = ax.table(cellText=rows, colLabels=["Metrik","Değer"],
                       cellLoc="center", loc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.3, 1.5)
        for j in range(2):
            tbl[(0,j)].set_facecolor("#2c5f8a")
            tbl[(0,j)].set_text_props(color="white", fontweight="bold")
        for i in range(1, len(rows)+1):
            for j in range(2):
                tbl[(i,j)].set_facecolor("#ddeeff" if i%2==0 else "white")
        ax.set_title(f"{model_name}  |  {test_name}\nRecall ≥ {target:.0%} eşiği",
                     fontsize=10, fontweight="bold", pad=12)
        plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
def main():
    print("\n── Modeller yükleniyor ──────────────────────")
    loaded_models = {}
    for mname, mpath in MODELS.items():
        if not mpath.exists():
            print(f"  [ATLA] {mpath.name} bulunamadı")
            continue
        loaded_models[mname] = load_model(mpath)

    print("\n── Test verileri yükleniyor ─────────────────")
    test_data = {}
    for tname, tpath in TESTS.items():
        clips, labels, names = load_test_dir(tpath)
        test_data[tname] = (clips, labels, names)

    results = {}

    for model_name, model in loaded_models.items():
        print(f"\n{'█'*60}")
        print(f"  {model_name}")
        print(f"{'█'*60}")

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.4,label="Rastgele")
        ax.set(xlabel="FPR", ylabel="TPR",
               title=f"AUC-ROC — {model_name}",
               xlim=[0,1], ylim=[0,1])

        colors = {"Winter": "#1f4e8c", "Summer": "#a8321c"}

        for test_name, (clips, labels, names) in test_data.items():
            print(f"\n  → {test_name}")
            scores, recons = score_all(model, clips)
            fpr_a, tpr_a, thr_a = roc_curve(labels, scores)
            roc_auc = auc(fpr_a, tpr_a)
            j_i     = np.argmax(tpr_a - fpr_a)
            thresh  = thr_a[j_i]
            print(f"  AUC={roc_auc:.4f}  eşik={thresh:.6f}")

            color = colors.get(test_name, "#555555")
            ax.plot(fpr_a, tpr_a, color=color, lw=2.5,
                    label=f"{test_name}  AUC={roc_auc:.4f}")
            ax.scatter([fpr_a[j_i]], [tpr_a[j_i]], color=color,
                       s=80, zorder=5, edgecolors="white", lw=0.8)
            ax.fill_between(fpr_a, tpr_a, alpha=0.07, color=color)

            plot_dashboard(labels, scores, model_name, test_name)
            recall_tables(labels, scores, fpr_a, tpr_a, thr_a,
                          model_name, test_name)

            results[(model_name, test_name)] = roc_auc

        ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)
        plt.tight_layout(); plt.show()

    # Özet tablo
    print("\n" + "="*60)
    print(f"  {'Model':<18} {'Test':<14} {'AUC':>8}")
    print(f"  {'-'*44}")
    for (mn, tn), v in results.items():
        print(f"  {mn:<18} {tn:<14} {v:>8.4f}")

    # Model karşılaştırma matrisi
    print(f"\n  {'Model':<18}", end="")
    test_names = list(test_data.keys())
    for tn in test_names:
        print(f"  {tn:>12}", end="")
    print(f"  {'Toplam':>10}")
    print(f"  {'-'*56}")
    for mn in loaded_models:
        print(f"  {mn:<18}", end="")
        total = 0
        for tn in test_names:
            v = results.get((mn, tn), float("nan"))
            print(f"  {v:>12.4f}", end="")
            total += v
        print(f"  {total:>10.4f}")
    print("="*60)


if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
from pathlib import Path
from collections import defaultdict

BASE    = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")
NPY_DIR = BASE / "final_npycache"
TESTS   = {
    "Winter": BASE / "testing/testing_final_winter",
    "Summer": BASE / "testing/testing_final_summer",
}
SEP  = "=" * 60
SEP2 = "-" * 60

# ── BÖLÜM 4: KLİP KALİTE ANALİZİ ────────────────────────────
print(SEP)
print("  BÖLÜM 4 — KLİP KALİTE ANALİZİ")
print(SEP)

for test_name, test_path in TESTS.items():
    print(f"\n  [Test: {test_name}]")
    print(SEP2)
    frame_counts, empty, short = [], [], []

    for day in sorted(os.listdir(test_path)):
        dp = test_path / day
        if not dp.is_dir(): continue
        for cls in os.listdir(dp):
            cp = dp / cls
            if not cp.is_dir(): continue
            for cd in os.listdir(cp):
                full = cp / cd
                if not full.is_dir(): continue
                frames = [f for f in os.listdir(full)
                          if f.lower().endswith((".jpg",".jpeg",".png"))]
                n = len(frames)
                frame_counts.append(n)
                if n == 0:   empty.append(f"{day}/{cls}/{cd}")
                elif n < 16: short.append(f"{day}/{cls}/{cd} ({n} frame)")

    arr = np.array(frame_counts)
    print(f"  Toplam klip        : {len(arr)}")
    print(f"  Frame min/max/ort  : {arr.min()} / {arr.max()} / {arr.mean():.1f}")
    print(f"  Tam 16 frame       : {int((arr==16).sum())}  ({(arr==16).mean()*100:.1f}%)")
    print(f"  Boş klip           : {len(empty)}")
    print(f"  Kısa klip (<16)    : {len(short)}")
    for c in empty[:5]:  print(f"    [BOŞ]  {c}")
    for c in short[:5]:  print(f"    [KISA] {c}")

# ── BÖLÜM 6: DATA LEAKAGE KONTROLÜ ───────────────────────────
print(f"\n{SEP}")
print("  BÖLÜM 6 — EĞİTİM / TEST GÜN ÖRTÜŞME KONTROLÜ")
print(SEP)

# Mevcut NPY'leri otomatik bul
print(f"\n  NPY klasörü: {NPY_DIR}")
all_npys = sorted(NPY_DIR.glob("*.npy")) if NPY_DIR.exists() else []
print(f"  Bulunan dosyalar ({len(all_npys)}):")
for f in all_npys:
    print(f"    {f.name:<50}  {f.stat().st_size/1e6:>7.1f} MB")

# Model → names NPY eşleşmesi
candidates = {
    "M1_WinterOnly": ["pure_bg_winter_names.npy", "pure_bg_winter_128_names.npy",
                      "train_winter_names.npy"],
    "M2_SummerOnly": ["pure_bg_summer_names.npy", "pure_bg_summer_128_names.npy",
                      "train_summer_names.npy"],
    "M3_Mixed":      ["pure_bg_mixed_names.npy",  "train_mixed_names.npy"],
}

print()
for model_label, names_list in candidates.items():
    npy_path = next((NPY_DIR/n for n in names_list if (NPY_DIR/n).exists()), None)
    print(f"  [{model_label}]")
    if npy_path is None:
        print(f"  Names NPY bulunamadı ({', '.join(names_list)})\n")
        continue
    print(f"  Kaynak: {npy_path.name}")
    names = np.load(str(npy_path), allow_pickle=True)
    train_days = set()
    for n in names:
        parts = str(n).replace("\\", "/").split("/")
        day = next((p for p in parts if p.isdigit() and len(p) == 8), None)
        if day: train_days.add(day)

    print(f"  Eğitim günleri: {len(train_days)}")
    for test_name, test_path in TESTS.items():
        test_days = {d for d in os.listdir(test_path)
                     if (test_path/d).is_dir() and d.isdigit()}
        overlap = sorted(train_days & test_days)
        if overlap:
            print(f"  vs {test_name:<8}: ⚠️  {len(overlap)} örtüşen gün → {overlap}")
        else:
            print(f"  vs {test_name:<8}: ✓  leakage yok  (test={len(test_days)} gün)")
    print()

In [ ]:
"""
crae_finetune_m1_summer.py
===========================
M1_WinterOnly → pure_bg_summer fine-tune
Sadece eğitim. Test ayrı scriptte yapılacak.
Çıktı: models/crae_winter_finetuned.pth
"""

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, random, gc, json
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Image as IPImage
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

NPY_DIR  = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/final_npycache')
SAVE_DIR = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/models')
RES_DIR  = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/Testing_Results/finetuning_model')
M1_CKPT  = SAVE_DIR / 'crae_winter_only.pth'
SAVE_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'winter_finetuned'

CONFIG = {
    'frame_size'      : 128,
    'frames_per_clip' : 16,
    'batch_size'      : 32,
    'val_split'       : 0.15,
    'epochs'          : 30,
    'lr'              : 1e-4,
    'min_lr'          : 1e-7,
    'lstm_hidden'     : 256,
    'lstm_layers'     : 2,
    'dropout'         : 0.3,
    'patience'        : 10,
    'weight_decay'    : 1e-5,
    'encoder_filters' : [32, 64, 128, 256],
    'freeze_encoder'  : True,      # False → full fine-tune
    'winter_mix_ratio': 0.3,       # catastrophic forgetting koruması
}

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL (orijinalle birebir)
# ══════════════════════════════════════════════════════════════════════════════
class CRAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        f = cfg['encoder_filters']
        self.spatial = f[-1]*8*8
        lstm_h = cfg['lstm_hidden']
        drop   = cfg['dropout']
        self.encoder = nn.Sequential(
            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
            nn.Dropout2d(drop),
        )
        self.proj    = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
        self.lstm    = nn.LSTM(lstm_h, lstm_h, cfg['lstm_layers'], batch_first=True,
                               dropout=drop if cfg['lstm_layers']>1 else 0)
        self.unproj  = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
        )
    def forward(self, x):
        B,T,C,H,W = x.shape
        e = self.encoder(x.view(B*T,C,H,W))
        e = self.proj(e.view(B*T,-1)).view(B,T,-1)
        e,_ = self.lstm(e)
        d = self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)
        return self.decoder(d).view(B,T,1,H,W)

# ══════════════════════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════════════════════
class AugDataset(Dataset):
    def __init__(self, data, aug=True):
        self.data = torch.from_numpy(data)
        self.aug  = aug
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        clip = self.data[i].clone()
        if self.aug:
            if random.random() > 0.5:
                clip = torch.flip(clip, dims=[2])
            clip = torch.clamp(clip + random.uniform(-0.1, 0.1), 0, 1)
            clip = torch.clamp(clip + torch.randn_like(clip) * 0.02, 0, 1)
        return clip.unsqueeze(1)   # (T,1,H,W)

# ══════════════════════════════════════════════════════════════════════════════
#  VERİ YÜKLEME
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("VERİ YÜKLEME")
print("="*60)
t0 = time.time()

summer_clips = np.load(str(NPY_DIR/'pure_bg_summer_128.npy')).astype(np.float32)
summer_names = np.load(str(NPY_DIR/'pure_bg_summer_names.npy'), allow_pickle=True)
print(f"Summer: {len(summer_clips)} klip")

if CONFIG['winter_mix_ratio'] > 0:
    winter_clips = np.load(str(NPY_DIR/'pure_bg_winter_128.npy')).astype(np.float32)
    winter_names = np.load(str(NPY_DIR/'pure_bg_winter_names.npy'), allow_pickle=True)
    n_mix = int(len(summer_clips) * CONFIG['winter_mix_ratio'] / (1 - CONFIG['winter_mix_ratio']))
    n_mix = min(n_mix, len(winter_clips))
    w_idx = np.random.default_rng(SEED).choice(len(winter_clips), n_mix, replace=False)
    mixed_clips = np.concatenate([summer_clips, winter_clips[w_idx]])
    mixed_names = np.concatenate([summer_names, winter_names[w_idx]])
    del winter_clips, winter_names; gc.collect()
    print(f"Winter mix: {n_mix} klip  |  Toplam: {len(mixed_clips)} klip")
else:
    mixed_clips = summer_clips
    mixed_names = summer_names
del summer_clips; gc.collect()
print(f"Yükleme: {time.time()-t0:.1f}sn")

# ── Gün bazlı train/val split ─────────────────────────────────────────────
def day_stratified_split(names, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    day_to_idx    = defaultdict(list)
    month_to_days = defaultdict(set)
    for i, name in enumerate(names):
        d = str(name)[:8]; m = d[:6]
        day_to_idx[d].append(i); month_to_days[m].add(d)
    tr_idx=[]; val_idx=[]
    for month in sorted(month_to_days):
        days  = sorted(month_to_days[month])
        n_val = max(1, round(len(days)*val_ratio))
        shuf  = days[:]; rng.shuffle(shuf); vd = set(shuf[:n_val])
        for d in days:
            (val_idx if d in vd else tr_idx).extend(day_to_idx[d])
    return np.array(tr_idx), np.array(val_idx)

tr_idx, val_idx = day_stratified_split(mixed_names, CONFIG['val_split'], SEED)
train_data = mixed_clips[tr_idx]
val_data   = mixed_clips[val_idx]
print(f"Train: {len(train_data)}  Val: {len(val_data)}")
del mixed_clips; gc.collect()

train_dl = DataLoader(AugDataset(train_data, aug=True),  CONFIG['batch_size'],
                      shuffle=True,  num_workers=4, pin_memory=True,
                      persistent_workers=True, prefetch_factor=2)
val_dl   = DataLoader(AugDataset(val_data,   aug=False), CONFIG['batch_size'],
                      shuffle=False, num_workers=4, pin_memory=True,
                      persistent_workers=True, prefetch_factor=2)
del train_data, val_data; gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  M1 YÜKLE + FREEZE
# ══════════════════════════════════════════════════════════════════════════════
model = CRAE(CONFIG).to(device)
ck    = torch.load(str(M1_CKPT), map_location=device)
state = ck['model_state']
# torch.compile ile kaydedildiyse _orig_mod. prefix'ini temizle
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
model.load_state_dict(state)
print(f"\nM1 yüklendi → epoch={ck['epoch']+1}  val_loss={ck['val_loss']:.6f}")

if CONFIG['freeze_encoder']:
    for p in model.encoder.parameters(): p.requires_grad = False
    for p in model.proj.parameters():    p.requires_grad = False
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Freeze    : encoder + proj  ({frozen:,} param)")
    print(f"Trainable : LSTM + unproj + decoder  ({trainable:,} param)")
else:
    print(f"Full fine-tune: {sum(p.numel() for p in model.parameters()):,} param")

# ══════════════════════════════════════════════════════════════════════════════
#  EĞİTİM
# ══════════════════════════════════════════════════════════════════════════════
criterion = nn.MSELoss()
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=CONFIG['min_lr']
)

save_path = SAVE_DIR / f'crae_{MODEL_NAME}.pth'
best_val  = float('inf'); pat = 0
tl_h=[]; vl_h=[]; lr_h=[]

# Başlangıç referans
model.eval(); ref_vl=0; nv=0
with torch.no_grad():
    for batch in val_dl:
        clips = batch.to(device, non_blocking=True)
        ref_vl += criterion(model(clips), clips).item(); nv += 1
ref_vl /= max(nv, 1)
print(f"\nFine-tune öncesi val loss: {ref_vl:.6f}")
print(f"\n{'='*60}\nFINE-TUNE — {MODEL_NAME}\n{'='*60}")

t_start = time.time()
for ep in range(CONFIG['epochs']):
    model.train()
    if CONFIG['freeze_encoder']:
        model.encoder.eval()
        model.proj.eval()

    el=nb=0
    for batch in train_dl:
        clips = batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(clips), clips)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        el += loss.item(); nb += 1
    tl = el / max(nb, 1); tl_h.append(tl)

    model.eval(); vl=nv=0
    with torch.no_grad():
        for batch in val_dl:
            clips = batch.to(device, non_blocking=True)
            vl += criterion(model(clips), clips).item(); nv += 1
    vl /= max(nv, 1); vl_h.append(vl)
    lr_h.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    if vl < best_val:
        best_val = vl; pat = 0
        torch.save({
            'epoch': ep, 'model_state': model.state_dict(),
            'val_loss': vl, 'config': CONFIG,
            'model_name': MODEL_NAME, 'base_model': 'crae_winter_only.pth',
        }, str(save_path))
    else:
        pat += 1

    elapsed = time.time() - t_start
    eta     = elapsed / (ep+1) * (CONFIG['epochs'] - ep - 1)
    print(f"  Ep{ep+1:>3} T:{tl:.6f} V:{vl:.6f} Best:{best_val:.6f} "
          f"Pat:{pat}/{CONFIG['patience']} {elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)
    if pat >= CONFIG['patience']:
        print(f"  Early stopping ep{ep+1}"); break

total_time = time.time() - t_start
print(f"\nFine-tune tamamlandı: {total_time/60:.1f}dk")

# ── Grafik ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(tl_h, 'b-', label='Train', alpha=0.7)
axes[0].plot(vl_h, 'r-', label='Val',   alpha=0.7)
axes[0].axhline(ref_vl, color='orange', ls='--', alpha=0.7, label=f'Başlangıç={ref_vl:.5f}')
axes[0].set_title(f'Fine-tune Eğrisi — {MODEL_NAME}')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(lr_h, 'g-'); axes[1].set_title('LR Schedule'); axes[1].grid(alpha=0.3)
plt.tight_layout()
train_png = RES_DIR / f'training_{MODEL_NAME}.png'
plt.savefig(str(train_png), dpi=150); plt.close()
display(IPImage(str(train_png)))

# ── Özet ──────────────────────────────────────────────────────────────────
ck = torch.load(str(save_path), map_location='cpu')
summary = {
    'model_name'       : MODEL_NAME,
    'base_model'       : 'crae_winter_only.pth',
    'finetune_data'    : f'pure_bg_summer + {CONFIG["winter_mix_ratio"]*100:.0f}% winter_mix',
    'freeze_encoder'   : CONFIG['freeze_encoder'],
    'winter_mix_ratio' : CONFIG['winter_mix_ratio'],
    'config'           : CONFIG,
    'start_val_loss'   : float(ref_vl),
    'best_val_loss'    : float(best_val),
    'improvement'      : float(ref_vl - best_val),
    'best_epoch'       : int(ck['epoch']) + 1,
    'total_epochs'     : len(tl_h),
    'training_time_min': round(total_time/60, 1),
    'train_losses'     : [float(x) for x in tl_h],
    'val_losses'       : [float(x) for x in vl_h],
}
out_json = RES_DIR / f'{MODEL_NAME}_train_summary.json'
with open(str(out_json), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print(f"  Başlangıç val loss : {ref_vl:.6f}")
print(f"  En iyi val loss    : {best_val:.6f}  (ep {summary['best_epoch']})")
print(f"  İyileşme           : {summary['improvement']:.6f}")
print(f"  Model  : {save_path}")
print(f"  JSON   : {out_json}")
print("="*60)
gc.collect()

In [ ]:
"""
crae_test_finetuned2.py
======================
crae_winter_finetuned2.pth modelini winter ve summer test setleriyle test eder.
Veri: testing_final_winter / testing_final_summer klasörlerinden paralel yükleme.
Çıktı: Testing_Results/finetuning_model/
"""

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time, json, cv2
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Image as IPImage
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix

# ══════════════════════════════════════════════════════════════════════════════
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BASE        = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors')
CKPT_PATH   = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/models/crae_winter_finetuned2.pth')
TEST_WINTER = BASE / 'testing/testing_final_winter'
TEST_SUMMER = BASE / 'testing/testing_final_summer'
RES_DIR     = BASE / 'Testing_Results/finetuning_model'
RES_DIR.mkdir(parents=True, exist_ok=True)

FRAME_SIZE     = 128
FRAMES_PER_CLIP = 16
BATCH_SIZE     = 32
N_WORKERS      = 8     # paralel thread sayısı
IMG_EXTS       = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
def build_model(cfg):
    import torch.nn as nn
    f = cfg['encoder_filters']
    class CRAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.spatial = f[-1]*8*8
            lstm_h = cfg['lstm_hidden']; drop = cfg['dropout']
            self.encoder = nn.Sequential(
                nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
                nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
                nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
                nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),
                nn.Dropout2d(drop),
            )
            self.proj    = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))
            self.lstm    = nn.LSTM(lstm_h, lstm_h, cfg['lstm_layers'], batch_first=True,
                                   dropout=drop if cfg['lstm_layers']>1 else 0)
            self.unproj  = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),
                nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),
                nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),
                nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),
            )
        def forward(self, x):
            B,T,C,H,W = x.shape
            e = self.encoder(x.view(B*T,C,H,W))
            e = self.proj(e.view(B*T,-1)).view(B,T,-1)
            e,_ = self.lstm(e)
            d = self.unproj(e.reshape(B*T,-1)).view(B*T,f[-1],8,8)
            return self.decoder(d).view(B,T,1,H,W)
    return CRAE()

# ══════════════════════════════════════════════════════════════════════════════
#  PARALEL KLASÖR YÜKLEYİCİ
# ══════════════════════════════════════════════════════════════════════════════
def _load_clip(args):
    """Tek klip klasörünü yükler — thread worker."""
    idx, clip_dir, label = args
    frames_paths = sorted(
        f for f in clip_dir.iterdir()
        if f.is_file() and f.suffix.lower() in IMG_EXTS
    )
    if len(frames_paths) < FRAMES_PER_CLIP:
        return idx, None
    frames = []
    for fp in frames_paths[:FRAMES_PER_CLIP]:
        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return idx, None
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_AREA)
        frames.append(img.astype(np.float32) / 255.0)
    return idx, np.stack(frames, axis=0)   # (T, H, W)

def load_test_folder(test_root: Path, n_workers: int = 8):
    """
    Klasör yapısı: test_root / <gun> / <sinif> / <klip> / frames
    Etiket: klasör adında 'normal' geçiyorsa → 0, geçmiyorsa → 1
    Paralel ThreadPoolExecutor ile yükler, RAM'e alır.
    """
    print(f"\n{test_root.name} yükleniyor...")

    # Tüm klip yollarını topla
    jobs = []
    for day_dir in sorted(test_root.iterdir()):
        if not day_dir.is_dir(): continue
        for cls_dir in sorted(day_dir.iterdir()):
            if not cls_dir.is_dir(): continue
            label = 0 if 'normal' in cls_dir.name.lower() else 1
            for clip_dir in sorted(cls_dir.iterdir()):
                if clip_dir.is_dir():
                    jobs.append((len(jobs), clip_dir, label))

    print(f"  Toplam klip: {len(jobs)} — {n_workers} thread ile yükleniyor...")
    t0 = time.time()

    buf = [None] * len(jobs)
    labels_buf = [None] * len(jobs)
    done = 0
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_load_clip, j): j for j in jobs}
        for fut in as_completed(futs):
            idx, arr = fut.result()
            orig = futs[fut]
            labels_buf[idx] = orig[2]
            if arr is not None:
                buf[idx] = arr
            done += 1
            if done % 50 == 0:
                print(f"    {done}/{len(jobs)}...", flush=True)

    valid_idx = [i for i, x in enumerate(buf) if x is not None]
    clips  = np.stack([buf[i] for i in valid_idx]).astype(np.float32)
    labels = np.array([labels_buf[i] for i in valid_idx], dtype=np.int64)

    print(f"  Yüklendi: {len(clips)} klip  ({time.time()-t0:.1f}sn)")
    print(f"  Normal: {(labels==0).sum()}  Anomali: {(labels==1).sum()}")
    return clips, labels

# ══════════════════════════════════════════════════════════════════════════════
#  DATASET (RAM tensor)
# ══════════════════════════════════════════════════════════════════════════════
class RamTestDS(Dataset):
    def __init__(self, clips, labels):
        self.clips  = torch.from_numpy(clips).unsqueeze(2)   # (N,T,1,H,W)
        self.labels = torch.from_numpy(labels)
    def __len__(self): return len(self.clips)
    def __getitem__(self, i): return self.clips[i], self.labels[i]

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL YÜKLE
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nCheckpoint yükleniyor: {CKPT_PATH.name}")
ck = torch.load(str(CKPT_PATH), map_location=device)
cfg = ck['config']
model = build_model(cfg).to(device)
state = ck['model_state']
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
model.load_state_dict(state)
model.eval()
print(f"Model yüklendi → epoch={ck['epoch']+1}  val_loss={ck['val_loss']:.6f}")

# ══════════════════════════════════════════════════════════════════════════════
#  VERİ YÜKLE (test klasörlerinden)
# ══════════════════════════════════════════════════════════════════════════════
tw_clips, tw_labels = load_test_folder(TEST_WINTER, N_WORKERS)
ts_clips, ts_labels = load_test_folder(TEST_SUMMER, N_WORKERS)

tw_dl = DataLoader(RamTestDS(tw_clips, tw_labels), BATCH_SIZE, shuffle=False, num_workers=0)
ts_dl = DataLoader(RamTestDS(ts_clips, ts_labels), BATCH_SIZE, shuffle=False, num_workers=0)

# ══════════════════════════════════════════════════════════════════════════════
#  TEST FONKSİYONU
# ══════════════════════════════════════════════════════════════════════════════
def run_test(dl, tag):
    scores=[]; labs=[]
    with torch.no_grad():
        for clips, lbls in dl:
            clips = clips.to(device, non_blocking=True)
            out   = model(clips)
            mse   = ((clips - out)**2).mean(dim=(1,2,3,4))
            scores.extend(mse.cpu().numpy().tolist())
            labs.extend(lbls.numpy().tolist())

    scores = np.array(scores, dtype=np.float32)
    labs   = np.array(labs,   dtype=np.int64)
    ns  = scores[labs==0]
    als = scores[labs==1]

    auc = roc_auc_score(labs, scores)
    prec, rec, thr = precision_recall_curve(labs, scores)
    f1s   = 2*prec*rec / (prec+rec+1e-8)
    opt_t = thr[np.argmax(f1s[:-1])]
    preds = (scores >= opt_t).astype(int)
    tn, fp, fn, tp = confusion_matrix(labs, preds).ravel()
    fpr_r, tpr_r, _ = roc_curve(labs, scores)
    eer_i = np.nanargmin(np.abs(fpr_r - (1-tpr_r)))
    eer   = float((fpr_r[eer_i] + (1-tpr_r[eer_i])) / 2)

    print(f"\n  {'='*50}")
    print(f"  {tag}")
    print(f"  {'='*50}")
    print(f"  Normal MSE : {ns.mean():.6f} ± {ns.std():.6f}")
    print(f"  Anomali MSE: {als.mean():.6f} ± {als.std():.6f}")
    print(f"  AUC        : {auc:.4f}")
    print(f"  EER        : {eer:.4f}")
    print(f"  F1         : {f1s.max():.4f}")
    print(f"  Threshold  : {opt_t:.6f}")
    print(f"  TP:{tp}  FP:{fp}  FN:{fn}  TN:{tn}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'M1_Finetuned — {tag}', fontsize=13)

    axes[0].plot(fpr_r, tpr_r, 'b-', lw=2, label=f'AUC={auc:.4f}')
    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
    axes[0].set_title('ROC'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')

    axes[1].hist(ns,  bins=50, alpha=0.6, color='green', label=f'Normal ({len(ns)})')
    axes[1].hist(als, bins=50, alpha=0.6, color='red',   label=f'Anomali ({len(als)})')
    axes[1].axvline(opt_t, color='black', ls='--', label=f'Thr={opt_t:.5f}')
    axes[1].set_title('Score Dağılımı'); axes[1].legend(); axes[1].grid(alpha=0.3)

    sns.heatmap([[tn,fp],[fn,tp]], annot=True, fmt='d', ax=axes[2],
                cmap='Blues', xticklabels=['Normal','Anomali'],
                yticklabels=['Normal','Anomali'])
    axes[2].set_title('Confusion Matrix')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

    plt.tight_layout()
    out_png = RES_DIR / f'M1_finetuned_{tag.lower()}.png'
    plt.savefig(str(out_png), dpi=150); plt.close()
    display(IPImage(str(out_png)))

    return {
        'tag': tag, 'auc': float(auc), 'eer': float(eer),
        'f1': float(f1s.max()), 'threshold': float(opt_t),
        'precision': float(tp/(tp+fp+1e-8)),
        'recall':    float(tp/(tp+fn+1e-8)),
        'specificity': float(tn/(tn+fp+1e-8)),
        'normal_mse':  float(ns.mean()), 'normal_mse_std':  float(ns.std()),
        'anomaly_mse': float(als.mean()), 'anomaly_mse_std': float(als.std()),
        'n_normal': int((labs==0).sum()), 'n_anomaly': int((labs==1).sum()),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    }

# ══════════════════════════════════════════════════════════════════════════════
#  ÇALIŞTIR
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}\nTEST\n{'='*60}")
r_winter = run_test(tw_dl, 'WINTER')
r_summer = run_test(ts_dl, 'SUMMER')

# JSON kaydet
results = {
    'model'       : 'crae_winter_finetuned2.pth',
    'checkpoint'  : {'epoch': int(ck['epoch'])+1, 'val_loss': float(ck['val_loss'])},
    'test_winter' : r_winter,
    'test_summer' : r_summer,
}
out_json = RES_DIR / 'M1_finetuned_test_results.json'
with open(str(out_json), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print(f"ÖZET — M1_Finetuned")
print(f"{'='*60}")
print(f"  {'Test':<12} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print(f"  {'-'*55}")
for r in [r_winter, r_summer]:
    print(f"  {r['tag']:<12} {r['auc']:>8.4f} {r['eer']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")
print(f"\n  JSON: {out_json}")
print("="*60)